## Condition Configuration

Run this notebook after the rephrasing section in `gen_proposals.ipynb`.

- `CONDITION`: condition name whose rephrased proposal files should be analyzed.
- `REUSE_CACHED_PROPOSAL_EMBEDDINGS`: load proposal embeddings if they already exist.
- `REUSE_CACHED_MAIN_IDEA_EMBEDDINGS`: load main-idea embeddings if they already exist.
- `REUSE_CACHED_LITERATURE_EMBEDDINGS`: load literature embeddings if they already exist.


In [ ]:
CONDITION = 'minimal'
REUSE_CACHED_PROPOSAL_EMBEDDINGS = True
REUSE_CACHED_MAIN_IDEA_EMBEDDINGS = True
REUSE_CACHED_LITERATURE_EMBEDDINGS = True


# Compare AI vs Human Research Proposals — Style-Controlled (Rephrased)

This notebook mirrors `compare_proposals_baseline.ipynb` but uses proposals rephrased by `gemini-2.0-flash` into a standardized neutral academic style, removing stylistic fingerprints before analysis.

**Data sources:**
- `data/ai-proposals/rephrased/ai_proposals_rephrased_*.csv`
- `data/human-proposals/rephrased/human_proposals_rephrased_y*.json`

All analyses (diversity, novelty, thematic, style baseline) are identical to the baseline notebook.

# Setup and Imports

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# NLP and embeddings
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from scipy.spatial.distance import cdist
from tqdm import tqdm

# Statistics
from scipy import stats
from scipy.stats import mannwhitneyu
import itertools

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11


# Bootstrap CI overlays and standardized boxplot grammar.
# Boxes show median/IQR/whiskers/fliers; jittered points show proposal-level observations;
# diamonds show bootstrapped mean with 95% CI. When proposal metadata are available,
# funded Human proposals receive a magenta ring and top-ranked proposals receive a black ring.
def bootstrap_mean_ci(values, n_boot=2000, random_state=42, alpha=0.05):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan, np.nan, np.nan
    mean = float(np.mean(arr))
    if len(arr) == 1:
        return mean, mean, mean
    rng = np.random.default_rng(random_state)
    boot_means = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        boot_means[b] = np.mean(rng.choice(arr, size=len(arr), replace=True))
    lo, hi = np.quantile(boot_means, [alpha / 2, 1 - alpha / 2])
    return mean, float(lo), float(hi)


def _palette_lookup(palette, key, fallback='#808080'):
    if isinstance(palette, dict):
        return palette.get(key, fallback)
    return fallback


def _rgba(color, alpha=1.0):
    import matplotlib.colors as mcolors
    return mcolors.to_rgba(color, alpha=alpha)


def _coerce_bool(value):
    if pd.isna(value):
        return False
    if isinstance(value, str):
        return value.strip().lower() in {'1', 'true', 'yes', 'y'}
    return bool(value)


def _augment_plot_metadata(data):
    df = data.copy()
    if 'proposal_uid' not in df.columns or 'proposal_meta' not in globals():
        return df
    meta_cols = [
        c for c in ['proposal_uid', 'funding', 'is_top5_ranked', 'ranking', 'ranking_AI_reviews']
        if c in proposal_meta.columns
    ]
    if len(meta_cols) <= 1:
        return df
    meta = proposal_meta[meta_cols].drop_duplicates('proposal_uid')
    for col in meta_cols:
        if col == 'proposal_uid' or col in df.columns:
            continue
        df = df.merge(meta[['proposal_uid', col]], on='proposal_uid', how='left')
    return df


FUNDED_HUMAN_RING_COLOR = '#FF00FF'


def _is_funded_human(row, group):
    group_binary = str(row.get('group_binary', ''))
    is_human = (str(group) == 'Human') or (group_binary == 'Human')
    if not is_human:
        return False
    funding = pd.to_numeric(row.get('funding', np.nan), errors='coerce')
    if np.isfinite(funding):
        return funding == 1
    funding_txt = str(row.get('funding', '')).strip().lower()
    return funding_txt in {'funded', 'yes', 'true'}


def _point_visual_style(row, group, palette):
    color = _palette_lookup(palette, group, '#808080')
    if str(row.get('group_binary', '')) == 'Human' or str(group) == 'Human':
        color = _palette_lookup(palette, 'Human', color)
    funded_human = _is_funded_human(row, group)
    top5 = _coerce_bool(row.get('is_top5_ranked', False))
    return color, funded_human, top5


def add_mean_ci_to_boxplot(ax, data_arrays, positions=None, colors_for_points=None, n_boot=2000,
                           random_state=42, marker_size=50, line_color='black', zorder=12):
    if positions is None:
        positions = np.arange(len(data_arrays))
    if colors_for_points is None:
        colors_for_points = ['#808080'] * len(data_arrays)
    for pos, values, point_color in zip(positions, data_arrays, colors_for_points):
        mean, lo, hi = bootstrap_mean_ci(values, n_boot=n_boot, random_state=random_state)
        if not np.isfinite(mean):
            continue
        yerr = np.array([[max(0.0, mean - lo)], [max(0.0, hi - mean)]])
        ax.errorbar(
            pos, mean, yerr=yerr,
            fmt='D', markersize=np.sqrt(marker_size),
            markerfacecolor=_rgba(point_color, 0.95), markeredgecolor=line_color, markeredgewidth=1.2,
            ecolor=line_color, elinewidth=1.4, capsize=4, capthick=1.2,
            zorder=zorder,
        )


def _scatter_box_points(ax, rows, x_center, group, y, palette, rng, jitter=0.15,
                        point_size=20, point_alpha=0.50, zorder=7):
    if len(rows) == 0:
        return
    offsets = rng.uniform(-jitter, jitter, size=len(rows))
    for offset, (_, row) in zip(offsets, rows.iterrows()):
        val = pd.to_numeric(row[y], errors='coerce')
        if not np.isfinite(val):
            continue
        color, funded_human, top5 = _point_visual_style(row, group, palette)
        x_pos = x_center + offset
        ax.scatter(
            x_pos, val,
            s=point_size,
            facecolors=[_rgba(color, point_alpha)],
            edgecolors='none',
            linewidths=0,
            zorder=zorder,
        )
        if funded_human:
            ax.scatter(
                x_pos, val,
                s=point_size + 34,
                facecolors='none',
                edgecolors=FUNDED_HUMAN_RING_COLOR,
                linewidths=1.7,
                zorder=zorder + 0.25,
            )
        if top5:
            ax.scatter(
                x_pos, val,
                s=point_size + 68,
                facecolors='none',
                edgecolors='black',
                linewidths=1.5,
                zorder=zorder + 0.45,
            )


def _add_boxplot_metadata_legend(ax, include_group_legend=False, group_order=None, palette=None,
                                 include_funding=True, include_top5=True, loc='best'):
    import matplotlib.patches as mpatches
    import matplotlib.lines as mlines
    handles = []
    if include_group_legend and group_order is not None:
        for g in group_order:
            handles.append(mpatches.Patch(facecolor=_rgba(_palette_lookup(palette, g), 0.70),
                                          edgecolor='black', label=str(g)))
    if include_funding:
        handles.append(
            mlines.Line2D([], [], marker='o', linestyle='None', markersize=7,
                          markerfacecolor='white', markeredgecolor=FUNDED_HUMAN_RING_COLOR,
                          markeredgewidth=1.7, label='Funded human')
        )
    if include_top5:
        handles.append(mlines.Line2D([], [], marker='o', linestyle='None', markersize=7,
                                     markerfacecolor='white', markeredgecolor='black', markeredgewidth=1.4,
                                     label='Top-ranked proposal'))
    if handles:
        ax.legend(handles=handles, fontsize=8, loc=loc, framealpha=0.88)


def styled_boxplot_with_points(ax, data, x, y, order=None, palette=None, positions=None,
                               box_width=0.45, jitter=0.15, point_size=20, point_alpha=0.50,
                               box_alpha=0.70, n_boot=2000, random_state=42,
                               show_metadata_legend=False, metadata_legend_loc='best'):
    df = _augment_plot_metadata(data)
    if order is None:
        order = [v for v in pd.unique(df[x]) if pd.notna(v)]
    if positions is None:
        positions = np.arange(len(order), dtype=float)
    arrays = [df.loc[df[x].eq(cat), y].dropna().to_numpy(dtype=float) for cat in order]
    bp = ax.boxplot(
        arrays,
        positions=positions,
        widths=box_width,
        patch_artist=True,
        showfliers=True,
        medianprops=dict(color='black', linewidth=2.0),
        whiskerprops=dict(color='black', linewidth=1.2),
        capprops=dict(color='black', linewidth=1.2),
        flierprops=dict(marker='o', markersize=3, markerfacecolor='white',
                        markeredgecolor='black', alpha=0.55, linestyle='none'),
    )
    for patch, cat in zip(bp['boxes'], order):
        patch.set_facecolor(_rgba(_palette_lookup(palette, cat), box_alpha))
        patch.set_edgecolor('black')
        patch.set_linewidth(1.2)
        patch.set_zorder(3)
    rng = np.random.default_rng(random_state)
    for pos, cat in zip(positions, order):
        rows = df.loc[df[x].eq(cat)]
        _scatter_box_points(ax, rows, pos, cat, y, palette, rng,
                            jitter=jitter, point_size=point_size, point_alpha=point_alpha)
    add_mean_ci_to_boxplot(
        ax, arrays, positions=positions,
        colors_for_points=[_palette_lookup(palette, cat) for cat in order],
        n_boot=n_boot, random_state=random_state, marker_size=50,
    )
    ax.set_xticks(positions)
    ax.set_xticklabels(order)
    if show_metadata_legend and 'proposal_uid' in df.columns:
        _add_boxplot_metadata_legend(ax, include_group_legend=False, include_funding=True,
                                     include_top5=True, loc=metadata_legend_loc)
    return bp


def styled_grouped_boxplot_with_points(ax, data, x, y, hue, order=None, hue_order=None,
                                        palette=None, width=0.82, jitter=0.15,
                                        point_size=20, point_alpha=0.50,
                                        box_alpha=0.70, n_boot=2000, random_state=42,
                                        show_group_legend=True, show_metadata_legend=False,
                                        legend_loc='best'):
    df = _augment_plot_metadata(data)
    if order is None:
        order = [v for v in pd.unique(df[x]) if pd.notna(v)]
    if hue_order is None:
        hue_order = [v for v in pd.unique(df[hue]) if pd.notna(v)]
    n_hue = max(1, len(hue_order))
    step = width / n_hue
    positions, arrays, group_keys = [], [], []
    for xi, cat in enumerate(order):
        for hi, hv in enumerate(hue_order):
            vals = df.loc[df[x].eq(cat) & df[hue].eq(hv), y].dropna().to_numpy(dtype=float)
            if len(vals) == 0:
                continue
            positions.append(xi - width / 2 + step * (hi + 0.5))
            arrays.append(vals)
            group_keys.append(hv)
    if arrays:
        bp = ax.boxplot(
            arrays,
            positions=positions,
            widths=min(step * 0.76, 0.22),
            patch_artist=True,
            showfliers=True,
            medianprops=dict(color='black', linewidth=1.8),
            whiskerprops=dict(color='black', linewidth=1.1),
            capprops=dict(color='black', linewidth=1.1),
            flierprops=dict(marker='o', markersize=2.5, markerfacecolor='white',
                            markeredgecolor='black', alpha=0.45, linestyle='none'),
        )
        for patch, hv in zip(bp['boxes'], group_keys):
            patch.set_facecolor(_rgba(_palette_lookup(palette, hv), box_alpha))
            patch.set_edgecolor('black')
            patch.set_linewidth(1.0)
            patch.set_zorder(3)
    rng = np.random.default_rng(random_state)
    point_jitter = min(jitter, step * 0.35)
    for xi, cat in enumerate(order):
        for hi, hv in enumerate(hue_order):
            pos = xi - width / 2 + step * (hi + 0.5)
            rows = df.loc[df[x].eq(cat) & df[hue].eq(hv)]
            _scatter_box_points(ax, rows, pos, hv, y, palette, rng,
                                jitter=point_jitter, point_size=point_size, point_alpha=point_alpha)
    add_mean_ci_to_boxplot(
        ax, arrays, positions=positions,
        colors_for_points=[_palette_lookup(palette, hv) for hv in group_keys],
        n_boot=n_boot, random_state=random_state, marker_size=50,
    )
    ax.set_xticks(np.arange(len(order)))
    ax.set_xticklabels(order)
    ax.set_xlim(-0.55, len(order) - 0.45)
    if show_group_legend:
        _add_boxplot_metadata_legend(
            ax, include_group_legend=True, group_order=hue_order, palette=palette,
            include_funding=show_metadata_legend and 'proposal_uid' in df.columns,
            include_top5=show_metadata_legend and 'proposal_uid' in df.columns,
            loc=legend_loc,
        )
    elif show_metadata_legend and 'proposal_uid' in df.columns:
        _add_boxplot_metadata_legend(ax, include_group_legend=False, include_funding=True,
                                     include_top5=True, loc=legend_loc)


# Backward-compatible alias used by older cells that only need the mean-CI overlay.
def add_grouped_mean_ci_to_boxplot(ax, data, x, y, hue=None, order=None, hue_order=None, palette=None,
                                   width=0.8, n_boot=2000, random_state=42, marker_size=42):
    if order is None:
        order = [v for v in pd.unique(data[x]) if pd.notna(v)]
    if hue is None:
        arrays = [data.loc[data[x].eq(cat), y].dropna().to_numpy(dtype=float) for cat in order]
        colors_for_points = [_palette_lookup(palette, cat, '#808080') for cat in order]
        add_mean_ci_to_boxplot(
            ax, arrays, positions=np.arange(len(order)), colors_for_points=colors_for_points,
            n_boot=n_boot, random_state=random_state, marker_size=marker_size,
        )
        return
    if hue_order is None:
        hue_order = [v for v in pd.unique(data[hue]) if pd.notna(v)]
    n_hue = max(1, len(hue_order))
    step = width / n_hue
    arrays, positions, colors_for_points = [], [], []
    for xi, cat in enumerate(order):
        for hi, hv in enumerate(hue_order):
            vals = data.loc[data[x].eq(cat) & data[hue].eq(hv), y].dropna().to_numpy(dtype=float)
            if len(vals) == 0:
                continue
            positions.append(xi - width / 2 + step * (hi + 0.5))
            arrays.append(vals)
            colors_for_points.append(_palette_lookup(palette, hv, '#808080'))
    add_mean_ci_to_boxplot(
        ax, arrays, positions=positions, colors_for_points=colors_for_points,
        n_boot=n_boot, random_state=random_state, marker_size=marker_size,
    )
def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    for candidate in candidates:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate.resolve()
    raise RuntimeError('Could not find project root containing src/ and data/.')


PROJECT_ROOT = find_project_root()

print('✓ Imports successful')
print(f'✓ Working directory: {os.getcwd()}')
print(f'✓ Project root: {PROJECT_ROOT}')
print(f'✓ PyTorch version: {torch.__version__}')
print(f'✓ CUDA available: {torch.cuda.is_available()}')

# Visualization: 2D Embedding Space with UMAP
try:
    import umap
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'umap-learn'])
    import umap

condition = 'rephrased/minimal'
AI_PROPOSALS_PATH = PROJECT_ROOT / 'data' / 'ai-proposals' / condition
HUMAN_PROPOSALS_PATH = PROJECT_ROOT / 'data' / 'human-proposals' / condition
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures' / condition
TABLES_DIR = RESULTS_DIR / 'tables' / condition
PREPARED_DIR = PROJECT_ROOT / 'data' / 'prepared' / condition
PREPARED_ALL_PROPOSALS_PATH = PREPARED_DIR / 'all_proposals.json'
PREPARED_ALL_PROPOSALS_CSV = PREPARED_DIR / 'all_proposals.csv'
PREPARED_LITERATURE_CORPUS_PATH = PREPARED_DIR / 'literature_corpus_prepared.json'
PROPOSAL_EMBEDDINGS_FILE = PROJECT_ROOT / 'data' / 'embeddings' / condition / 'proposal_embeddings_human_ai_rephrased.pkl'
ABSTRACT_EMBEDDINGS_FILE = PROJECT_ROOT / 'data' / 'embeddings' / condition / 'proposal_embeddings_rephrased_abstract.pkl'
MAIN_IDEA_EMBEDDINGS_FILE = PROJECT_ROOT / 'data' / 'embeddings' / condition / 'proposal_embeddings_main_idea_only.pkl'
LITERATURE_EMBEDDINGS_FILE = PROJECT_ROOT / 'data' / 'embeddings' / 'literature' / 'relevant_literature_embeddings.pkl'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
PROPOSAL_EMBEDDINGS_FILE.parent.mkdir(parents=True, exist_ok=True)
ABSTRACT_EMBEDDINGS_FILE.parent.mkdir(parents=True, exist_ok=True)
LITERATURE_EMBEDDINGS_FILE.parent.mkdir(parents=True, exist_ok=True)
MAIN_IDEA_EMBEDDINGS_FILE.parent.mkdir(parents=True, exist_ok=True)



## Helper Functions

In [ ]:
# Normalize raw model identifiers from data files to display names used throughout
MODEL_NAME_MAP = {
    'claude-opus-4-5': 'Claude',
    'gemini-3-pro-preview': 'Gemini',
    'gpt-5.2': 'GPT-5.2',
}

# Define DISTINCT colors for each group (keyed by display names)
colors = {
    'Human': '#DC143C',   # Crimson red (PROMINENT)
    'Claude': '#4A90E2',  # Blue
    'Gemini': '#7B68EE',  # Purple
    'GPT-5.2': '#3CB371', # Green
}


In [ ]:
def retrieve_embeddings(path_to_embeddings):
    with open(path_to_embeddings, 'rb') as f:
        embeddings_data = pickle.load(f)
    return embeddings_data


In [ ]:
def cliffs_delta(group1, group2):
    """
    Calculate Cliff's Delta effect size.

    Interpretation:
    - |δ| < 0.147: negligible
    - |δ| < 0.33: small
    - |δ| < 0.474: medium
    - |δ| ≥ 0.474: large
    """
    x = np.asarray(group1)
    y = np.asarray(group2)
    n1, n2 = len(x), len(y)

    dominance = 0
    for xi in x:
        dominance += np.sum(xi > y)
        dominance -= np.sum(xi < y)

    return dominance / (n1 * n2)


def interpret_cliffs_delta(delta):
    """Interpret Cliff's Delta magnitude."""
    abs_delta = abs(delta)
    if abs_delta < 0.147:
        return "negligible"
    if abs_delta < 0.33:
        return "small"
    if abs_delta < 0.474:
        return "medium"
    return "large"


def permutation_test(group1, group2, n_permutations=10000, random_state=42):
    """
    Permutation test for difference in means (group1 - group2).
    Returns p-value, observed difference, and null distribution.
    """
    rng = np.random.default_rng(random_state)

    group1 = np.asarray(group1)
    group2 = np.asarray(group2)
    obs_diff = np.mean(group1) - np.mean(group2)

    combined = np.concatenate([group1, group2])
    n1 = len(group1)

    perm_diffs = np.empty(n_permutations, dtype=float)
    for i in range(n_permutations):
        perm = rng.permutation(combined)
        perm_diffs[i] = np.mean(perm[:n1]) - np.mean(perm[n1:])

    p_value = (np.sum(np.abs(perm_diffs) >= abs(obs_diff)) + 1) / (n_permutations + 1)
    return p_value, obs_diff, perm_diffs


def bootstrap_mean_diff_ci(group1, group2, n_boot=5000, random_state=42, alpha=0.05):
    """Bootstrap CI for mean(group1) - mean(group2)."""
    rng = np.random.default_rng(random_state)
    g1 = np.asarray(group1)
    g2 = np.asarray(group2)

    boots = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        b1 = rng.choice(g1, size=len(g1), replace=True)
        b2 = rng.choice(g2, size=len(g2), replace=True)
        boots[i] = np.mean(b1) - np.mean(b2)

    lo = np.quantile(boots, alpha / 2)
    hi = np.quantile(boots, 1 - alpha / 2)
    return lo, hi, boots


def run_group_comparison(group1, group2, n_permutations=10000, n_boot=5000, random_state=42):
    """Unified comparison with MW, Cliff's delta, permutation p, and bootstrap CI."""
    u_stat, p_value_mw = mannwhitneyu(group1, group2, alternative='two-sided')
    delta = cliffs_delta(group1, group2)
    delta_interp = interpret_cliffs_delta(delta)
    p_value_perm, obs_diff, _ = permutation_test(
        group1, group2, n_permutations=n_permutations, random_state=random_state
    )
    ci_low, ci_high, _ = bootstrap_mean_diff_ci(
        group1, group2, n_boot=n_boot, random_state=random_state
    )

    return {
        'u_stat': u_stat,
        'p_value_mw': p_value_mw,
        'delta': delta,
        'delta_interp': delta_interp,
        'p_value_perm': p_value_perm,
        'obs_diff_mean': obs_diff,
        'obs_diff_median': np.median(group1) - np.median(group2),
        'ci_low': ci_low,
        'ci_high': ci_high,
    }


def apply_multiple_testing(results_df, p_cols=('p_value_mw', 'p_value_perm'), method='holm'):
    """Apply multiple-testing correction to selected p-value columns."""
    from statsmodels.stats.multitest import multipletests

    out = results_df.copy()
    for col in p_cols:
        if col in out.columns:
            _, p_adj, _, _ = multipletests(out[col].values, method=method)
            out[f'{col}_adj_{method}'] = p_adj
    return out


def proposal_mean_pairwise_distances(embeddings):
    """Per-proposal diversity proxy: mean cosine distance to all other proposals in same group."""
    dist_matrix = cosine_distances(embeddings)
    np.fill_diagonal(dist_matrix, np.nan)
    return np.nanmean(dist_matrix, axis=1)


print("✓ Helper functions defined")
print("  - Added proposal-level pairwise diversity metric (mean distance-to-others)")
print("  - Added bootstrap 95% CI for mean differences")
print("  - Added Holm multiple-testing adjustment helper")


In [ ]:

# --- Added helper utilities for proposal diversity/novelty patch ---
def build_group_indices(proposal_meta):
    groups = {}
    groups['Human'] = proposal_meta.index[proposal_meta['group_binary'] == 'Human'].to_numpy()
    model_names = sorted(proposal_meta.loc[proposal_meta['group_binary']=='AI', 'group_model'].dropna().unique().tolist())
    for m in model_names:
        groups[m] = proposal_meta.index[proposal_meta['group_model'] == m].to_numpy()
    groups['All AI'] = proposal_meta.index[proposal_meta['group_binary'] == 'AI'].to_numpy()
    return groups


def get_group_submatrix(D, idx):
    return D[np.ix_(idx, idx)]


def sort_distance_rows(D):
    idx = np.argsort(D, axis=1)
    dist = np.take_along_axis(D, idx, axis=1)
    return dist, idx


def safe_percentile(arr, q):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    return float(np.percentile(arr, q))


def proposal_mean_pairwise_from_submatrix(Dg):
    Dg = np.asarray(Dg, dtype=float).copy()
    np.fill_diagonal(Dg, np.nan)
    return np.nanmean(Dg, axis=1)


def proposal_min_pairwise_from_submatrix(Dg):
    Dg = np.asarray(Dg, dtype=float).copy()
    np.fill_diagonal(Dg, np.inf)
    return np.min(Dg, axis=1)


def proposal_mean_knn_from_submatrix(Dg, k=5):
    Dg = np.asarray(Dg, dtype=float).copy()
    np.fill_diagonal(Dg, np.inf)
    k_eff = max(1, min(k, Dg.shape[1]-1))
    part = np.partition(Dg, kth=k_eff-1, axis=1)[:, :k_eff]
    return np.mean(part, axis=1)


def proposal_centroid_distances(Xg, leave_one_out=False):
    Xg = np.asarray(Xg, dtype=float)
    if len(Xg) == 0:
        return np.array([])
    if not leave_one_out:
        c = Xg.mean(axis=0, keepdims=True)
        return cosine_distances(Xg, c).ravel()
    n = Xg.shape[0]
    if n <= 1:
        return np.zeros(n)
    d = np.zeros(n, dtype=float)
    sum_x = Xg.sum(axis=0)
    for i in range(n):
        c_loo = ((sum_x - Xg[i]) / (n - 1)).reshape(1, -1)
        d[i] = cosine_distances(Xg[i].reshape(1, -1), c_loo)[0, 0]
    return d


def proposal_global_centroid_distances(X_all, idx_group):
    X_all = np.asarray(X_all, dtype=float)
    c = X_all.mean(axis=0, keepdims=True)
    return cosine_distances(X_all[idx_group], c).ravel()


def proposal_medoid_distances(Dg):
    medoid_idx = group_medoid_index(Dg)
    return np.asarray(Dg)[:, medoid_idx]


def group_remote_clique(Dg):
    Dg = np.asarray(Dg, dtype=float)
    return float(Dg.mean())


def group_chamfer(Dg):
    Dg = np.asarray(Dg, dtype=float).copy()
    np.fill_diagonal(Dg, np.inf)
    return float(np.min(Dg, axis=1).mean())


def group_mst_dispersion(Dg):
    from scipy.sparse.csgraph import minimum_spanning_tree
    Dg = np.asarray(Dg, dtype=float)
    if Dg.shape[0] <= 1:
        return 0.0
    mst = minimum_spanning_tree(Dg)
    return float(mst.sum() / max(1, Dg.shape[0]-1))


def group_span_percentile(Xg, q=90):
    d = proposal_centroid_distances(Xg, leave_one_out=False)
    return safe_percentile(d, q)


def group_medoid_index(Dg):
    Dg = np.asarray(Dg, dtype=float)
    return int(np.argmin(Dg.sum(axis=0)))


def group_sparseness(Dg):
    medoid_idx = group_medoid_index(Dg)
    return float(np.asarray(Dg, dtype=float)[:, medoid_idx].mean())


def group_grid_entropy(coords_2d, idx_group, bins=5, normalize=True):
    pts = np.asarray(coords_2d)[idx_group]
    if len(pts) == 0:
        return np.nan
    H, _, _ = np.histogram2d(pts[:,0], pts[:,1], bins=bins)
    p = H.ravel().astype(float)
    if p.sum() == 0:
        return 0.0
    p = p / p.sum()
    p = p[p > 0]
    ent = -np.sum(p * np.log(p))
    if normalize:
        max_ent = np.log(bins * bins)
        return float(ent / max_ent) if max_ent > 0 else 0.0
    return float(ent)


def compute_element_novel_percentiles(D_pl, q_list=[0,1,5,10]):
    out = {}
    out['element_novel_0'] = np.min(D_pl, axis=1)
    for q in q_list:
        if q == 0:
            continue
        out[f'element_novel_{q}'] = np.percentile(D_pl, q, axis=1)
    return out


def compute_mean_knn_novelty(D_pl_sorted_dist, ks=[5,10,20,50]):
    out = {}
    n_lit = D_pl_sorted_dist.shape[1]
    for k in ks:
        k_eff = min(k, n_lit)
        out[f'mean_knn_{k}'] = D_pl_sorted_dist[:, :k_eff].mean(axis=1)
    return out


def compute_local_density_normalized_novelty(mean_knn_10, proposal_top10_lit_idx, lit_mean_knn_10):
    lit_local = lit_mean_knn_10[proposal_top10_lit_idx].mean(axis=1)
    novelty_ratio = mean_knn_10 / np.where(lit_local == 0, np.nan, lit_local)
    novelty_z = (mean_knn_10 - lit_local) / np.where(np.nanstd(lit_local) == 0, np.nan, np.nanstd(lit_local))
    return novelty_ratio, novelty_z, lit_local


def bootstrap_group_metric(X_or_D, metric_fn, n_boot=5000, random_state=42, metric_type='distance'):
    rng = np.random.default_rng(random_state)
    X_or_D = np.asarray(X_or_D)
    n = X_or_D.shape[0]
    vals = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if metric_type == 'distance':
            sample = X_or_D[np.ix_(idx, idx)]
        else:
            sample = X_or_D[idx]
        vals[b] = metric_fn(sample)
    return {
        'boot_mean': float(np.mean(vals)),
        'ci_low': float(np.percentile(vals, 2.5)),
        'ci_high': float(np.percentile(vals, 97.5)),
        'boot_values': vals,
    }


def permutation_test_group_metric(X_prop, labels, idx_a, idx_b, metric_fn, n_perm=10000, random_state=42, use_distance_submat=False):
    rng = np.random.default_rng(random_state)
    X_prop = np.asarray(X_prop)
    idx_a = np.asarray(idx_a)
    idx_b = np.asarray(idx_b)
    idx = np.concatenate([idx_a, idx_b])
    n_a = len(idx_a)
    if use_distance_submat:
        D = X_prop
        obs = metric_fn(D[np.ix_(idx_a, idx_a)]) - metric_fn(D[np.ix_(idx_b, idx_b)])
    else:
        obs = metric_fn(X_prop[idx_a]) - metric_fn(X_prop[idx_b])
    perm = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        shuf = rng.permutation(idx)
        a = shuf[:n_a]
        b = shuf[n_a:]
        if use_distance_submat:
            perm[i] = metric_fn(D[np.ix_(a, a)]) - metric_fn(D[np.ix_(b, b)])
        else:
            perm[i] = metric_fn(X_prop[a]) - metric_fn(X_prop[b])
    p = (np.sum(np.abs(perm) >= abs(obs)) + 1) / (n_perm + 1)
    return {'obs_diff': float(obs), 'perm_p_value': float(p)}


def flag_top_percentile(values, pct=90, strict=True):
    values = np.asarray(values, dtype=float)
    thr = np.percentile(values[np.isfinite(values)], pct)
    if strict:
        return values > thr, float(thr)
    return values >= thr, float(thr)


def fisher_group_prevalence_tests(flag_series, group_series, reference_group='Human'):
    from scipy.stats import fisher_exact
    rows = []
    flags = np.asarray(flag_series).astype(bool)
    groups = np.asarray(group_series).astype(str)
    ref_mask = groups == reference_group
    ref_pos = int(flags[ref_mask].sum())
    ref_neg = int(ref_mask.sum() - ref_pos)
    for g in sorted(set(groups)):
        if g == reference_group:
            continue
        m = groups == g
        pos = int(flags[m].sum())
        neg = int(m.sum() - pos)
        table = np.array([[pos, neg], [ref_pos, ref_neg]])
        try:
            odds, p = fisher_exact(table)
        except Exception:
            odds, p = np.nan, np.nan
        rows.append({
            'group': g,
            'reference_group': reference_group,
            'n_group': int(m.sum()),
            'n_reference': int(ref_mask.sum()),
            'flag_rate_group': pos / max(1, int(m.sum())),
            'flag_rate_reference': ref_pos / max(1, int(ref_mask.sum())),
            'odds_ratio': odds,
            'p_value': p,
            'count_group_flag': pos,
            'count_ref_flag': ref_pos,
        })
    return pd.DataFrame(rows)


## Load Prepared Proposal Data

Run `prepare_data_for_analysis.ipynb` first when raw proposal files change. This notebook expects the prepared proposal artifact and full-proposal embeddings produced there.

In [ ]:
# Load prepared proposal records and expose the same ai_df/human_df variables used below.
prepared_path = PREPARED_ALL_PROPOSALS_PATH if PREPARED_ALL_PROPOSALS_PATH.exists() else TABLES_DIR / 'all_proposals.json'
if not prepared_path.exists():
    raise FileNotFoundError(
        f'Prepared proposals not found at {prepared_path}. Run prepare_data_for_analysis.ipynb first.'
    )

with open(prepared_path) as f:
    prepared_proposals = json.load(f)

ai_rows = []
human_rows = []
for rec in prepared_proposals:
    rephrased = rec.get('rephrased', {}) or {}
    row = {
        'title': rec.get('title'),
        'proposal_title': rec.get('title'),
        'model': rec.get('model'),
        'cohort': rec.get('cohort'),
        'source_file': rec.get('source_file'),
        'standardized_text': rephrased.get('standardized_text', ''),
        'abstract_text': rephrased.get('abstract_text', ''),
        'main_idea': rephrased.get('main_idea', ''),
        'group': 'AI' if rec.get('is_ai') else 'Human',
    }
    if rec.get('is_ai'):
        ai_rows.append(row)
    else:
        human_rows.append(row)

ai_df = pd.DataFrame(ai_rows)
human_df = pd.DataFrame(human_rows)

print(f'Loaded prepared proposals from: {prepared_path}')
print(f'  AI proposals: {len(ai_df)}')
print(f'  Human proposals: {len(human_df)}')
print(f"  AI models: {ai_df['model'].value_counts().to_dict()}")


## Load Prepared NCEMS Reviews

Load human + AI NCEMS review data (`ncems_criteria_all_reviews.csv`) and aggregate proposal-level ranking/funding fields for embedding-space visualizations.

In [ ]:

# Proposal-level ranking, funding, and cohort are now in all_proposals.json
# (moved there from the reviews CSV; the reviews CSV contains only review-level rows).
with open(PREPARED_ALL_PROPOSALS_PATH) as f:
    _all_props_meta = json.load(f)

_prop_rows = []
for r in _all_props_meta:
    is_ai = bool(r.get('is_ai', False))
    _prop_rows.append({
        'title': r.get('title', ''),
        'is_ai': is_ai,
        'cohort': r.get('cohort'),
        'group_binary': 'AI' if is_ai else 'Human',
        'group_model': (r.get('model') or r.get('group', '')) if is_ai else 'Human',
        'ranking': r.get('ranking'),
        'ranking_AI_reviews': r.get('ranking_AI_reviews'),
        'funding': r.get('funding'),
    })

review_rankings_df = pd.DataFrame(_prop_rows)
review_rankings_df['title_norm'] = (
    review_rankings_df['title'].astype(str)
    .str.strip().str.lower()
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
for c in ['ranking', 'ranking_AI_reviews', 'funding']:
    review_rankings_df[c] = pd.to_numeric(review_rankings_df[c], errors='coerce')

print(f'Loaded proposal metadata from: {PREPARED_ALL_PROPOSALS_PATH}')
print(f'  proposals: {len(review_rankings_df)}')
print(f'  with ranking (panel): {review_rankings_df["ranking"].notna().sum()}')
print(f'  with ranking_AI_reviews: {review_rankings_df["ranking_AI_reviews"].notna().sum()}')
print(f'  with funding: {review_rankings_df["funding"].notna().sum()}')

## Prepare Proposal Texts

Combine all sections of proposals into full text for embedding.

In [ ]:
import re

# Section headers produced by rephrase_proposals.py template
_SECTION_HEADERS = [
    "SCIENTIFIC BACKGROUND AND RESEARCH QUESTION",
    "METHODOLOGY AND ANALYTICAL APPROACH",
    "DATA SOURCES AND SYNTHESIS PLAN",
    "FEASIBILITY AND TIMELINE",
    "OPEN SCIENCE AND TEAM COMPOSITION",
]

def strip_section_headers(text: str) -> str:
    """Remove section heading lines, keeping only prose content."""
    for header in _SECTION_HEADERS:
        text = re.sub(rf'^\s*{re.escape(header)}\s*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    return re.sub(r'\n{3,}', '\n\n', text).strip()

# Both AI and human proposals now share a unified "standardized_text" field
# produced by rephrase_proposals.py — strip section headers to keep only prose.
ai_df["full_text"] = ai_df["standardized_text"].fillna("").astype(str).apply(strip_section_headers)
human_df["full_text"] = human_df["standardized_text"].fillna("").astype(str).apply(strip_section_headers)

# Add group labels
ai_df["group"] = "AI"
human_df["group"] = "Human"

print(f"✓ Using standardized_text (headers stripped) as full_text for all proposals")
print(f"  AI avg length: {ai_df['full_text'].str.len().mean():.0f} characters")
print(f"  Human avg length: {human_df['full_text'].str.len().mean():.0f} characters")


## Load Prepared Full-Proposal Embeddings

Full rephrased proposal embeddings are prepared in `prepare_data_for_analysis.ipynb`. This notebook only computes them as a fallback when the cache is missing or stale.

In [ ]:
# Embedding model is loaded lazily only if a required cache is missing.
model_name = 'michiyasunaga/BioLinkBERT-large'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = None
embedding_model = None


def ensure_embedding_model_loaded():
    global tokenizer, embedding_model
    if tokenizer is not None and embedding_model is not None:
        return tokenizer, embedding_model
    print(f'Loading BioLinkBERT model: {model_name}')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    embedding_model = AutoModel.from_pretrained(model_name).to(device)
    embedding_model.eval()
    print(f'Model loaded on: {device}')
    return tokenizer, embedding_model


In [ ]:
def get_embeddings(texts, batch_size=8):
    """Generate BioLinkBERT [CLS] embeddings for texts."""
    ensure_embedding_model_loaded()
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc='Embedding texts'):
            batch = ['' if pd.isna(t) else str(t) for t in texts[i:i + batch_size]]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors='pt',
            ).to(device)
            outputs = embedding_model(**encoded)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

print('Embedding function ready; model will load only if a cache must be generated.')


In [ ]:
# Load full rephrased proposal embeddings (or generate a fallback cache).
embeddings_file = PROPOSAL_EMBEDDINGS_FILE
rephrased_embeddings_were_generated = False

if embeddings_file.exists():
    print(f'Loading prepared full-proposal embeddings from: {embeddings_file}')
    embeddings_data = retrieve_embeddings(str(embeddings_file))
    ai_embeddings = np.asarray(embeddings_data['ai_embeddings'])
    human_embeddings = np.asarray(embeddings_data['human_embeddings'])
    ai_metadata = embeddings_data.get('ai_metadata', ai_df[['model', 'title', 'group']].to_dict('records'))
    human_metadata = embeddings_data.get('human_metadata', human_df[['proposal_title', 'group', 'source_file']].to_dict('records'))
else:
    print('No prepared full-proposal embedding cache found - generating fallback now.')
    ai_embeddings = get_embeddings(ai_df['full_text'].tolist())
    human_embeddings = get_embeddings(human_df['full_text'].tolist())
    ai_metadata = ai_df[['model', 'title', 'group']].to_dict('records')
    human_metadata = human_df[['proposal_title', 'group', 'source_file']].to_dict('records')
    embeddings_data = {
        'ai_embeddings': ai_embeddings,
        'human_embeddings': human_embeddings,
        'ai_metadata': ai_metadata,
        'human_metadata': human_metadata,
        'model_name': model_name,
        'text_field': 'rephrased_full_text',
        'timestamp': datetime.now().isoformat(),
    }
    with open(embeddings_file, 'wb') as f:
        pickle.dump(embeddings_data, f)
    rephrased_embeddings_were_generated = True
    print(f'Saved fallback embeddings to: {embeddings_file}')

print(f'  AI embeddings: {ai_embeddings.shape}')
print(f'  Human embeddings: {human_embeddings.shape}')


In [ ]:
# Save: proposal metadata (title + group)
import pandas as pd; from pathlib import Path
out_dir = TABLES_DIR; out_dir.mkdir(parents=True, exist_ok=True)
rows = ([{'title': r.get('proposal_title', r.get('title','')), 'group': 'Human'} for r in human_metadata]
      + [{'title': r.get('title', r.get('proposal_title','')), 'group': r.get('model','AI')} for r in ai_metadata])
pd.DataFrame(rows).to_csv(out_dir/'proposal_metadata.csv', index=False)
print(f"Saved proposal_metadata.csv  ({len(rows)} rows)")


## Shared Distance-Matrix Precomputation

This block computes and caches the reusable proposal-space and literature-space distance objects used throughout Part I and Part II. All later sections should reuse these objects rather than recomputing distances.

In [ ]:

# Canonical proposal metadata in embedding order
human_titles = [str(r.get('proposal_title', r.get('title', ''))) for r in human_metadata]
ai_titles = [str(r.get('title', r.get('proposal_title', ''))) for r in ai_metadata]
ai_models_ordered = [str(r.get('model', 'AI')) for r in ai_metadata]

proposal_meta = pd.DataFrame({
    'title': human_titles + ai_titles,
    'group_model': ['Human'] * len(human_titles) + ai_models_ordered,
    'group_binary': ['Human'] * len(human_titles) + ['AI'] * len(ai_titles),
    'is_ai': [False] * len(human_titles) + [True] * len(ai_titles),
    'source_file': [r.get('source_file', None) for r in human_metadata] + [r.get('source_file', None) for r in ai_metadata],
})
proposal_meta['proposal_uid'] = [f'P_{i:03d}' for i in range(len(proposal_meta))]
proposal_meta['title_norm'] = (
    proposal_meta['title'].astype(str).str.strip().str.lower()
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Merge proposal-level review ranking/funding fields
if 'review_rankings_df' in globals():
    proposal_meta = proposal_meta.merge(
        review_rankings_df[['title_norm', 'group_binary', 'group_model', 'ranking', 'ranking_AI_reviews', 'funding']].copy(),
        on=['title_norm', 'group_binary', 'group_model'],
        how='left',
    )
else:
    proposal_meta['ranking'] = np.nan
    proposal_meta['ranking_AI_reviews'] = np.nan
    proposal_meta['funding'] = np.nan

# Normalize raw model identifiers AFTER merge so join keys match
proposal_meta['group_model'] = proposal_meta['group_model'].map(lambda x: MODEL_NAME_MAP.get(x, x))

# Unified rank field: Human uses ranking, AI uses ranking_AI_reviews
proposal_meta['viz_rank'] = np.where(
    proposal_meta['group_binary'].eq('Human'),
    proposal_meta['ranking'],
    proposal_meta['ranking_AI_reviews'],
)
proposal_meta['viz_rank'] = pd.to_numeric(proposal_meta['viz_rank'], errors='coerce')

# Discard size variance: use constant marker size for all proposals.
# Highlight top-ranked proposals via black outline instead.
MARKER_SIZE_CONST = 145.0
proposal_meta['viz_marker_size'] = MARKER_SIZE_CONST
proposal_meta['is_top5_ranked'] = False
for _grp in proposal_meta['group_model'].unique():
    _mask = proposal_meta['group_model'] == _grp
    _ranks = proposal_meta.loc[_mask, 'viz_rank']
    if _ranks.notna().any():
        _top5_idx = _ranks.dropna().nsmallest(5).index
        proposal_meta.loc[_top5_idx, 'is_top5_ranked'] = True

# Human color shading by funding status
proposal_meta['human_color_shade'] = np.where(
    proposal_meta['funding'].fillna(0).astype(float).eq(1.0),
    '#8B0000',  # dark red for funded
    '#F08080',  # lighter coral for non-funded/unknown
)

# X_prop in exact row order of proposal_meta
X_prop = np.vstack([human_embeddings, ai_embeddings]).astype(np.float32)

# L2 normalize once
X_prop = X_prop / np.clip(np.linalg.norm(X_prop, axis=1, keepdims=True), 1e-12, None)

GROUPS = build_group_indices(proposal_meta)
ai_models = sorted(proposal_meta.loc[proposal_meta['group_binary'] == 'AI', 'group_model'].unique().tolist())

# proposal-proposal cosine distance
S_pp = X_prop @ X_prop.T
D_pp = 1.0 - S_pp
np.fill_diagonal(D_pp, 0.0)

D_pp_infdiag = D_pp.copy()
np.fill_diagonal(D_pp_infdiag, np.inf)

# deterministic 2D reference
from sklearn.decomposition import PCA
pca_2d = PCA(n_components=2, random_state=42).fit_transform(X_prop)

group_cache = {}
for g, idx in GROUPS.items():
    Dg = D_pp[np.ix_(idx, idx)]
    Xg = X_prop[idx]
    group_cache[g] = {'idx': idx, 'D': Dg, 'X': Xg, 'n': len(idx)}

cache_dir = TABLES_DIR / 'cached'
cache_dir.mkdir(parents=True, exist_ok=True)
np.save(cache_dir / 'proposal_distance_matrix.npy', D_pp)
np.save(cache_dir / 'proposal_pca2d.npy', pca_2d)
proposal_meta.to_csv(cache_dir / 'proposal_meta.csv', index=False)

review_cov = proposal_meta['viz_rank'].notna().mean() * 100
top5_cov = proposal_meta['is_top5_ranked'].mean() * 100
fund_cov = proposal_meta.loc[proposal_meta['group_binary']=='Human', 'funding'].notna().mean() * 100
print('Saved shared proposal caches to:', cache_dir)
print('X_prop shape:', X_prop.shape)
print('D_pp shape:', D_pp.shape)
print('Groups:', {k: len(v) for k, v in GROUPS.items()})
print(f'Review ranking coverage (all proposals): {review_cov:.1f}%')
print(f'Top-5 outlined proposals: {top5_cov:.1f}% of all proposals')
print(f'Human funding coverage: {fund_cov:.1f}%')


# PART I: THEMATIC AND CLUSTER ANALYSIS

Examine whether human and AI proposals cluster in distinct semantic regions and differ in thematic coverage.

**⚠️ Sample Size Note:** With n=23 human and n=69 AI proposals, we use conservative approaches with strong regularization and permutation-based validation.

In [ ]:
from collections import Counter
import math
import re
from sklearn.feature_extraction.text import CountVectorizer

PROPOSAL_LABEL_STRATEGY = 'contrastive_phrase_v4'
PROPOSAL_LABEL_GENERIC_SINGLETONS = {
    'cell', 'cells', 'protein', 'proteins', 'gene', 'genes', 'expression', 'expressions',
    'cancer', 'cancers', 'disease', 'diseases', 'patient', 'patients', 'clinical',
    'study', 'studies', 'research', 'result', 'results', 'review', 'reviews',
    'project', 'proposal', 'title', 'abstract', 'background', 'method', 'methods',
    'using', 'based', 'model', 'models', 'data', 'analysis', 'potential', 'outcomes',
}


def _label_tokens(term):
    return re.findall(r'[a-z0-9]+', str(term).lower())


def _canonical_label_term(term):
    tokens = _label_tokens(term)
    canon = []
    for tok in tokens:
        if len(tok) > 4 and tok.endswith('s'):
            tok = tok[:-1]
        canon.append(tok)
    return ' '.join(canon)


def _is_contrastive_label_candidate(term):
    term = str(term).strip().lower()
    tokens = _label_tokens(term)
    if not term or len(term) < 3 or not tokens:
        return False
    canon = _canonical_label_term(term)
    # Drop generic single-word labels, but keep specific phrases such as "therapeutic potential".
    if len(tokens) == 1 and (tokens[0] in PROPOSAL_LABEL_GENERIC_SINGLETONS or canon in PROPOSAL_LABEL_GENERIC_SINGLETONS):
        return False
    if all(tok in PROPOSAL_LABEL_GENERIC_SINGLETONS for tok in tokens):
        return False
    return sum(ch.isalpha() for ch in term) >= 3


def _phrase_bonus(term):
    n = len(_label_tokens(term))
    if n >= 3:
        return 1.45
    if n == 2:
        return 1.30
    return 0.72


def _token_overlap(a, b):
    ta, tb = set(_label_tokens(a)), set(_label_tokens(b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


def build_contrastive_phrase_labels(texts, labels, label_order=None, n_terms=4, min_df=1, max_df=0.85,
                                    max_features=20000, ngram_range=(1, 3), prefix='Region'):
    """Region-vs-rest phrase labels, reused for LDA topics and semantic clusters."""
    texts = ['' if pd.isna(t) else str(t) for t in texts]
    labels = np.asarray(labels)
    if label_order is None:
        label_order = sorted(pd.Series(labels).dropna().unique().tolist(), key=str)
    label_order = list(label_order)

    vectorizer = CountVectorizer(
        stop_words='english', ngram_range=ngram_range, min_df=min_df,
        max_df=max_df, max_features=max_features,
    )
    X = vectorizer.fit_transform(texts)
    terms = np.asarray(vectorizer.get_feature_names_out())
    if X.shape[1] == 0:
        label_map = {lab: f'{prefix} {lab}' for lab in label_order}
        info = pd.DataFrame({'region': label_order, 'display_label': [label_map[x] for x in label_order]})
        return label_map, info

    X_bin = X.copy()
    X_bin.data = np.ones_like(X_bin.data)
    total_tf = np.asarray(X.sum(axis=0)).ravel().astype(float)
    total_df = np.asarray(X_bin.sum(axis=0)).ravel().astype(float)
    total_tokens = float(total_tf.sum())
    n_docs = X.shape[0]
    vocab_size = max(1, X.shape[1])

    region_stats = {}
    term_region_presence = np.zeros(vocab_size, dtype=int)
    for lab in label_order:
        mask = labels == lab
        n_region = int(mask.sum())
        if n_region == 0:
            continue
        region_tf = np.asarray(X[mask].sum(axis=0)).ravel().astype(float)
        region_df = np.asarray(X_bin[mask].sum(axis=0)).ravel().astype(float)
        min_region_docs = max(1, min(5, int(math.ceil(n_region * 0.08))))
        term_region_presence += (region_df >= min_region_docs).astype(int)
        region_stats[lab] = {
            'n_region': n_region,
            'region_tf': region_tf,
            'region_df': region_df,
            'region_tokens': float(region_tf.sum()),
            'min_region_docs': min_region_docs,
        }

    candidates_by_region = {}
    for lab, stats in region_stats.items():
        n_region = stats['n_region']
        n_rest = max(1, n_docs - n_region)
        region_tf = stats['region_tf']
        region_df = stats['region_df']
        region_tokens = max(1.0, stats['region_tokens'])
        rest_tf = total_tf - region_tf
        rest_df = total_df - region_df
        rest_tokens = max(1.0, total_tokens - region_tokens)

        log_odds = np.log((region_tf + 0.5) / (region_tokens + 0.5 * vocab_size)) - np.log((rest_tf + 0.5) / (rest_tokens + 0.5 * vocab_size))
        prevalence_lift = np.log((region_df + 1.0) / (n_region + 2.0)) - np.log((rest_df + 1.0) / (n_rest + 2.0))
        support = np.log1p(region_df)
        base_score = (log_odds + 0.65 * prevalence_lift) * support

        candidate_idx = np.where((region_df >= stats['min_region_docs']) & (base_score > 0))[0]
        candidates = []
        for j in candidate_idx:
            term = str(terms[j])
            if not _is_contrastive_label_candidate(term):
                continue
            overlap_penalty = 1.0 / math.sqrt(max(1, int(term_region_presence[j])))
            score = float(base_score[j]) * _phrase_bonus(term) * overlap_penalty
            candidates.append({
                'term': term,
                'canon': _canonical_label_term(term),
                'score': score,
                'region_df': int(region_df[j]),
                'rest_df': int(rest_df[j]),
                'region_presence': int(term_region_presence[j]),
                'n_tokens': len(_label_tokens(term)),
            })
        candidates.sort(key=lambda d: (-d['score'], -d['n_tokens'], d['region_presence'], d['term']))
        candidates_by_region[lab] = candidates

    label_map = {}
    rows = []
    used_canons = set()
    ordered_by_size = sorted(label_order, key=lambda lab: region_stats.get(lab, {}).get('n_region', 0), reverse=True)
    for lab in ordered_by_size:
        candidates = candidates_by_region.get(lab, [])
        selected = []
        selected_canons = set()
        for min_tokens in [2, 1]:
            while len(selected) < n_terms:
                best = None
                best_mmr = None
                for cand in candidates:
                    if cand['n_tokens'] < min_tokens:
                        continue
                    if cand['canon'] in selected_canons or cand['canon'] in used_canons:
                        continue
                    redundancy = max([_token_overlap(cand['term'], s) for s in selected], default=0.0)
                    mmr = cand['score'] - 0.85 * redundancy * cand['score']
                    if best is None or mmr > best_mmr:
                        best = cand
                        best_mmr = mmr
                if best is None:
                    break
                selected.append(best['term'])
                selected_canons.add(best['canon'])
            if len(selected) >= max(2, min(n_terms, 3)):
                break
        if not selected:
            for cand in candidates:
                if cand['canon'] in selected_canons:
                    continue
                selected.append(cand['term'])
                selected_canons.add(cand['canon'])
                if len(selected) >= n_terms:
                    break
        display = ', '.join(selected[:n_terms]) if selected else f'{prefix} {lab}'
        label_map[lab] = display
        used_canons.update(selected_canons)
        rows.append({
            'region': lab,
            'n_documents': int(region_stats.get(lab, {}).get('n_region', 0)),
            'display_label': display,
            'contrastive_terms': '; '.join(selected[:n_terms]),
            'label_strategy': PROPOSAL_LABEL_STRATEGY,
        })

    # Restore requested label order in the info table.
    info = pd.DataFrame(rows)
    if len(info):
        info['_order'] = info['region'].map({lab: i for i, lab in enumerate(label_order)})
        info = info.sort_values('_order').drop(columns=['_order']).reset_index(drop=True)
    return label_map, info


def build_class_ctfidf_terms(texts, labels, label_order=None, n_terms=10, min_df=1, max_df=0.90,
                             max_features=20000, ngram_range=(1, 2)):
    """Small BERTopic-style c-TF-IDF representation for pre-defined regions."""
    texts = ['' if pd.isna(t) else str(t) for t in texts]
    labels = np.asarray(labels)
    if label_order is None:
        label_order = sorted(pd.Series(labels).dropna().unique().tolist(), key=str)
    label_order = list(label_order)
    vectorizer = CountVectorizer(stop_words='english', ngram_range=ngram_range, min_df=min_df, max_df=max_df, max_features=max_features)
    X = vectorizer.fit_transform(texts)
    terms = np.asarray(vectorizer.get_feature_names_out())
    class_tf = []
    for lab in label_order:
        mask = labels == lab
        class_tf.append(np.asarray(X[mask].sum(axis=0)).ravel().astype(float))
    class_tf = np.vstack(class_tf) if class_tf else np.zeros((0, X.shape[1]))
    class_len = np.clip(class_tf.sum(axis=1, keepdims=True), 1.0, None)
    tf = class_tf / class_len
    df = np.maximum((class_tf > 0).sum(axis=0), 1)
    avg_class_len = float(class_len.mean()) if len(class_len) else 1.0
    idf = np.log(1.0 + avg_class_len / df)
    ctfidf = tf * idf
    rows = []
    term_map = {}
    for row_i, lab in enumerate(label_order):
        if ctfidf.shape[1] == 0:
            top_terms = []
        else:
            idx = np.argsort(ctfidf[row_i])[-n_terms:][::-1]
            top_terms = [str(terms[j]) for j in idx if ctfidf[row_i, j] > 0]
        term_map[lab] = top_terms
        rows.append({'region': lab, 'ctfidf_terms': '; '.join(top_terms), 'ctfidf_label_raw': ', '.join(top_terms[:4])})
    return term_map, pd.DataFrame(rows)

print(f'Contrastive phrase-label helpers ready ({PROPOSAL_LABEL_STRATEGY}).')


## Analysis 1.1: Topic Modeling (LDA - Exploratory)

Use Latent Dirichlet Allocation with strong priors to identify thematic topics. LDA is more stable than BERTopic for small samples.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import KFold
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print("="*85)
print("TOPIC MODELING WITH LDA (EXPLORATORY)")
print("="*85)
print("Using normalized CONTENT text (title + abstract) to reduce formatting/template confounding.")

def _normalize_topic_text(text):
    text = str(text or "")
    text = re.sub(r'\b(title|abstract|background|methods?|research questions?|outcomes?|open science|budget)\s*:', ' ', text, flags=re.I)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

def _build_topic_text(row, is_ai=True):
    if is_ai:
        title = row.get('title', '')
        abstract = row.get('abstract', '')
    else:
        title = row.get('proposal_title', row.get('title', ''))
        abstract = row.get('abstract', '')
    return _normalize_topic_text(f"{title}. {abstract}")

ai_abstracts = [_build_topic_text(row, is_ai=True) for _, row in ai_df.iterrows()]
human_abstracts = [_build_topic_text(row, is_ai=False) for _, row in human_df.iterrows()]
all_abstracts = human_abstracts + ai_abstracts
all_titles = ([row.get('proposal_title', row.get('title', '')) for _, row in human_df.iterrows()]
              + [row.get('title', '') for _, row in ai_df.iterrows()])

source_labels = ['Human'] * len(human_abstracts) + ['AI'] * len(ai_abstracts)
model_labels = (['Human'] * len(human_abstracts)
                + [MODEL_NAME_MAP.get(m, m) for m in ai_df['model'].tolist()])

print()
print(f"✓ Prepared {len(all_abstracts)} topic-model texts")
print(f"  Human: {len(human_abstracts)}, AI: {len(ai_abstracts)}")

_unigram_probe = CountVectorizer(stop_words='english', ngram_range=(1, 1), min_df=2, max_df=1.0, max_features=5000)
_unigram_X = _unigram_probe.fit_transform(all_abstracts)
_unigram_terms = _unigram_probe.get_feature_names_out()
_unigram_df = np.asarray((_unigram_X > 0).mean(axis=0)).ravel()

auto_domain_unigram_stopwords = set(_unigram_terms[_unigram_df >= 0.50])
manual_domain_unigram_stopwords = {
    'cell', 'cells', 'protein', 'proteins', 'emergent', 'emergence', 'data', 'synthesizing', 'synthesis',
    'title', 'abstract', 'project', 'proposal', 'outcomes',
    'background', 'method', 'methods',
    'using', 'nup', '000', 'nan', 'fg', 'npc', 'ii', 'et'
}
domain_unigram_stopwords = auto_domain_unigram_stopwords | manual_domain_unigram_stopwords

_top_common_idx = np.argsort(_unigram_df)[-20:][::-1]
print()
print("Most common unigrams in corpus (doc frequency):")
for i in _top_common_idx:
    print(f"  {_unigram_terms[i]}: {_unigram_df[i]:.2f}")
print()
print(f"✓ Domain unigram stopwords (auto+manual): {len(domain_unigram_stopwords)}")

print()
print("Creating document-term matrix...")
vectorizer = CountVectorizer(max_features=2000, min_df=2, max_df=0.7, stop_words='english', ngram_range=(1, 2))
_doc_term_matrix_full = vectorizer.fit_transform(all_abstracts)
_feature_names_full = np.array(vectorizer.get_feature_names_out())

_unigram_mask = np.array([' ' not in t for t in _feature_names_full])
_drop_mask = _unigram_mask & np.isin(_feature_names_full, list(domain_unigram_stopwords))
_keep_mask = ~_drop_mask

doc_term_matrix = _doc_term_matrix_full[:, _keep_mask]
feature_names = _feature_names_full[_keep_mask]

print(f"✓ Document-term matrix: {doc_term_matrix.shape}")
print(f"  Documents: {doc_term_matrix.shape[0]}, Vocabulary: {doc_term_matrix.shape[1]}")
print(f"  Dropped domain unigrams: {_drop_mask.sum()} (kept bigrams intact)")

print()
print("Selecting LDA topic count with conservative cross-validated criteria...")
print("  Candidate k: 2..8; criterion: smallest k within 5% of best held-out perplexity among eligible models")

candidate_topic_counts = list(range(2, 9))
lda_k_rows = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)
for k in candidate_topic_counts:
    fold_perplexities = []
    fold_log_likelihoods = []
    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(np.arange(doc_term_matrix.shape[0]))):
        lda_cv = LatentDirichletAllocation(
            n_components=k,
            doc_topic_prior=0.5,
            topic_word_prior=0.5,
            max_iter=100,
            learning_method='batch',
            random_state=1000 + k * 10 + fold_idx,
            n_jobs=-1,
        )
        lda_cv.fit(doc_term_matrix[train_idx])
        fold_perplexities.append(float(lda_cv.perplexity(doc_term_matrix[test_idx])))
        fold_log_likelihoods.append(float(lda_cv.score(doc_term_matrix[test_idx])))

    lda_probe = LatentDirichletAllocation(
        n_components=k,
        doc_topic_prior=0.5,
        topic_word_prior=0.5,
        max_iter=100,
        learning_method='batch',
        random_state=42,
        n_jobs=-1,
    )
    dt_probe = lda_probe.fit_transform(doc_term_matrix)
    dominant_probe = dt_probe.argmax(axis=1)
    topic_counts_probe = np.bincount(dominant_probe, minlength=k)
    topic_mix_probe = topic_counts_probe / max(1, topic_counts_probe.sum())
    topic_entropy_probe = -np.sum(topic_mix_probe[topic_mix_probe > 0] * np.log(topic_mix_probe[topic_mix_probe > 0]))

    lda_k_rows.append({
        'k': k,
        'cv_perplexity_mean': float(np.mean(fold_perplexities)),
        'cv_perplexity_sd': float(np.std(fold_perplexities, ddof=1)),
        'cv_log_likelihood_mean': float(np.mean(fold_log_likelihoods)),
        'active_topics': int((topic_counts_probe > 0).sum()),
        'min_dominant_topic_count': int(topic_counts_probe.min()),
        'max_dominant_topic_frac': float(topic_counts_probe.max() / max(1, topic_counts_probe.sum())),
        'dominant_topic_entropy': float(topic_entropy_probe),
    })

lda_k_selection_df = pd.DataFrame(lda_k_rows)
_strict_eligible = lda_k_selection_df[
    (lda_k_selection_df['active_topics'] == lda_k_selection_df['k'])
    & (lda_k_selection_df['min_dominant_topic_count'] >= 3)
].copy()
if len(_strict_eligible) == 0:
    _strict_eligible = lda_k_selection_df[
        (lda_k_selection_df['active_topics'] >= lda_k_selection_df['k'] - 1)
        & (lda_k_selection_df['min_dominant_topic_count'] >= 2)
    ].copy()
if len(_strict_eligible) == 0:
    _strict_eligible = lda_k_selection_df.copy()

_best_perplexity = float(_strict_eligible['cv_perplexity_mean'].min())
_within_tolerance = _strict_eligible[_strict_eligible['cv_perplexity_mean'] <= _best_perplexity * 1.05].copy()
n_topics = int(_within_tolerance.sort_values(['k', 'cv_perplexity_mean']).iloc[0]['k'])
lda_k_selection_df['selected'] = lda_k_selection_df['k'].eq(n_topics)
lda_k_selection_df.to_csv(TABLES_DIR / 'lda_topic_k_selection.csv', index=False)

print(lda_k_selection_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"✓ Selected n_topics={n_topics} by conservative data-driven rule")
print(f"✓ Saved: {TABLES_DIR / 'lda_topic_k_selection.csv'}")

print()
print("Fitting final LDA with conservative parameters...")
print(f"  n_topics={n_topics}, alpha=0.5, beta=0.5 (strong regularization)")

lda_model = LatentDirichletAllocation(
    n_components=n_topics,
    doc_topic_prior=0.5,
    topic_word_prior=0.5,
    max_iter=100,
    learning_method='batch',
    random_state=42,
    n_jobs=-1
)
doc_topic_dist = lda_model.fit_transform(doc_term_matrix)

print("✓ LDA fitted")
print(f"  Perplexity: {lda_model.perplexity(doc_term_matrix):.2f}")
print(f"  Log-likelihood: {lda_model.score(doc_term_matrix):.2f}")

def display_topics(model, feature_names, n_top_words=10):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        topics.append(top_words)
    return topics

topics = display_topics(lda_model, feature_names, n_top_words=10)

print()
print("="*85)
print("TOP 10 WORDS PER TOPIC")
print("="*85)
for i, topic_words in enumerate(topics):
    print()
    print(f"Topic {i+1}: {', '.join(topic_words)}")

doc_topic_df = pd.DataFrame(doc_topic_dist, columns=[f'Topic_{i+1}' for i in range(n_topics)])
doc_topic_df['title'] = all_titles
doc_topic_df['source'] = source_labels
doc_topic_df['model'] = model_labels
doc_topic_df['group'] = ['Human' if s == 'Human' else 'AI' for s in source_labels]

doc_topic_df['dominant_topic'] = doc_topic_df[[f'Topic_{i+1}' for i in range(n_topics)]].idxmax(axis=1)
doc_topic_df['dominant_topic_prob'] = doc_topic_df[[f'Topic_{i+1}' for i in range(n_topics)]].max(axis=1)

# Contrastive display labels make the topic differences easier to see than repeated raw LDA top words.
lda_topic_ids = [f'Topic_{i+1}' for i in range(n_topics)]
lda_topic_label_map, lda_topic_label_info_df = build_contrastive_phrase_labels(
    all_abstracts,
    doc_topic_df['dominant_topic'].to_numpy(),
    label_order=lda_topic_ids,
    n_terms=4,
    min_df=1,
    max_df=0.85,
    prefix='Topic',
)
lda_raw_top_terms = {f'Topic_{i+1}': ', '.join(topics[i][:4]) for i in range(n_topics)}
lda_topic_label_info_df['raw_lda_top_terms'] = lda_topic_label_info_df['region'].map(lda_raw_top_terms)
lda_topic_label_info_df = lda_topic_label_info_df.rename(columns={'region': 'topic'})
lda_topic_axis_labels = {topic: f'{topic}: {lda_topic_label_map.get(topic, topic)}' for topic in lda_topic_ids}
doc_topic_df['dominant_topic_label'] = doc_topic_df['dominant_topic'].map(lda_topic_label_map)
lda_topic_label_info_df.to_csv(TABLES_DIR / 'lda_topic_contrastive_labels.csv', index=False)

print()
print("CONTRASTIVE LDA TOPIC DISPLAY LABELS")
print(lda_topic_label_info_df[['topic', 'n_documents', 'display_label', 'raw_lda_top_terms']].to_string(index=False))
print(f"✓ Saved: {TABLES_DIR / 'lda_topic_contrastive_labels.csv'}")

print()
print("="*85)
print("DOCUMENT-TOPIC DISTRIBUTION SUMMARY")
print("="*85)
print("Mean topic probabilities per document:")
for i in range(n_topics):
    print(f"  Topic {i+1}: {doc_topic_df[f'Topic_{i+1}'].mean():.3f} ± {doc_topic_df[f'Topic_{i+1}'].std():.3f}")

print()
print("Dominant topic assignment:")
print(doc_topic_df['dominant_topic'].value_counts().sort_index())

print()
print("⚠️ LABEL AS EXPLORATORY - Small sample size limits topic stability")
print("="*85)


In [ ]:
from scipy.optimize import linear_sum_assignment
from scipy.stats import chi2_contingency

print("="*85)
print("TOPIC STABILITY VALIDATION")
print("="*85)
print("Running LDA with 10 different random seeds to assess stability (with topic alignment)...")

n_stability_runs = 10
n_top_words = 10
stability_topics = []
stability_topic_dists = []

for seed in range(n_stability_runs):
    lda_temp = LatentDirichletAllocation(
        n_components=n_topics,
        doc_topic_prior=0.5,
        topic_word_prior=0.5,
        max_iter=100,
        learning_method='batch',
        random_state=seed,
        n_jobs=-1
    )
    lda_temp.fit(doc_term_matrix)

    topic_word = lda_temp.components_.astype(float)
    topic_word = topic_word / topic_word.sum(axis=1, keepdims=True)

    topics_temp = display_topics(lda_temp, feature_names, n_top_words=n_top_words)
    stability_topics.append(topics_temp)
    stability_topic_dists.append(topic_word)

print(f"✓ Completed {n_stability_runs} stability runs")

ref_words_by_topic = [set(ws) for ws in stability_topics[0]]
ref_dist = stability_topic_dists[0]

def _cosine_sim_matrix(A, B, eps=1e-12):
    A = A / (np.linalg.norm(A, axis=1, keepdims=True) + eps)
    B = B / (np.linalg.norm(B, axis=1, keepdims=True) + eps)
    return A @ B.T

per_topic_overlap_all = [[] for _ in range(n_topics)]
per_topic_cos_all = [[] for _ in range(n_topics)]

print()
print("Topic stability (aligned to reference run):")
for run_idx in range(1, n_stability_runs):
    run_dist = stability_topic_dists[run_idx]
    sim = _cosine_sim_matrix(run_dist, ref_dist)

    row_ind, col_ind = linear_sum_assignment(-sim)

    aligned_words = [None] * n_topics
    aligned_cos = np.zeros(n_topics, dtype=float)
    for r_topic, ref_topic in zip(row_ind, col_ind):
        aligned_words[ref_topic] = set(stability_topics[run_idx][r_topic])
        aligned_cos[ref_topic] = sim[r_topic, ref_topic]

    overlaps = []
    for k in range(n_topics):
        overlap = len(ref_words_by_topic[k] & aligned_words[k]) / max(1, len(ref_words_by_topic[k]))
        overlaps.append(overlap)
        per_topic_overlap_all[k].append(overlap)
        per_topic_cos_all[k].append(aligned_cos[k])

    print(f"  Run {run_idx}: mean top-{n_top_words} overlap={np.mean(overlaps):.1%}, mean cosine={np.mean(aligned_cos):.3f}")

print()
print("Per-topic stability (mean across aligned runs):")
for k in range(n_topics):
    print(
        f"  Topic {k+1}: {np.mean(per_topic_overlap_all[k]):.1%} ± {np.std(per_topic_overlap_all[k]):.1%} top-{n_top_words} overlap, "
        f"{np.mean(per_topic_cos_all[k]):.3f} ± {np.std(per_topic_cos_all[k]):.3f} cosine"
    )

print()
print("="*85)
print("TOPIC COUNT SENSITIVITY (k = 2..8)")
print("="*85)

sensitivity_rows = []
for k in [2, 3, 4, 5, 6, 7, 8]:
    lda_k = LatentDirichletAllocation(
        n_components=k,
        doc_topic_prior=0.5,
        topic_word_prior=0.5,
        max_iter=100,
        learning_method='batch',
        random_state=42,
        n_jobs=-1
    )
    dt_k = lda_k.fit_transform(doc_term_matrix)
    dominant_k = pd.Series(dt_k.argmax(axis=1)).map(lambda x: f'Topic_{x+1}')
    cont_k = pd.crosstab(dominant_k, pd.Series(source_labels, name='group'))
    chi2_k, p_k, _, _ = chi2_contingency(cont_k)

    sensitivity_rows.append({
        'k': k,
        'perplexity': float(lda_k.perplexity(doc_term_matrix)),
        'log_likelihood': float(lda_k.score(doc_term_matrix)),
        'chi2': float(chi2_k),
        'chi2_p': float(p_k),
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
print(sensitivity_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print()
print("💡 If overlap/cosine are still low AFTER alignment, that reflects true instability.")
print("💡 k-sensitivity helps verify conclusions are not artifacts of a single topic count.")
print("="*85)


## Analysis 1.2: Topic Distribution and Coverage Per Model

Test whether topic distributions differ **per model** (Human / Claude / Gemini / GPT-5.2) using permutation tests and Fisher exact tests. Shannon entropy per model is also computed here, absorbing the former Analysis 1.3 step.

In [ ]:
from scipy.stats import fisher_exact, chi2_contingency
from statsmodels.stats.multitest import multipletests

print("="*85)
print("ANALYSIS 1.2: TOPIC DISTRIBUTION PER MODEL  (soft assignment >20%)")
print("="*85)

threshold  = 0.20
topic_cols = sorted([c for c in doc_topic_df.columns if c.startswith('Topic_')])

# Enrich doc_topic_df with per-model labels if not present
if 'model' not in doc_topic_df.columns:
    _meta = proposal_meta[['title', 'model']].copy() if 'model' in proposal_meta.columns \
            else proposal_meta[['title', 'group']].rename(columns={'group': 'model'})
    _meta['_tnorm'] = _meta['title'].str.strip().str.lower()
    _dtdf = doc_topic_df.copy()
    _dtdf['_tnorm'] = _dtdf['title'].str.strip().str.lower() if 'title' in _dtdf.columns else ''
    doc_topic_df = _dtdf.merge(_meta[['_tnorm', 'model']], on='_tnorm', how='left')
    doc_topic_df['model'] = doc_topic_df['model'].fillna(doc_topic_df['group'])
    doc_topic_df = doc_topic_df.drop(columns=['_tnorm'])

MODEL_NAMES = ['Human', 'Claude', 'Gemini', 'GPT-5.2']
model_dfs = {m: doc_topic_df[doc_topic_df['model'] == m] for m in MODEL_NAMES}

soft_counts = {m: {col: int((model_dfs[m][col] > threshold).sum()) for col in topic_cols}
               for m in MODEL_NAMES}
soft_fracs  = {m: {col: float((model_dfs[m][col] > threshold).mean()) for col in topic_cols}
               for m in MODEL_NAMES}

print(f"{'Model':22s}" + "  ".join(f"{t:>14s}" for t in topic_cols))
print("-"*(22 + 16*len(topic_cols)))
for m in MODEL_NAMES:
    n_m = len(model_dfs[m])
    row = "  ".join(f"{soft_counts[m][t]:>5d} ({soft_fracs[m][t]:.0%})" for t in topic_cols)
    print(f"  {m:20s}  {row}  (n={n_m})")

# 4-group chi-square permutation test
contingency = np.array([[soft_counts[m][c] for c in topic_cols] for m in MODEL_NAMES], dtype=float)+1e-9
obs_chi2, _, _, _ = chi2_contingency(contingency)
all_probs   = doc_topic_df[topic_cols].values
all_models  = doc_topic_df['model'].values
perm_chi2   = []
for _ in range(10_000):
    sh = np.random.permutation(len(all_probs))
    cm = np.array([[(all_probs[sh][all_models==m, j] > threshold).sum()
                    for j in range(len(topic_cols))] for m in MODEL_NAMES], dtype=float)+1e-9
    c2, *_ = chi2_contingency(cm)
    perm_chi2.append(c2)
p_chi2_4g = np.mean(np.array(perm_chi2) >= obs_chi2)
print(f"\n4-group chi2={obs_chi2:.4f}, perm-p={p_chi2_4g:.4f}  "
      f"({'significant' if p_chi2_4g<0.05 else 'not significant'})")

# Per-topic Fisher: each AI model vs Human
fisher_rows = []
n_hu = len(model_dfs['Human'])
for col in topic_cols:
    for m in ['Claude', 'Gemini', 'GPT-5.2']:
        n_m = len(model_dfs[m])
        a,b = soft_counts['Human'][col], n_hu - soft_counts['Human'][col]
        c,d = soft_counts[m][col],       n_m  - soft_counts[m][col]
        _, p = fisher_exact([[a,b],[c,d]])
        fisher_rows.append({'topic':col,'model_vs_human':m,'human_n':a,'model_n':c,'p_value':p})
fisher_df = pd.DataFrame(fisher_rows)
_, fisher_df['q_holm'], _, _ = multipletests(fisher_df['p_value'], method='holm')
sig_f = fisher_df[fisher_df['q_holm'] < 0.05]
print(f"Per-topic Fisher (Holm): {len(sig_f)}/{len(fisher_df)} significant")
print(fisher_df.to_string(index=False))

# Shannon entropy per model
from scipy.stats import entropy as scipy_entropy
print("\n" + "="*60 + "\nTOPIC ENTROPY PER MODEL")
ent_rows = []
for m in MODEL_NAMES:
    df_m = model_dfs[m]
    mean_p = df_m[topic_cols].mean().clip(1e-9).values
    mean_p /= mean_p.sum()
    H = scipy_entropy(mean_p, base=np.e)
    H_norm = H / np.log(len(topic_cols))
    cov = sum(1 for col in topic_cols if soft_counts[m][col] > 0)
    dom = max(topic_cols, key=lambda col: soft_fracs[m][col])
    ent_rows.append({'model':m,'n':len(df_m),'H':H,'H_norm':H_norm,
                     'topics_covered':cov,'dominant_topic':dom})
    print(f"  {m:20s}  H={H:.4f} (norm={H_norm:.4f}), covered={cov}/{len(topic_cols)}, "
          f"dominant={dom} ({soft_fracs[m][dom]:.0%})")
ent_df = pd.DataFrame(ent_rows)

# Save
pd.DataFrame({m: soft_counts[m] for m in MODEL_NAMES}).T.rename_axis('model').reset_index()\
  .to_csv(TABLES_DIR/'topic_distribution_per_model.csv', index=False)
fisher_df.to_csv(TABLES_DIR/'topic_distribution_per_model_tests.csv', index=False)
ent_df.to_csv(TABLES_DIR/'topic_entropy_per_model.csv', index=False)
print("\n✓ Saved: topic_distribution_per_model.csv, _tests.csv, topic_entropy_per_model.csv")


In [ ]:
import seaborn as sns

# Analysis 1.2 Visualization: 4-group topic heatmap
threshold  = 0.20
topic_cols = sorted([c for c in doc_topic_df.columns if c.startswith('Topic_')])
topic_display_cols = [lda_topic_axis_labels.get(c, c) for c in topic_cols] if 'lda_topic_axis_labels' in globals() else topic_cols
MODEL_NAMES_VIZ = ['Human', 'Claude', 'Gemini', 'GPT-5.2']

frac_df = pd.DataFrame(
    {m: {col: float((doc_topic_df[doc_topic_df['model']==m][col] > threshold).mean())
         for col in topic_cols} for m in MODEL_NAMES_VIZ}
).T

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Topic Distribution Per Model (soft assignment >20%)", fontsize=13, fontweight='bold')

sns.heatmap(frac_df, annot=True, fmt='.0%', cmap='Blues', vmin=0, vmax=1,
            ax=axes[0], linewidths=0.5)
axes[0].set_title("Fraction with >20% probability per topic")
axes[0].set_xlabel("Topic"); axes[0].set_ylabel("Group")
axes[0].set_xticklabels(topic_display_cols, rotation=35, ha='right')

x = np.arange(len(topic_cols))
w = 0.2
for i, m in enumerate(MODEL_NAMES_VIZ):
    col_val = colors.get(m, f'C{i}')
    axes[1].bar(x + i*w, frac_df.loc[m], w, label=m, color=col_val, alpha=0.85)
axes[1].set_xticks(x + w*1.5)
axes[1].set_xticklabels(topic_display_cols, rotation=35, ha='right')
axes[1].set_ylabel("Fraction with >20% probability")
axes[1].set_title("Per-model topic participation rates")
axes[1].legend(); axes[1].set_ylim(0, 1.05)

plt.tight_layout()
fig_path = FIGURES_DIR / 'topic_distribution_comparison.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {fig_path}")


## Analysis 1.3: Embedding Cluster Structure and UMAP (Primary Cluster Labels)

Ward agglomerative clustering on all 92 proposals (`X_prop`), with data-driven k selection by silhouette score. Produces the cluster labels used by Analyses 1.4, 1.5, 2.1d, and 2.4. Also computes and **caches UMAP 2D coordinates** so Analysis 2.4 loads from cache.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.patheffects as path_effects
import textwrap
import umap as umap_lib

print("="*85)
print("ANALYSIS 1.3: EMBEDDING CLUSTER STRUCTURE — WARD AGGLOMERATIVE + UMAP CACHE")
print("="*85)

# Step 1: Data-driven k selection (silhouette)
k_scores = {}
for k in [2, 3, 4, 5]:
    ward = AgglomerativeClustering(n_clusters=k, linkage='ward')
    lbl  = ward.fit_predict(X_prop)
    sil  = silhouette_score(X_prop, lbl, metric='cosine')
    ch   = calinski_harabasz_score(X_prop, lbl)
    k_scores[k] = {'silhouette': sil, 'calinski_harabasz': ch, 'labels': lbl}
    print(f"  k={k}: silhouette={sil:.4f}, calinski_harabasz={ch:.1f}")

best_k_ward = max(k_scores, key=lambda k: k_scores[k]['silhouette'])
ward_labels  = k_scores[best_k_ward]['labels']
print(f"\n→ Best k = {best_k_ward} (by silhouette).  "
      f"Cluster sizes: {np.bincount(ward_labels).tolist()}")

# Step 2: Assign labels, inspect subfields
proposal_meta = proposal_meta.copy()
proposal_meta['ward_cluster'] = ward_labels
cluster_label_map = {i: f'Cluster_{chr(65+i)}' for i in range(best_k_ward)}
proposal_meta['ward_cluster_name'] = proposal_meta['ward_cluster'].map(cluster_label_map)

# BERTopic-style representation layer for the semantic clusters:
# Ward clusters remain the primary embedding-space regions; this applies BERTopic's
# c-TF-IDF representation idea plus the contrastive phrase label helper to those regions.
cluster_display_label_map = {i: cluster_label_map[i] for i in range(best_k_ward)}
if 'build_contrastive_phrase_labels' in globals() and 'build_class_ctfidf_terms' in globals() and 'all_abstracts' in globals():
    cluster_region_order = [cluster_label_map[i] for i in range(best_k_ward)]
    cluster_doc_labels = proposal_meta['ward_cluster_name'].to_numpy()
    cluster_contrastive_label_map, cluster_contrastive_info_df = build_contrastive_phrase_labels(
        all_abstracts,
        cluster_doc_labels,
        label_order=cluster_region_order,
        n_terms=4,
        min_df=1,
        max_df=0.90,
        prefix='Cluster',
    )
    cluster_ctfidf_terms_map, cluster_ctfidf_info_df = build_class_ctfidf_terms(
        all_abstracts,
        cluster_doc_labels,
        label_order=cluster_region_order,
        n_terms=10,
        min_df=1,
        max_df=0.90,
    )
    cluster_topic_label_df = (
        cluster_contrastive_info_df
        .merge(cluster_ctfidf_info_df, on='region', how='left')
        .rename(columns={'region': 'ward_cluster_name'})
    )
    cluster_topic_label_df['ward_cluster'] = cluster_topic_label_df['ward_cluster_name'].map({v: k for k, v in cluster_label_map.items()})
    cluster_topic_label_df = cluster_topic_label_df[['ward_cluster', 'ward_cluster_name', 'n_documents', 'display_label', 'contrastive_terms', 'ctfidf_label_raw', 'ctfidf_terms', 'label_strategy']]
    cluster_topic_label_df.to_csv(TABLES_DIR / 'ward_cluster_bertopic_style_labels.csv', index=False)
    cluster_display_label_map = {
        k: cluster_contrastive_label_map.get(cluster_label_map[k], cluster_label_map[k])
        for k in range(best_k_ward)
    }
    proposal_meta['ward_cluster_display_label'] = proposal_meta['ward_cluster'].map(cluster_display_label_map)
    print("\nBERTopic-style cluster representation labels (Ward regions + c-TF-IDF/contrastive phrases):")
    print(cluster_topic_label_df[['ward_cluster_name', 'n_documents', 'display_label', 'ctfidf_label_raw']].to_string(index=False))
    print(f"✓ Saved: {TABLES_DIR / 'ward_cluster_bertopic_style_labels.csv'}")
else:
    proposal_meta['ward_cluster_display_label'] = proposal_meta['ward_cluster_name']
    print("\n⚠  Contrastive/c-TF-IDF label helpers not found; Ward cluster labels will use Cluster_A/B IDs only.")

# Optional true BERTopic sensitivity model on proposals using the existing proposal embeddings.
# This is kept secondary because n≈92 proposals is small for HDBSCAN-style topic discovery.
proposal_bertopic_topic_info_df = pd.DataFrame()
proposal_bertopic_topics = None
try:
    from bertopic import BERTopic
    from hdbscan import HDBSCAN

    print("\nFitting optional BERTopic sensitivity model on proposal texts with precomputed embeddings...")
    prop_bt_umap_model = umap_lib.UMAP(
        n_neighbors=min(15, max(2, len(all_abstracts) - 1)),
        n_components=min(5, max(2, len(all_abstracts) - 1)),
        min_dist=0.0,
        metric='cosine',
        random_state=42,
    )
    prop_bt_cluster_model = HDBSCAN(
        min_cluster_size=5,
        min_samples=2,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True,
    )
    prop_bt_vectorizer = CountVectorizer(
        stop_words='english',
        ngram_range=(1, 3),
        min_df=1,
        max_df=0.90,
    )
    proposal_bertopic_model = BERTopic(
        embedding_model=None,
        umap_model=prop_bt_umap_model,
        hdbscan_model=prop_bt_cluster_model,
        vectorizer_model=prop_bt_vectorizer,
        calculate_probabilities=False,
        verbose=False,
    )
    proposal_bertopic_topics, _proposal_bertopic_probs = proposal_bertopic_model.fit_transform(all_abstracts, X_prop)
    proposal_bertopic_topics = np.asarray(proposal_bertopic_topics, dtype=int)
    proposal_meta['proposal_bertopic_topic'] = proposal_bertopic_topics

    prop_bt_valid_topics = sorted([int(t) for t in np.unique(proposal_bertopic_topics) if int(t) >= 0])
    prop_bt_raw_label_map = {
        f'BT_{t}': ', '.join([term for term, _score in (proposal_bertopic_model.get_topic(t) or [])[:4]])
        for t in prop_bt_valid_topics
    }
    if len(prop_bt_valid_topics) > 0 and 'build_contrastive_phrase_labels' in globals():
        prop_bt_named_labels = np.asarray([f'BT_{int(t)}' if int(t) >= 0 else 'Outlier' for t in proposal_bertopic_topics])
        prop_bt_order = [f'BT_{t}' for t in prop_bt_valid_topics]
        prop_bt_label_map, prop_bt_label_info_df = build_contrastive_phrase_labels(
            all_abstracts,
            prop_bt_named_labels,
            label_order=prop_bt_order,
            n_terms=4,
            min_df=1,
            max_df=0.90,
            prefix='BERTopic',
        )
        proposal_bertopic_topic_info_df = prop_bt_label_info_df.rename(columns={'region': 'bertopic_topic_name'})
        proposal_bertopic_topic_info_df['bertopic_topic'] = proposal_bertopic_topic_info_df['bertopic_topic_name'].str.replace('BT_', '', regex=False).astype(int)
        proposal_bertopic_topic_info_df['raw_bertopic_ctfidf_terms'] = proposal_bertopic_topic_info_df['bertopic_topic_name'].map(prop_bt_raw_label_map)
        proposal_bertopic_topic_info_df = proposal_bertopic_topic_info_df[['bertopic_topic', 'bertopic_topic_name', 'n_documents', 'display_label', 'contrastive_terms', 'raw_bertopic_ctfidf_terms', 'label_strategy']]
        prop_bt_display_label_by_int = {
            int(row['bertopic_topic']): row['display_label']
            for _, row in proposal_bertopic_topic_info_df.iterrows()
        }
    else:
        prop_bt_display_label_by_int = {t: prop_bt_raw_label_map.get(f'BT_{t}', f'BERTopic {t}') for t in prop_bt_valid_topics}
        proposal_bertopic_topic_info_df = pd.DataFrame([
            {
                'bertopic_topic': t,
                'bertopic_topic_name': f'BT_{t}',
                'n_documents': int((proposal_bertopic_topics == t).sum()),
                'display_label': prop_bt_display_label_by_int[t],
                'contrastive_terms': '',
                'raw_bertopic_ctfidf_terms': prop_bt_raw_label_map.get(f'BT_{t}', ''),
                'label_strategy': 'raw_bertopic_ctfidf',
            }
            for t in prop_bt_valid_topics
        ])

    proposal_meta['proposal_bertopic_display_label'] = [
        prop_bt_display_label_by_int.get(int(t), 'Outlier / unassigned') if int(t) >= 0 else 'Outlier / unassigned'
        for t in proposal_bertopic_topics
    ]
    _prop_bt_save_cols = [c for c in ['title', 'group_binary', 'group_model', 'proposal_bertopic_topic', 'proposal_bertopic_display_label'] if c in proposal_meta.columns]
    proposal_meta[_prop_bt_save_cols].to_csv(
        TABLES_DIR / 'proposal_bertopic_assignments.csv', index=False
    )
    proposal_bertopic_topic_info_df.to_csv(TABLES_DIR / 'proposal_bertopic_topic_labels.csv', index=False)
    print(f"✓ Proposal BERTopic produced {len(prop_bt_valid_topics)} non-outlier topics; outlier fraction={(proposal_bertopic_topics < 0).mean():.1%}")
    if len(proposal_bertopic_topic_info_df):
        print(proposal_bertopic_topic_info_df[['bertopic_topic', 'n_documents', 'display_label', 'raw_bertopic_ctfidf_terms']].to_string(index=False))
    print(f"✓ Saved: {TABLES_DIR / 'proposal_bertopic_assignments.csv'}")
    print(f"✓ Saved: {TABLES_DIR / 'proposal_bertopic_topic_labels.csv'}")
except Exception as e:
    proposal_meta['proposal_bertopic_topic'] = np.nan
    proposal_meta['proposal_bertopic_display_label'] = np.nan
    print(f"\n⚠  Optional proposal BERTopic sensitivity model skipped: {type(e).__name__}: {e}")
    print("   Ward clusters with c-TF-IDF/contrastive labels remain the primary proposal semantic regions.")

print("\nTop proposal titles per Ward cluster:")
for k in range(best_k_ward):
    idx_k = proposal_meta[proposal_meta['ward_cluster'] == k].index[:5]
    print(f"  {cluster_label_map[k]} — {cluster_display_label_map.get(k, cluster_label_map[k])}:")
    for j in idx_k:
        t = proposal_meta.loc[j, 'title'] if 'title' in proposal_meta.columns else f'Proposal {j}'
        print(f"    • {str(t)[:80]}")

# Step 3: Per-group cluster membership + Fisher tests
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

print("\n" + "="*60)
print("CLUSTER MEMBERSHIP PER GROUP")
grp_col = 'group_model' if 'group_model' in proposal_meta.columns else 'group'
GROUP_NAMES_W = ['Human', 'Claude', 'Gemini', 'GPT-5.2']
membership = {}
for g in GROUP_NAMES_W:
    if g == 'Human':
        mask = proposal_meta['group_binary'] == 'Human'
    else:
        mask = proposal_meta[grp_col].str.contains(g, case=False, na=False)
    cnts = proposal_meta.loc[mask, 'ward_cluster'].value_counts().sort_index()
    membership[g] = {cluster_label_map[k]: cnts.get(k, 0) for k in range(best_k_ward)}
    print(f"  {g:20s}: " + ", ".join(f"{cluster_label_map[k]}={membership[g][cluster_label_map[k]]}"
                                      for k in range(best_k_ward)))

hu_mask = proposal_meta['group_binary'] == 'Human'
hu_cnts = proposal_meta.loc[hu_mask, 'ward_cluster'].value_counts()
fisher_rows_ward = []
for g in ['Claude', 'Gemini', 'GPT-5.2']:
    g_mask = proposal_meta[grp_col].str.contains(g, case=False, na=False)
    g_cnts = proposal_meta.loc[g_mask, 'ward_cluster'].value_counts()
    for k in range(best_k_ward):
        n_hu_k = hu_cnts.get(k, 0); n_hu_ot = hu_mask.sum() - n_hu_k
        n_g_k  = g_cnts.get(k, 0);  n_g_ot  = g_mask.sum()  - n_g_k
        _, p = fisher_exact([[n_hu_k, n_hu_ot],[n_g_k, n_g_ot]])
        fisher_rows_ward.append({'model':g,'cluster':cluster_label_map[k],
                                 'cluster_display_label': cluster_display_label_map.get(k, cluster_label_map[k]),
                                 'human_n':n_hu_k,'model_n':n_g_k,'p_value':p})
fw_df = pd.DataFrame(fisher_rows_ward)
if len(fw_df) > 0:
    _, fw_df['q_holm'], _, _ = multipletests(fw_df['p_value'], method='holm')
    print("\nFisher exact (AI model vs Human, Holm):")
    print(fw_df.to_string(index=False))

# Step 4: Per-group silhouette scores
print("\n" + "="*60 + "\nPER-GROUP SILHOUETTE SCORES")
for g in GROUP_NAMES_W:
    mask = proposal_meta['group_binary'] == 'Human' if g == 'Human' \
           else proposal_meta[grp_col].str.contains(g, case=False, na=False)
    idx = proposal_meta[mask].index
    if len(set(ward_labels[idx])) < 2:
        sil_g = float('nan')
    else:
        sil_g = silhouette_score(X_prop[idx], ward_labels[idx], metric='cosine')
    dom_frac = pd.Series(ward_labels[idx]).value_counts(normalize=True).max() if len(idx)>0 else 0
    print(f"  {g:20s}: silhouette={sil_g:.4f}, dominant-cluster fraction={dom_frac:.0%}")

# Step 5: Compute and cache UMAP 2D coordinates
umap_cache_path = TABLES_DIR / 'cached' / 'proposal_umap2d.npy'
umap_cache_path.parent.mkdir(parents=True, exist_ok=True)
print("\nComputing UMAP projection (cached for Analysis 2.4)...")
reducer_1_3 = umap_lib.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
                              metric='cosine', random_state=42)
umap_2d_cached = reducer_1_3.fit_transform(X_prop)
np.save(umap_cache_path, umap_2d_cached)
print(f"✓ UMAP cached to {umap_cache_path}")

CLUSTER_PALETTE = ['#e6194b', '#3cb44b', '#4363d8', '#f58231'][:best_k_ward]

_umap_x_range = float(np.ptp(umap_2d_cached[:, 0])) or 1.0
_umap_y_range = float(np.ptp(umap_2d_cached[:, 1])) or 1.0
_umap_x_mid = float(np.median(umap_2d_cached[:, 0]))
_umap_y_min = float(np.min(umap_2d_cached[:, 1]))


def _cluster_label_position(points, order_idx=0):
    """Place labels beside or below a cluster instead of on top of its proposal dots."""
    cx, cy = np.median(points, axis=0)
    x_min, y_min = np.min(points, axis=0)
    x_max, _y_max = np.max(points, axis=0)
    x_gap = 0.075 * _umap_x_range
    y_gap = 0.085 * _umap_y_range
    stagger_y = ((order_idx % 3) - 1) * 0.035 * _umap_y_range
    stagger_x = ((order_idx % 3) - 1) * 0.045 * _umap_x_range

    # Edge clusters read best with side labels; central clusters get bottom labels.
    if cx < _umap_x_mid - 0.12 * _umap_x_range:
        return cx, cy, x_min - x_gap, cy + stagger_y, 'right'
    if cx > _umap_x_mid + 0.12 * _umap_x_range:
        return cx, cy, x_max + x_gap, cy + stagger_y, 'left'
    return cx, cy, cx + stagger_x, y_min - y_gap, 'center'


def _label_text_box(ax, x, y, label, color, fontsize=8, zorder=7, xy=None, ha='center'):
    label = textwrap.fill(str(label), width=28)
    bbox = dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor=color, linewidth=1.1, alpha=0.90)
    if xy is None:
        txt = ax.text(
            x, y, label,
            ha=ha, va='center', fontsize=fontsize, fontweight='bold', color='black',
            bbox=bbox, zorder=zorder,
        )
    else:
        txt = ax.annotate(
            label,
            xy=xy,
            xytext=(x, y),
            textcoords='data',
            ha=ha,
            va='center',
            fontsize=fontsize,
            fontweight='bold',
            color='black',
            bbox=bbox,
            arrowprops=dict(arrowstyle='-', color=color, linewidth=1.0, alpha=0.75, shrinkA=4, shrinkB=4),
            annotation_clip=False,
            zorder=zorder,
        )
        ax.update_datalim(np.array([[x, y]]))
        ax.autoscale_view()
    txt.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground='white')])
    return txt


def _add_ward_cluster_text_labels(ax, fontsize=7):
    for k in range(best_k_ward):
        mask_k = ward_labels == k
        if not mask_k.any():
            continue
        pts_k = umap_2d_cached[mask_k]
        cx, cy, tx, ty, ha = _cluster_label_position(pts_k, order_idx=k)
        label = f"{cluster_label_map[k]}: {cluster_display_label_map.get(k, cluster_label_map[k])}"
        _label_text_box(ax, tx, ty, label, CLUSTER_PALETTE[k], fontsize=fontsize, xy=(cx, cy), ha=ha)

# Step 6a: Overall labeled Ward-cluster UMAP
fig_over, ax_over = plt.subplots(figsize=(9.5, 7.5))
for k in range(best_k_ward):
    mask_k = ward_labels == k
    ax_over.scatter(
        umap_2d_cached[mask_k, 0], umap_2d_cached[mask_k, 1],
        c=CLUSTER_PALETTE[k], s=42, alpha=0.82,
        edgecolors='white', linewidths=0.45,
        label=f"{cluster_label_map[k]}: {cluster_display_label_map.get(k, cluster_label_map[k])}",
    )
_add_ward_cluster_text_labels(ax_over, fontsize=8)
ax_over.set_title("Proposal UMAP Colored by Ward Embedding Clusters\nLabels use BERTopic-style c-TF-IDF + contrastive phrases", fontsize=12, fontweight='bold')
ax_over.set_xlabel("UMAP-1")
ax_over.set_ylabel("UMAP-2")
ax_over.legend(fontsize=8, loc='best', framealpha=0.90)
plt.tight_layout()
fig_path_over = FIGURES_DIR / 'proposal_ward_clusters_labeled_umap.png'
fig_over.savefig(fig_path_over, dpi=180, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {fig_path_over}")

# Step 6b: Optional proposal-BERTopic topic map on the same proposal UMAP
if proposal_bertopic_topics is not None and np.isfinite(pd.Series(proposal_bertopic_topics).astype(float)).all():
    prop_bt_valid_topics = sorted([int(t) for t in np.unique(proposal_bertopic_topics) if int(t) >= 0])
    if len(prop_bt_valid_topics) > 0:
        bt_palette = plt.cm.get_cmap('tab20', max(20, len(prop_bt_valid_topics)))
        bt_color_map = {t: bt_palette(i % 20) for i, t in enumerate(prop_bt_valid_topics)}
        fig_bt, ax_bt = plt.subplots(figsize=(9.5, 7.5))
        outlier_mask = proposal_bertopic_topics < 0
        if outlier_mask.any():
            ax_bt.scatter(umap_2d_cached[outlier_mask, 0], umap_2d_cached[outlier_mask, 1], c='#d9d9d9', s=28, alpha=0.55, label='Outlier / unassigned')
        for t in prop_bt_valid_topics:
            mask_t = proposal_bertopic_topics == t
            label_t = proposal_bertopic_topic_info_df.set_index('bertopic_topic')['display_label'].to_dict().get(t, f'BERTopic {t}') if len(proposal_bertopic_topic_info_df) else f'BERTopic {t}'
            ax_bt.scatter(
                umap_2d_cached[mask_t, 0], umap_2d_cached[mask_t, 1],
                c=[bt_color_map[t]], s=42, alpha=0.82,
                edgecolors='white', linewidths=0.45,
                label=f"BT_{t}: {label_t}",
            )
            pts_t = umap_2d_cached[mask_t]
            cx, cy, tx, ty, ha = _cluster_label_position(pts_t, order_idx=t)
            _label_text_box(ax_bt, tx, ty, f"BT_{t}: {label_t}", bt_color_map[t], fontsize=8, xy=(cx, cy), ha=ha)
        ax_bt.set_title("Proposal UMAP Colored by BERTopic Topics\nSensitivity analysis using precomputed proposal embeddings", fontsize=12, fontweight='bold')
        ax_bt.set_xlabel("UMAP-1")
        ax_bt.set_ylabel("UMAP-2")
        ax_bt.legend(fontsize=8, loc='best', framealpha=0.90)
        plt.tight_layout()
        fig_path_bt = FIGURES_DIR / 'proposal_bertopic_topics_labeled_umap.png'
        fig_bt.savefig(fig_path_bt, dpi=180, bbox_inches='tight')
        plt.show()
        print(f"✓ Saved: {fig_path_bt}")
    else:
        print("⚠  Proposal BERTopic found no non-outlier topics to visualize.")

# Step 6c: 2×2 UMAP panel per group with visible region labels
fig, axes = plt.subplots(2, 2, figsize=(12, 11), sharex=True, sharey=True)
fig.suptitle("Cluster Membership Per Group (Ward Agglomerative, joint fit)",
             fontsize=13, fontweight='bold')
axes_flat = axes.flatten()
for ax_i, g in enumerate(GROUP_NAMES_W):
    ax = axes_flat[ax_i]
    mask = proposal_meta['group_binary'] == 'Human' if g == 'Human' \
           else proposal_meta[grp_col].str.contains(g, case=False, na=False)
    idx = proposal_meta[mask].index
    ax.scatter(umap_2d_cached[:,0], umap_2d_cached[:,1], c='lightgray', s=10, alpha=0.22, linewidths=0)
    for k in range(best_k_ward):
        k_idx = [j for j in idx if ward_labels[j] == k]
        if k_idx:
            ax.scatter(umap_2d_cached[k_idx,0], umap_2d_cached[k_idx,1],
                       c=CLUSTER_PALETTE[k], s=44, alpha=0.88, label=cluster_label_map[k],
                       edgecolors='white', linewidths=0.45)
    _add_ward_cluster_text_labels(ax, fontsize=6.4)
    sil_g = silhouette_score(X_prop[idx], ward_labels[idx], metric='cosine') \
             if len(set(ward_labels[idx])) >= 2 else float('nan')
    dom_frac = pd.Series(ward_labels[idx]).value_counts(normalize=True).max() if len(idx)>0 else 0
    ax.set_title(f"{g}  (sil={sil_g:.3f}, dom={dom_frac:.0%})", fontsize=10)
    ax.legend(fontsize=8, loc='best', framealpha=0.88)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
plt.tight_layout()
fig_path = FIGURES_DIR / 'cluster_membership_umap_per_group.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {fig_path}")

# Save tables
k_sel_df = pd.DataFrame(
    [{'k': k, 'silhouette': k_scores[k]['silhouette'],
      'calinski_harabasz': k_scores[k]['calinski_harabasz'],
      'selected': (k == best_k_ward)} for k in [2,3,4,5]])
k_sel_df.to_csv(TABLES_DIR / 'diversity_cluster_k_selection.csv', index=False)
_cluster_save_cols = ['title','group_binary','ward_cluster','ward_cluster_name','ward_cluster_display_label']
for _optional_col in ['proposal_bertopic_topic', 'proposal_bertopic_display_label']:
    if _optional_col in proposal_meta.columns:
        _cluster_save_cols.append(_optional_col)
proposal_meta[_cluster_save_cols].to_csv(
    TABLES_DIR / 'diversity_cluster_membership_by_group.csv', index=False)
print("✓ Saved: diversity_cluster_k_selection.csv, diversity_cluster_membership_by_group.csv")


## Analysis 1.4: Cluster Segregation — GMM Convergent Validity and Per-Model Composition

Gaussian Mixture Model clustering on full embedding space. Convergent validity check against Ward clusters from Analysis 1.3. Per-model composition reveals which model drives AI segregation.

In [ ]:
from pathlib import Path

emb_path = PROPOSAL_EMBEDDINGS_FILE
if not emb_path.exists():
    legacy_paths = sorted(emb_path.parent.glob('proposal_embeddings_[0-9]*.pkl'))
    if not legacy_paths:
        raise FileNotFoundError(f"No embedding pickle found at {emb_path}. Run the embedding generation cell first.")
    emb_path = legacy_paths[-1]

embeddings_data = retrieve_embeddings(str(emb_path))

ai_embeddings = embeddings_data['ai_embeddings']
human_embeddings = embeddings_data['human_embeddings']
# ai_metadata = embeddings_data['ai_metadata']
# human_metadata = embeddings_data['human_metadata']

print(f"✓ Loaded embeddings from: {emb_path}")
print(f"  AI: {ai_embeddings.shape}, Human: {human_embeddings.shape}")


In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, normalized_mutual_info_score, adjusted_rand_score

print("="*85)
print("CLUSTER ANALYSIS: STEP 1 - OPTIMAL k SELECTION")
print("="*85)

# Use embeddings from diversity analysis (already computed)
all_embeddings_cluster = np.vstack([human_embeddings, ai_embeddings])
source_labels_cluster = np.array(['Human'] * len(human_embeddings) + ['AI'] * len(ai_embeddings))

print(f"\nUsing embeddings: {all_embeddings_cluster.shape}")
print(f"  Human: {len(human_embeddings)}, AI: {len(ai_embeddings)}")

# Test different values of k
k_values = [3, 4, 5, 6, 7, 8]
metrics_results = []

print(f"\nTesting k = {k_values}...")
print(f"\n{'k':<5} {'Silhouette':<12} {'Davies-Bouldin':<16} {'BIC':<12}")
print("-"*50)

for k in k_values:
    # Fit GMM
    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42,
        n_init=10
    )
    cluster_labels = gmm.fit_predict(all_embeddings_cluster)
    
    # Compute metrics
    silhouette = silhouette_score(all_embeddings_cluster, cluster_labels)
    davies_bouldin = davies_bouldin_score(all_embeddings_cluster, cluster_labels)
    bic = gmm.bic(all_embeddings_cluster)
    
    metrics_results.append({
        'k': k,
        'silhouette': silhouette,
        'davies_bouldin': davies_bouldin,
        'bic': bic
    })
    
    print(f"{k:<5} {silhouette:<12.4f} {davies_bouldin:<16.4f} {bic:<12.2f}")

metrics_df = pd.DataFrame(metrics_results)

# Select best k using BIC (lower is better)
best_k_bic = metrics_df.loc[metrics_df['bic'].idxmin(), 'k']
best_k_silhouette = metrics_df.loc[metrics_df['silhouette'].idxmax(), 'k']

print(f"\n✓ Best k by BIC (elbow method): {best_k_bic}")
print(f"✓ Best k by Silhouette score: {best_k_silhouette}")

# Use BIC as primary criterion
best_k = int(best_k_bic)
print(f"\n→ Selected k = {best_k} (using BIC)")

# Visualize selection metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: BIC
ax1 = axes[0]
ax1.plot(metrics_df['k'], metrics_df['bic'], 'o-', linewidth=2, markersize=8, color='steelblue')
ax1.axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'Selected k={best_k}')
ax1.set_xlabel('Number of clusters (k)', fontsize=12, fontweight='bold')
ax1.set_ylabel('BIC (lower = better)', fontsize=12, fontweight='bold')
ax1.set_title('BIC vs. k (Elbow Method)', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.legend()

# Panel 2: Silhouette
ax2 = axes[1]
ax2.plot(metrics_df['k'], metrics_df['silhouette'], 'o-', linewidth=2, markersize=8, color='darkorange')
ax2.axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'Selected k={best_k}')
ax2.set_xlabel('Number of clusters (k)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Silhouette Score (higher = better)', fontsize=12, fontweight='bold')
ax2.set_title('Silhouette Score vs. k', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)
ax2.legend()

# Panel 3: Davies-Bouldin
ax3 = axes[2]
ax3.plot(metrics_df['k'], metrics_df['davies_bouldin'], 'o-', linewidth=2, markersize=8, color='darkgreen')
ax3.axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'Selected k={best_k}')
ax3.set_xlabel('Number of clusters (k)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Davies-Bouldin Index (lower = better)', fontsize=12, fontweight='bold')
ax3.set_title('Davies-Bouldin Index vs. k', fontsize=14, fontweight='bold')
ax3.grid(alpha=0.3)
ax3.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cluster_k_selection.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Figure saved to: {FIGURES_DIR / 'cluster_k_selection.png'}")
print("="*85)


In [ ]:
print("="*85)
print(f"CLUSTER ANALYSIS: STEP 2 - CLUSTERING WITH k={best_k}")
print("="*85)

# Fit final GMM with best k
gmm_final = GaussianMixture(
    n_components=best_k,
    covariance_type='full',
    random_state=42,
    n_init=10
)

cluster_labels_final = gmm_final.fit_predict(all_embeddings_cluster)
cluster_probs_final = gmm_final.predict_proba(all_embeddings_cluster)

print(f"\n✓ Clustered {len(all_embeddings_cluster)} proposals into {best_k} clusters")
print(f"  Cluster sizes: {np.bincount(cluster_labels_final)}")

# Cluster composition analysis
print(f"\n" + "-"*85)
print("CLUSTER COMPOSITION ANALYSIS")
print("-"*85)

cluster_composition = []

for cluster_id in range(best_k):
    cluster_mask = cluster_labels_final == cluster_id
    cluster_sources = source_labels_cluster[cluster_mask]
    
    n_total = len(cluster_sources)
    n_human = (cluster_sources == 'Human').sum()
    n_ai = (cluster_sources == 'AI').sum()
    
    pct_human = n_human / n_total * 100 if n_total > 0 else 0
    pct_ai = n_ai / n_total * 100 if n_total > 0 else 0
    
    # Classify cluster dominance (baseline: 25% human, 75% AI)
    if pct_human > 60:
        dominance = 'Human-dominated'
    elif pct_human < 15:  # Less than expected given 25% baseline
        dominance = 'AI-dominated'
    else:
        dominance = 'Mixed'
    
    cluster_composition.append({
        'cluster': cluster_id,
        'total': n_total,
        'human': n_human,
        'ai': n_ai,
        'pct_human': pct_human,
        'pct_ai': pct_ai,
        'dominance': dominance
    })

composition_df = pd.DataFrame(cluster_composition)

print(f"\n{'Cluster':<10} {'Total':<8} {'Human':<8} {'AI':<8} {'%Human':<10} {'%AI':<10} {'Dominance'}")
print("-"*85)

for _, row in composition_df.iterrows():
    print(f"{row['cluster']:<10} {row['total']:<8} {row['human']:<8} {row['ai']:<8} "
          f"{row['pct_human']:<10.1f} {row['pct_ai']:<10.1f} {row['dominance']}")

print(f"\n{'Dominance':<20} {'Count'}")
print("-"*30)
dominance_counts = composition_df['dominance'].value_counts()
for dom, count in dominance_counts.items():
    print(f"{dom:<20} {count}")

print(f"\n💡 Baseline: 25% human, 75% AI")
print(f"   Human-dominated: >60% human")
print(f"   AI-dominated: <15% human")
print(f"   Mixed: 15-60% human")
print("="*85)


In [ ]:
print("="*85)
print("CLUSTER ANALYSIS: STEP 3 - SEGREGATION METRICS (with Permutation Tests)")
print("="*85)

# Convert source labels to binary (0 = Human, 1 = AI)
source_binary = (source_labels_cluster == 'AI').astype(int)

# 1. Normalized Mutual Information (NMI)
nmi_observed = normalized_mutual_info_score(cluster_labels_final, source_binary)

print(f"\n1. NORMALIZED MUTUAL INFORMATION (NMI):")
print(f"   Observed NMI: {nmi_observed:.4f}")

# Permutation test for NMI
n_perm_seg = 10000
nmi_null = []

for _ in range(n_perm_seg):
    shuffled_sources = np.random.permutation(source_binary)
    nmi_perm = normalized_mutual_info_score(cluster_labels_final, shuffled_sources)
    nmi_null.append(nmi_perm)

nmi_null = np.array(nmi_null)
p_nmi = np.mean(nmi_null >= nmi_observed)

print(f"   Null distribution: {nmi_null.mean():.4f} ± {nmi_null.std():.4f}")
print(f"   p-value: {p_nmi:.4f}")

if p_nmi < 0.05:
    print(f"   → Significant segregation: clusters predict source better than chance")
else:
    print(f"   → No significant segregation: clustering independent of source")

# 2. Adjusted Rand Index (ARI)
ari_observed = adjusted_rand_score(cluster_labels_final, source_binary)

print(f"\n2. ADJUSTED RAND INDEX (ARI):")
print(f"   Observed ARI: {ari_observed:.4f}")

# Permutation test for ARI
ari_null = []

for _ in range(n_perm_seg):
    shuffled_sources = np.random.permutation(source_binary)
    ari_perm = adjusted_rand_score(cluster_labels_final, shuffled_sources)
    ari_null.append(ari_perm)

ari_null = np.array(ari_null)
p_ari = np.mean(ari_null >= ari_observed)

print(f"   Null distribution: {ari_null.mean():.4f} ± {ari_null.std():.4f}")
print(f"   p-value: {p_ari:.4f}")

if p_ari < 0.05:
    print(f"   → Significant agreement: cluster assignments correlate with source")
else:
    print(f"   → No significant agreement: random overlap")

# 3. Within-group vs. Between-group distances
print(f"\n3. WITHIN-GROUP VS. BETWEEN-GROUP DISTANCES:")

# Calculate pairwise distances
from sklearn.metrics.pairwise import cosine_distances

distances_all = cosine_distances(all_embeddings_cluster)

# Within-human distances
human_mask = source_labels_cluster == 'Human'
within_human_dists = distances_all[np.ix_(human_mask, human_mask)]
within_human_dists = within_human_dists[np.triu_indices_from(within_human_dists, k=1)]

# Within-AI distances
ai_mask = source_labels_cluster == 'AI'
within_ai_dists = distances_all[np.ix_(ai_mask, ai_mask)]
within_ai_dists = within_ai_dists[np.triu_indices_from(within_ai_dists, k=1)]

# Between human-AI distances
between_dists = distances_all[np.ix_(human_mask, ai_mask)].flatten()

print(f"   Within-human mean distance: {within_human_dists.mean():.4f} ± {within_human_dists.std():.4f}")
print(f"   Within-AI mean distance: {within_ai_dists.mean():.4f} ± {within_ai_dists.std():.4f}")
print(f"   Between human-AI mean distance: {between_dists.mean():.4f} ± {between_dists.std():.4f}")

# Ratio of between to within
within_mean = np.mean([within_human_dists.mean(), within_ai_dists.mean()])
between_within_ratio = between_dists.mean() / within_mean

print(f"\n   Between/Within ratio: {between_within_ratio:.4f}")

# Permutation test for between/within ratio
ratio_null = []

for _ in range(n_perm_seg):
    shuffled = np.random.permutation(source_labels_cluster)
    h_mask = shuffled == 'Human'
    a_mask = shuffled == 'AI'
    
    within_h = distances_all[np.ix_(h_mask, h_mask)]
    within_h = within_h[np.triu_indices_from(within_h, k=1)]
    
    within_a = distances_all[np.ix_(a_mask, a_mask)]
    within_a = within_a[np.triu_indices_from(within_a, k=1)]
    
    between = distances_all[np.ix_(h_mask, a_mask)].flatten()
    
    within_m = np.mean([within_h.mean(), within_a.mean()])
    ratio_perm = between.mean() / within_m
    ratio_null.append(ratio_perm)

ratio_null = np.array(ratio_null)
p_ratio = np.mean(ratio_null >= between_within_ratio)

print(f"   Null distribution: {ratio_null.mean():.4f} ± {ratio_null.std():.4f}")
print(f"   p-value: {p_ratio:.4f}")

if p_ratio < 0.05:
    if between_within_ratio > 1:
        print(f"   → Human and AI proposals are significantly MORE distant from each other than within groups")
    else:
        print(f"   → Human and AI proposals are significantly CLOSER to each other than within groups")
else:
    print(f"   → No significant difference in between vs. within-group distances")

print("\n" + "="*85)
print("INTERPRETATION SUMMARY")
print("="*85)

if p_nmi < 0.05 or p_ari < 0.05:
    print("✓ SEGREGATION: Human and AI proposals cluster in distinct semantic regions")
    print("  → They generate different KINDS of ideas")
elif p_nmi > 0.10 and p_ari > 0.10:
    print("✓ INTEGRATION: Human and AI proposals intermixed in embedding space")
    print("  → Ideas are similar regardless of source")
else:
    print("✓ INTERMEDIATE: Some thematic clustering but not strictly by source")

print("="*85)


In [ ]:
# Analysis 1.4: Visualization using cached UMAP + per-model GMM composition + Ward ARI
print("="*85)
print("ANALYSIS 1.4: GMM VISUALIZATION, PER-MODEL COMPOSITION, GMM vs WARD ARI")
print("="*85)

import umap as umap_lib
umap_cache_path = TABLES_DIR / 'cached' / 'proposal_umap2d.npy'
if umap_cache_path.exists():
    embeddings_2d_cluster = np.load(umap_cache_path)
    print(f"✓ Loaded cached UMAP from {umap_cache_path}")
else:
    print("⚠  Cache missing — recomputing UMAP (run Analysis 1.3 first for consistent coordinates)")
    reducer_cluster = umap_lib.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
                                     metric='cosine', random_state=42)
    embeddings_2d_cluster = reducer_cluster.fit_transform(X_prop)
print(f"UMAP shape: {embeddings_2d_cluster.shape}")

# Step 4: Per-model GMM composition
grp_col_c = 'group_model' if 'group_model' in proposal_meta.columns else 'group'
GROUP_NAMES_C = ['Human', 'Claude', 'Gemini', 'GPT-5.2']
comp_rows = []
print("\nGMM cluster composition per model:")
for g in GROUP_NAMES_C:
    mask_g = proposal_meta['group_binary'] == 'Human' if g == 'Human' \
             else proposal_meta[grp_col_c].str.contains(g, case=False, na=False)
    g_gmm = cluster_labels_final[mask_g]
    cnts  = np.bincount(g_gmm, minlength=best_k)
    frac  = cnts / max(mask_g.sum(), 1)
    for k in range(best_k):
        comp_rows.append({'group':g,'gmm_cluster':k,'count':int(cnts[k]),'fraction':float(frac[k])})
    print(f"  {g:20s}: " + ", ".join(f"GMM-{k}={cnts[k]} ({frac[k]:.0%})" for k in range(best_k)))
comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv(TABLES_DIR / 'cluster_gmm_composition_per_model.csv', index=False)

# Step 7: ARI between GMM and Ward solutions
if 'ward_labels' in globals():
    from sklearn.metrics import adjusted_rand_score
    if len(cluster_labels_final) == len(ward_labels):
        ari_gmm_ward = adjusted_rand_score(ward_labels, cluster_labels_final)
        nmi_gmm_ward = normalized_mutual_info_score(ward_labels, cluster_labels_final)
        print(f"\nGMM (k={best_k}) vs Ward (k={best_k_ward}) ARI={ari_gmm_ward:.4f}, NMI={nmi_gmm_ward:.4f}")
        interp = 'high agreement — same latent clusters' if ari_gmm_ward > 0.3 else 'low agreement — different granularity levels'
        print(f"→ {interp}")
        pd.DataFrame([{'ari_gmm_ward': ari_gmm_ward, 'nmi_gmm_ward': nmi_gmm_ward,
                       'best_k_gmm': best_k, 'best_k_ward': best_k_ward}])\
          .to_csv(TABLES_DIR / 'cluster_gmm_vs_ward_agreement.csv', index=False)
        print("✓ Saved: cluster_gmm_vs_ward_agreement.csv")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(f"GMM Cluster Segregation (k={best_k}) — Embedding Space",
             fontsize=13, fontweight='bold')
cmap_pts = plt.cm.tab10(np.linspace(0, 1, best_k))
for k in range(best_k):
    mask_k = cluster_labels_final == k
    axes[0].scatter(embeddings_2d_cluster[mask_k, 0], embeddings_2d_cluster[mask_k, 1],
                    c=[cmap_pts[k]], s=30, alpha=0.7, label=f'GMM-{k} (n={mask_k.sum()})')
axes[0].set_title("GMM Clusters"); axes[0].legend(fontsize=8)
axes[0].set_xlabel("UMAP-1"); axes[0].set_ylabel("UMAP-2")
for g in GROUP_NAMES_C:
    mask_g = proposal_meta['group_binary'] == 'Human' if g == 'Human' \
             else proposal_meta[grp_col_c].str.contains(g, case=False, na=False)
    idx_g = proposal_meta[mask_g].index
    axes[1].scatter(embeddings_2d_cluster[idx_g, 0], embeddings_2d_cluster[idx_g, 1],
                    c=[colors.get(g,'gray')], s=30, alpha=0.75, label=g)
axes[1].set_title("By Source Group"); axes[1].legend(fontsize=8)
axes[1].set_xlabel("UMAP-1"); axes[1].set_ylabel("UMAP-2")
plt.tight_layout()
fig_path = FIGURES_DIR / 'cluster_analysis_visualization.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ Saved: {fig_path}")

# Per-model composition bar chart
fig2, ax2 = plt.subplots(figsize=(8, 4))
x = np.arange(best_k); w = 0.18
for gi, g in enumerate(GROUP_NAMES_C):
    fracs = comp_df[comp_df['group']==g].sort_values('gmm_cluster')['fraction'].values
    ax2.bar(x + gi*w, fracs, w, label=g, color=colors.get(g, f'C{gi}'), alpha=0.85)
ax2.set_xticks(x + w*1.5); ax2.set_xticklabels([f'GMM-{k}' for k in range(best_k)])
ax2.set_ylabel("Fraction of group"); ax2.set_title("GMM cluster composition per model")
ax2.legend(); plt.tight_layout()
fig2.savefig(FIGURES_DIR / 'cluster_composition_per_model.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: cluster_composition_per_model.png")


## Analysis 1.5: LDA Topic–Cluster Correspondence

Cross-validate LDA topics (Analysis 1.1) against Ward embedding clusters (Analysis 1.3). Determines whether topics and clusters are redundant or complementary.

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import textwrap

print("="*85)
print("ANALYSIS 1.5: LDA TOPIC–CLUSTER CORRESPONDENCE")
print("="*85)

if 'ward_labels' not in globals() or 'doc_topic_df' not in globals():
    print("⚠  ward_labels or doc_topic_df not found — run Analysis 1.3 and 1.1 first.")
else:
    topic_cols_c = sorted([c for c in doc_topic_df.columns if c.startswith('Topic_')])

    # Step 1: Align LDA topics with Ward cluster labels and descriptive display labels.
    _cluster_cols = ['title', 'ward_cluster', 'ward_cluster_name']
    if 'ward_cluster_display_label' in proposal_meta.columns:
        _cluster_cols.append('ward_cluster_display_label')
    _align_df = proposal_meta[_cluster_cols].copy()
    _align_df['_tnorm'] = _align_df['title'].str.strip().str.lower() \
                           if 'title' in _align_df.columns else _align_df.index.astype(str)
    _dtdf_c = doc_topic_df.copy()
    _dtdf_c['_tnorm'] = _dtdf_c['title'].str.strip().str.lower() \
                         if 'title' in _dtdf_c.columns else ''
    _joined = _dtdf_c.merge(_align_df, on='_tnorm', how='inner', suffixes=('', '_cluster'))

    if len(_joined) == 0:
        print("⚠  Could not align doc_topic_df with proposal_meta on title.")
    else:
        _joined['lda_topic'] = _joined[topic_cols_c].idxmax(axis=1)
        _joined['ward_k']    = _joined['ward_cluster']

        # Use existing LDA contrastive labels if Analysis 1.1 created them; otherwise build them here.
        if 'lda_topic_label_map' not in globals() and 'build_contrastive_phrase_labels' in globals() and 'all_abstracts' in globals():
            _all_dom = doc_topic_df[topic_cols_c].idxmax(axis=1).to_numpy()
            _lda_order = sorted(topic_cols_c, key=lambda s: int(s.split('_')[-1]))
            lda_topic_label_map, lda_topic_label_info_df = build_contrastive_phrase_labels(
                all_abstracts,
                _all_dom,
                label_order=_lda_order,
                n_terms=4,
                min_df=1,
                max_df=0.85,
                prefix='Topic',
            )
            lda_topic_label_info_df = lda_topic_label_info_df.rename(columns={'region': 'topic'})
            lda_topic_label_info_df.to_csv(TABLES_DIR / 'lda_topic_contrastive_labels.csv', index=False)
            print(f"✓ Built fallback LDA contrastive labels and saved: {TABLES_DIR / 'lda_topic_contrastive_labels.csv'}")

        _lda_label_map = globals().get('lda_topic_label_map', {})
        _joined['lda_topic_display'] = _joined['lda_topic'].map(lambda t: f"{t}: {_lda_label_map.get(t, t)}")
        if 'ward_cluster_display_label' in _joined.columns:
            _joined['ward_cluster_display'] = _joined.apply(
                lambda r: f"{r['ward_cluster_name']}: {r['ward_cluster_display_label']}", axis=1
            )
        else:
            _joined['ward_cluster_display'] = _joined['ward_cluster_name']

        # Step 2: ARI and NMI on raw assignments, not display labels.
        ari_lda_ward = adjusted_rand_score(_joined['ward_k'], _joined['lda_topic'])
        nmi_lda_ward = normalized_mutual_info_score(_joined['ward_k'], _joined['lda_topic'])
        print(f"\nAdjusted Rand Index (LDA topics vs Ward clusters): {ari_lda_ward:.4f}")
        print(f"Normalized Mutual Information:                       {nmi_lda_ward:.4f}")
        if ari_lda_ward >= 0.3:
            print("→ HIGH agreement: topics and clusters capture same subfield structure.")
            print("  Use Ward cluster labels as primary thematic axis; LDA topics as supplementary.")
        else:
            print("→ LOW agreement: LDA=lexical structure, clusters=embedding-space structure.")
            print("  Report both as complementary axes.")

        # Step 3: Contingency table with descriptive labels.
        contingency_lda = pd.crosstab(_joined['lda_topic_display'], _joined['ward_cluster_display'])
        print(f"\nContingency table (rows=LDA topic labels, cols=Ward cluster labels):")
        print(contingency_lda.to_string())

        # Step 4: Decision rule
        print(f"\nDecision rule: "
              f"{'Ward primary, LDA supplementary' if ari_lda_ward>=0.3 else 'report both as complementary'}")

        # Step 5: Visualization with readable LDA and cluster labels.
        fig_w = max(9, 1.8 * len(contingency_lda.columns) + 4)
        fig_h = max(5.5, 1.0 * len(contingency_lda.index) + 2.5)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        im = ax.imshow(contingency_lda.values, cmap='Blues', aspect='auto')
        ax.set_xticks(range(len(contingency_lda.columns)))
        ax.set_xticklabels([textwrap.fill(str(c), width=28) for c in contingency_lda.columns], rotation=25, ha='right')
        ax.set_yticks(range(len(contingency_lda.index)))
        ax.set_yticklabels([textwrap.fill(str(c), width=34) for c in contingency_lda.index])
        ax.set_xlabel("Ward embedding cluster label")
        ax.set_ylabel("LDA lexical topic label")
        ax.set_title(f"LDA Topic–Ward Cluster Correspondence\n(ARI={ari_lda_ward:.3f}, NMI={nmi_lda_ward:.3f})")
        for i in range(contingency_lda.shape[0]):
            for j in range(contingency_lda.shape[1]):
                ax.text(j, i, str(contingency_lda.values[i,j]),
                        ha='center', va='center', fontsize=11, fontweight='bold')
        plt.colorbar(im, ax=ax, label='Proposal count')
        plt.tight_layout()
        fig_path = FIGURES_DIR / 'topic_cluster_correspondence.png'
        fig.savefig(fig_path, dpi=170, bbox_inches='tight')
        plt.show()
        print(f"✓ Saved: {fig_path}")

        contingency_lda.to_csv(TABLES_DIR / 'topic_cluster_contingency.csv')
        _joined[['title', 'lda_topic', 'lda_topic_display', 'ward_cluster_name', 'ward_cluster_display']].to_csv(
            TABLES_DIR / 'topic_cluster_assignment_labels.csv', index=False
        )
        pd.DataFrame([{'metric':'ARI','value':ari_lda_ward},
                      {'metric':'NMI','value':nmi_lda_ward}])\
          .to_csv(TABLES_DIR / 'topic_cluster_agreement.csv', index=False)
        print("✓ Saved: topic_cluster_contingency.csv, topic_cluster_assignment_labels.csv, topic_cluster_agreement.csv")


### PART I Summary

In [ ]:
print("="*85)
print("PART I SUMMARY: THEMATIC AND CLUSTER ANALYSIS")
print("="*85)

# ── Topic Modeling (Analysis 1.1) ──────────────────────────────────────────
print()
print("1. TOPIC MODELING (LDA — exploratory)")
print("-"*60)
if 'lda_model' in globals() and 'doc_term_matrix' in globals() and 'topics' in globals():
    print(f"   n_topics={n_topics}, alpha=0.5, beta=0.5")
    print(f"   Perplexity: {lda_model.perplexity(doc_term_matrix):.2f}")
    for i, topic_words in enumerate(topics):
        print(f"   Topic {i+1}: {', '.join(topic_words[:6])}...")
else:
    print("   ⚠  LDA not yet run.")

# ── Topic Distribution by Model (Analysis 1.2) ────────────────────────────
print()
print("2. TOPIC DISTRIBUTION PER MODEL (Analysis 1.2)")
print("-"*60)
if 'obs_chi2' in globals() and 'p_chi2_4g' in globals():
    print(f"   4-group chi2={obs_chi2:.4f}, perm-p={p_chi2_4g:.4f} "
          f"({'significant' if p_chi2_4g<0.05 else 'not significant'})")
else:
    print("   ⚠  Analysis 1.2 not yet run (obs_chi2/p_chi2_4g missing).")

if 'fisher_df' in globals() and 'q_holm' in fisher_df.columns:
    sig_f = fisher_df[fisher_df['q_holm'] < 0.05]
    print(f"   Per-topic Fisher (Holm): {len(sig_f)}/{len(fisher_df)} significant")
    if len(sig_f):
        print(f"   Significant comparisons: {sig_f[['topic','model_vs_human','p_value','q_holm']].to_string(index=False)}")

if 'ent_df' in globals():
    print("   Shannon entropy (H) per model:")
    for _, er in ent_df.iterrows():
        print(f"     {er['model']:25s}  H={er['H']:.4f} (norm={er['H_norm']:.4f}), "
              f"covered={er['topics_covered']}/{n_topics}, dominant={er['dominant_topic']}")

# ── Embedding Cluster Structure (Analysis 1.3 — Ward) ─────────────────────
print()
print("3. EMBEDDING CLUSTER STRUCTURE — WARD (Analysis 1.3)")
print("-"*60)
if 'best_k_ward' in globals() and 'ward_labels' in globals():
    print(f"   Best k={best_k_ward} (by silhouette); cluster sizes: {list(np.bincount(ward_labels))}")
    if 'k_scores' in globals():
        sil = k_scores[best_k_ward]['silhouette']
        ch  = k_scores[best_k_ward]['calinski_harabasz']
        print(f"   silhouette={sil:.4f}, calinski-harabasz={ch:.1f}")
else:
    print("   ⚠  Analysis 1.3 not yet run.")

# ── GMM Cluster Segregation (Analysis 1.4) ────────────────────────────────
print()
print("4. GMM CLUSTER SEGREGATION (Analysis 1.4)")
print("-"*60)
if 'best_k' in globals() and 'nmi_observed' in globals():
    print(f"   GMM k={best_k} (by BIC)")
    if 'composition_df' in globals():
        dom = composition_df['dominance'].value_counts().to_dict()
        print(f"   Human-dominated={dom.get('Human-dominated',0)}, "
              f"AI-dominated={dom.get('AI-dominated',0)}, Mixed={dom.get('Mixed',0)}")
    print(f"   NMI={nmi_observed:.4f} (p={p_nmi:.4f}{'*' if p_nmi<0.05 else ''})")
    print(f"   ARI={ari_observed:.4f} (p={p_ari:.4f}{'*' if p_ari<0.05 else ''})")
    if 'between_within_ratio' in globals():
        print(f"   Between/Within ratio={between_within_ratio:.4f} (p={p_ratio:.4f})")
else:
    print("   ⚠  Analysis 1.4 not yet run.")

# ── LDA–Cluster Correspondence (Analysis 1.5) ─────────────────────────────
print()
print("5. LDA–CLUSTER CORRESPONDENCE (Analysis 1.5)")
print("-"*60)
if 'ari_lda_ward' in globals():
    print(f"   ARI={ari_lda_ward:.4f}, NMI={nmi_lda_ward:.4f}")
    print(f"   → {'HIGH agreement: Ward primary, LDA supplementary' if ari_lda_ward>=0.3 else 'LOW agreement: report both as complementary'}")
else:
    print("   ⚠  Analysis 1.5 not yet run.")

# ── Overall interpretation ─────────────────────────────────────────────────
print()
print("="*85)
print("OVERALL INTERPRETATION")
print("="*85)
if 'p_chi2_4g' in globals():
    msg = "differ in topic distributions" if p_chi2_4g < 0.05 else "no significant difference in topic distributions"
    print(f"  Topics: {msg} (p={p_chi2_4g:.4f})")
if 'nmi_observed' in globals():
    if p_nmi < 0.05 or p_ari < 0.05:
        print("  Clustering: SEGREGATION — Human and AI occupy distinct semantic regions")
    elif p_nmi > 0.10 and p_ari > 0.10:
        print("  Clustering: INTEGRATION — Human and AI ideas intermixed")
    else:
        print("  Clustering: INTERMEDIATE — Some thematic patterns, not strict segregation")

print()
print("  LIMITATIONS: small sample (n=23 Human, n=69 AI); LDA results exploratory;")
print("  segregation may reflect prompt differences, not inherent ideation differences.")
print("="*85)


# PART II: DIVERSITY

## Analysis 2.1: Within-Group Pairwise Diversity (Remote-Clique + proposal-level mean pairwise distance)

This section reports exact group-level Remote-Clique and proposal-level mean pairwise distance for inferential tests.

In [ ]:

print('='*85)
print('ANALYSIS 1.1: WITHIN-GROUP PAIRWISE DIVERSITY')
print('='*85)

summary_rows = []
proposal_rows = []

# Backward-compatible containers used by existing plotting cells
human_idx = GROUPS['Human']
ai_idx = GROUPS['All AI']
human_pairwise = D_pp[np.ix_(human_idx, human_idx)][np.triu_indices(len(human_idx), k=1)]
ai_pairwise = D_pp[np.ix_(ai_idx, ai_idx)][np.triu_indices(len(ai_idx), k=1)]
model_pairwise = {}
model_embeddings_dict = {}

for g, gc in group_cache.items():
    idx = gc['idx']
    Dg = gc['D']
    n = gc['n']
    tri = Dg[np.triu_indices(n, k=1)] if n > 1 else np.array([])
    mpd = proposal_mean_pairwise_from_submatrix(Dg) if n > 1 else np.zeros(n)
    remote = group_remote_clique(Dg)

    if g in ai_models:
        model_pairwise[g] = tri
        model_embeddings_dict[g] = X_prop[idx]

    summary_rows.append({
        'group': g,
        'n': n,
        'remote_clique': remote,
        'upper_tri_mean': float(np.mean(tri)) if len(tri) else np.nan,
        'upper_tri_median': float(np.median(tri)) if len(tri) else np.nan,
        'upper_tri_sd': float(np.std(tri)) if len(tri) else np.nan,
        'proposal_mean_pairwise_mean': float(np.mean(mpd)) if len(mpd) else np.nan,
        'proposal_mean_pairwise_median': float(np.median(mpd)) if len(mpd) else np.nan,
        'proposal_mean_pairwise_sd': float(np.std(mpd)) if len(mpd) else np.nan,
        'proposal_mean_pairwise_iqr': float(np.percentile(mpd, 75) - np.percentile(mpd, 25)) if len(mpd) else np.nan,
    })

    for pos, pid in enumerate(idx):
        proposal_rows.append({
            'proposal_uid': proposal_meta.loc[pid, 'proposal_uid'],
            'title': proposal_meta.loc[pid, 'title'],
            'group_model': proposal_meta.loc[pid, 'group_model'],
            'group_binary': proposal_meta.loc[pid, 'group_binary'],
            'mean_pairwise_dist': float(mpd[pos]) if len(mpd) else np.nan,
        })

pairwise_group_summary_df = pd.DataFrame(summary_rows)
pairwise_proposal_df = pd.DataFrame(proposal_rows)

# backward-compatible variables
human_pairwise_proposal_means = pairwise_proposal_df.loc[pairwise_proposal_df['group_binary']=='Human', 'mean_pairwise_dist'].to_numpy()
ai_pairwise_proposal_means = pairwise_proposal_df.loc[pairwise_proposal_df['group_binary']=='AI', 'mean_pairwise_dist'].to_numpy()
model_pairwise_proposal_means = {
    m: pairwise_proposal_df.loc[pairwise_proposal_df['group_model']==m, 'mean_pairwise_dist'].to_numpy()
    for m in ai_models
}

print(pairwise_group_summary_df[['group', 'n', 'remote_clique', 'proposal_mean_pairwise_mean']].to_string(index=False))

# Inference: AI vs Human on proposal-level mean pairwise distance
tests = []
human_vals = pairwise_proposal_df.loc[pairwise_proposal_df['group_binary']=='Human', 'mean_pairwise_dist'].to_numpy()
for g in ['All AI'] + ai_models:
    g_vals = pairwise_proposal_df.loc[
        (pairwise_proposal_df['group_binary']=='AI') if g=='All AI' else (pairwise_proposal_df['group_model']==g),
        'mean_pairwise_dist'
    ].to_numpy()
    if len(g_vals) == 0:
        continue
    r = run_group_comparison(g_vals, human_vals, n_permutations=10000, n_boot=5000, random_state=42)
    r.update({'comparison': f'{g} vs Human', 'group1': g, 'group2': 'Human', 'n_group1': len(g_vals), 'n_group2': len(human_vals)})
    tests.append(r)

pairwise_tests_df = pd.DataFrame(tests)
if len(pairwise_tests_df):
    pairwise_tests_df = apply_multiple_testing(pairwise_tests_df, p_cols=('p_value_mw', 'p_value_perm'), method='holm')

pairwise_group_summary_df.to_csv(TABLES_DIR / 'diversity_remote_clique_group_summary.csv', index=False)
pairwise_proposal_df.to_csv(TABLES_DIR / 'diversity_pairwise_proposal_level.csv', index=False)
pairwise_tests_df.to_csv(TABLES_DIR / 'diversity_pairwise_tests.csv', index=False)

print('\nSaved diversity_remote_clique_group_summary.csv')
print('Saved diversity_pairwise_proposal_level.csv')
print('Saved diversity_pairwise_tests.csv')


In [ ]:

# Keep a compact stats summary for existing visualization cells
comparison_df = pairwise_tests_df.copy()
comparison_results = comparison_df.to_dict('records') if len(comparison_df) else []
print('Pairwise proposal-level inference ready (variable: pairwise_tests_df).')


In [ ]:
# visualization (PAIRWISE distances + effect size/significance), using your predefined `colors`
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def bootstrap_cliffs_delta_ci(group1, group2, n_boot=2000, random_state=42, alpha=0.05):
    rng = np.random.default_rng(random_state)
    g1 = np.asarray(group1, dtype=float)
    g2 = np.asarray(group2, dtype=float)

    boots = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        b1 = rng.choice(g1, size=len(g1), replace=True)
        b2 = rng.choice(g2, size=len(g2), replace=True)
        boots[i] = cliffs_delta(b1, b2)  # reuse your existing function

    lo = np.quantile(boots, alpha / 2)
    hi = np.quantile(boots, 1 - alpha / 2)
    return lo, hi


# ---- Build long df for PAIRWISE distances (matches your summary table) ----
plot_specs = [("Human", human_pairwise)]

for m in ai_models:
    if len(model_pairwise.get(m, [])) > 0:
        plot_specs.append((m, model_pairwise[m]))

plot_specs.append(("All AI (combined)", ai_pairwise))

dist_df = pd.concat(
    [pd.DataFrame({"group": g, "pairwise_distance": np.asarray(v, dtype=float)}) for g, v in plot_specs],
    ignore_index=True,
)

group_order = [g for g, _ in plot_specs]

palette = {g: colors.get(g, "#808080") for g in group_order}
palette["All AI (combined)"] = palette.get("All AI (combined)", "#808080")


# ---- Recompute stats on PAIRWISE distances (so right panel matches left panel’s metric) ----
comparison_specs = [("All AI (combined)", ai_pairwise)] + [
    (m, model_pairwise[m]) for m in ai_models if len(model_pairwise.get(m, [])) > 0
]

comparison_results = []
for group_name, vals in comparison_specs:
    res = run_group_comparison(vals, human_pairwise, n_permutations=10000, n_boot=5000, random_state=42)
    res["group"] = group_name
    comparison_results.append(res)

eff_df = pd.DataFrame(comparison_results)
eff_df = apply_multiple_testing(eff_df, p_cols=("p_value_mw", "p_value_perm"), method="holm")

# bootstrap CI for δ (PAIRWISE; group vs Human)
delta_ci_low, delta_ci_high = [], []
for g in eff_df["group"].tolist():
    gvals = ai_pairwise if g == "All AI (combined)" else model_pairwise[g]
    lo, hi = bootstrap_cliffs_delta_ci(gvals, human_pairwise, n_boot=2000, random_state=42)
    delta_ci_low.append(lo)
    delta_ci_high.append(hi)

eff_df["delta_ci_low"] = delta_ci_low
eff_df["delta_ci_high"] = delta_ci_high

eff_order = [g for g in group_order if g not in ("Human",)]
eff_df["group"] = pd.Categorical(eff_df["group"], categories=eff_order, ordered=True)
eff_df = eff_df.sort_values("group")


def fmt_p(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "<1e-4"
    return f"{p:.4f}"


# ---- Plot: left = pairwise distribution, right = δ + CI + Holm p-values ----
sns.set_theme(style="whitegrid", context="talk")

fig = plt.figure(figsize=(16, 7), dpi=140)
gs = fig.add_gridspec(1, 2, width_ratios=[1.7, 1.0], wspace=0.18)
axL = fig.add_subplot(gs[0, 0])
axR = fig.add_subplot(gs[0, 1])

# Left panel: PAIRWISE distance distributions.
# Pairwise-distance points are pair observations, not proposal observations, so funding/ranking
# metadata are not applicable in this specific panel.
styled_boxplot_with_points(
    axL,
    dist_df,
    x='group',
    y='pairwise_distance',
    order=group_order,
    palette=palette,
    box_width=0.45,
    jitter=0.15,
    point_size=20,
    point_alpha=0.50,
    random_state=42,
)
if axL.get_legend() is not None:
    axL.get_legend().remove()

axL.set_title("Pairwise cosine distance (box + mean 95% CI)")
axL.set_xlabel("")
axL.set_ylabel("Pairwise cosine distance")
axL.tick_params(axis="x", rotation=35, labelsize=10)

# Right panel: Cliff’s δ with bootstrap CI + Holm-adjusted p-values (PAIRWISE)
axR.axvline(0, color="black", linewidth=1.0, alpha=0.8)

ypos = np.arange(len(eff_df))
axR.set_yticks(ypos)
axR.set_yticklabels(eff_df["group"].astype(str))
axR.invert_yaxis()

for i, row in enumerate(eff_df.itertuples(index=False)):
    g = str(row.group)
    c = palette.get(g, "#808080")

    axR.errorbar(
        row.delta,
        i,
        xerr=np.array([[row.delta - row.delta_ci_low], [row.delta_ci_high - row.delta]]),
        fmt="o",
        color=c,
        ecolor=c,
        elinewidth=1.2,
        capsize=3,
    )

    txt = (
        f"MW(Holm)={fmt_p(getattr(row, 'p_value_mw_adj_holm', np.nan))} | "
        f"Perm(Holm)={fmt_p(getattr(row, 'p_value_perm_adj_holm', np.nan))}"
    )
    axR.text(1.03, i, txt, va="center", ha="left", fontsize=10, clip_on=False)

axR.set_title("Effect size (AI − Human) + significance")
axR.set_xlabel("Cliff's δ (bootstrap 95% CI)")
axR.set_ylabel("")
axR.set_xlim(-1.05, 1.05)

plt.suptitle("Pairwise distance distribution + effect size + significance", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Visualization: Improved distribution comparison for all groups
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

# Layout: 4 rows: ridge plot, standardized boxplot,
#         summary table (full width), log-scale histogram (full width, new row)
fig = plt.figure(figsize=(22, 18))
gs = fig.add_gridspec(4, 1, hspace=0.45)

group_order = ['Human'] + [m for m in ai_models if len(model_pairwise[m]) > 0]

# ── PANEL 1 (Row 0, full width): Ridge plot — x-range covers ALL group medians/means
# Determine x-limit: at least 0.2, but extend to fit the furthest median or mean + margin
all_stats = []
for group in group_order:
    d = human_pairwise if group == 'Human' else model_pairwise[group]
    all_stats += [float(np.median(d)), float(np.mean(d))]
x_limit = max(0.2, max(all_stats) * 1.12)   # 12% margin past the furthest stat line

ax1 = fig.add_subplot(gs[0])
RIDGE_HEIGHT = 1.0   # normalized peak height for every group (equal visual weight)
RIDGE_GAP    = 0.3   # gap between ridges
offset = 0

for idx, group in enumerate(group_order):
    data = human_pairwise if group == 'Human' else model_pairwise[group]
    grp_color = colors.get(group, 'gray')

    if len(data) > 1:
        kde = gaussian_kde(data)
        x_range = np.linspace(0, x_limit, 500)
        density = kde(x_range)
        # Normalize so every ridge peaks at RIDGE_HEIGHT (equal visual weight)
        density_norm = density / density.max() * RIDGE_HEIGHT

        ax1.fill_between(x_range, offset + density_norm, offset,
                         color=grp_color, alpha=0.55,
                         edgecolor=grp_color, linewidth=1.5, label=group)

        ridge_top = offset + RIDGE_HEIGHT

        # Median — solid line spanning full ridge height
        median_val = float(np.median(data))
        ax1.plot([median_val, median_val], [offset, ridge_top],
                 color=grp_color, linestyle='-', linewidth=2.5, alpha=1.0, zorder=5)

        # Mean — dashed line spanning full ridge height
        mean_val = float(np.mean(data))
        ax1.plot([mean_val, mean_val], [offset, ridge_top],
                 color=grp_color, linestyle='--', linewidth=2.0, alpha=0.9, zorder=5)

        offset += RIDGE_HEIGHT + RIDGE_GAP

ax1.set_xlabel('Pairwise Cosine Distance', fontsize=13, fontweight='bold')
ax1.set_ylabel('Density (stacked by group)', fontsize=13, fontweight='bold')
ax1.set_title('Ridge Plot: Pairwise Distance Distributions\nSolid = Median, Dashed = Mean (color-coded by group)',
              fontsize=14, fontweight='bold')
ax1.set_xlim(0, x_limit)
ax1.set_yticks([])
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(alpha=0.3, axis='x')

# -- PANEL 2 (Row 1, full width): Standardized boxplots with jittered observations
ax3 = fig.add_subplot(gs[1])

box_data = []
for dist, label in [(human_pairwise, 'Human')] + [(model_pairwise[m], m) for m in ai_models if len(model_pairwise[m]) > 0]:
    for val in dist:
        box_data.append({'Distance': val, 'Group': label})

box_df = pd.DataFrame(box_data)
palette_dict = {g: colors.get(g, 'gray') for g in group_order}

# Pairwise-distance observations are proposal pairs, so proposal-level funding/ranking
# metadata cannot be assigned to a single datapoint here.
styled_boxplot_with_points(
    ax3,
    box_df,
    x='Group',
    y='Distance',
    order=group_order,
    palette=palette_dict,
    box_width=0.45,
    jitter=0.15,
    point_size=20,
    point_alpha=0.50,
    random_state=42,
)

ax3.set_ylabel('Pairwise Cosine Distance', fontsize=13, fontweight='bold')
ax3.set_title('Pairwise Distance Boxplots\n(Box = median/IQR; diamond/error bar = mean bootstrap 95% CI)',
              fontsize=14, fontweight='bold')
ax3.tick_params(axis='x', rotation=20, labelsize=11)
ax3.grid(alpha=0.3, axis='y')

# ── PANEL 3 (Row 2, full width): Statistical summary table
ax4 = fig.add_subplot(gs[2])
ax4.axis('off')

# Create summary statistics
summary_data = []
for group in group_order:
    if group == 'Human':
        data = human_pairwise
    else:
        data = model_pairwise[group]
    
    summary_data.append([
        group,
        f"{len(data)}",
        f"{data.mean():.4f}",
        f"{np.median(data):.4f}",
        f"{np.percentile(data, 25):.4f}",
        f"{np.percentile(data, 75):.4f}",
        f"{data.max():.4f}",
        f"{data.mean() - np.median(data):.4f}"
    ])

# Create table
table = ax4.table(cellText=summary_data,
                 colLabels=['Group', 'N Pairs', 'Mean', 'Median', 'Q25', 'Q75', 'Max', 'Mean-Median\n(outlier effect)'],
                 cellLoc='center',
                 loc='center',
                 colWidths=[0.18, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.12])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Color code the header
for i in range(8):
    table[(0, i)].set_facecolor('#E8E8E8')
    table[(0, i)].set_text_props(weight='bold')

# Color code the groups
for i, group in enumerate(group_order):
    table[(i+1, 0)].set_facecolor(colors.get(group, 'gray'))
    table[(i+1, 0)].set_alpha(0.3)

ax4.text(0.5, -0.15, 
         'Key: Mean > Median indicates right-skewed distribution with high outliers pulling mean up',
         ha='center', va='top', transform=ax4.transAxes, fontsize=11, style='italic',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# ── PANEL 4 (Row 3, full width): Full-range histogram with log-scale y-axis
ax2 = fig.add_subplot(gs[3])
for group in group_order:
    data = human_pairwise if group == 'Human' else model_pairwise[group]
    ax2.hist(data, bins=50, alpha=0.5, label=group,
             color=colors.get(group, 'gray'), edgecolor='black', linewidth=0.8)

ax2.set_xlabel('Pairwise Cosine Distance', fontsize=13, fontweight='bold')
ax2.set_ylabel('Count (log scale)', fontsize=13, fontweight='bold')
ax2.set_title('Full Range View (log scale for clarity)', fontsize=14, fontweight='bold')
ax2.set_yscale('log')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.suptitle('Pairwise Diversity Analysis: Multiple Views for Clarity', 
             fontsize=16, fontweight='bold', y=0.995)

plt.savefig(FIGURES_DIR / 'pairwise_diversity_by_model.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Figure saved to: {FIGURES_DIR / 'pairwise_diversity_by_model.png'}")


In [ ]:
# Box plot summary + Cliff's delta panel for pairwise distances
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, (ax, ax_delta) = plt.subplots(2, 1, figsize=(9, 11),
                                    gridspec_kw={'height_ratios': [3, 2], 'hspace': 0.45})

box_arrays = []
for group in group_order:
    d = human_pairwise if group == 'Human' else model_pairwise[group]
    box_arrays.append(d)

# Standardized boxplot grammar. Pairwise-distance points are pair observations,
# not proposal observations, so funding/ranking metadata are not applicable here.
box_plot_df = pd.concat(
    [pd.DataFrame({'Group': group, 'Pairwise Cosine Distance': np.asarray(vals, dtype=float)})
     for group, vals in zip(group_order, box_arrays)],
    ignore_index=True,
)
styled_boxplot_with_points(
    ax,
    box_plot_df,
    x='Group',
    y='Pairwise Cosine Distance',
    order=group_order,
    palette={g: colors.get(g, 'gray') for g in group_order},
    box_width=0.45,
    jitter=0.15,
    point_size=20,
    point_alpha=0.50,
    random_state=42,
)
ax.set_xticks(range(len(group_order)))
ax.set_xticklabels(group_order, fontsize=11, rotation=15, ha='right')
ax.set_ylabel('Pairwise Cosine Distance', fontsize=12, fontweight='bold')
ax.set_title('Pairwise Distance by Group\nBox = IQR/Median  |  Diamond/Error bar = Mean bootstrap 95% CI',
             fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.35)
ax.set_ylim(bottom=0)


plt.savefig(FIGURES_DIR / 'pairwise_diversity_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print(f"{'Group':<28} {'N':>5}  {'Mean':>7}  {'Median':>7}  {'SD':>7}  {'Min':>7}  {'Max':>7}")
print('-' * 72)
for group, d in zip(group_order, box_arrays):
    print(f"{group:<28} {len(d):>5}  {np.mean(d):>7.4f}  {np.median(d):>7.4f}"
          f"  {np.std(d, ddof=1):>7.4f}  {d.min():>7.4f}  {d.max():>7.4f}")


## Analysis 2.1b: Pairwise Distance Bimodality Test

Hartigan's dip test (non-parametric) per group + GMM with BIC-based k selection. Does NOT assume k=2 — Claude is expected unimodal (k=1).

In [ ]:
# Analysis 2.1b: Bimodality test + per-group GMM
try:
    import diptest
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'diptest'])
    import diptest

from sklearn.mixture import GaussianMixture
from scipy.stats import norm as sp_norm

print("="*85)
print("ANALYSIS 2.1b: PAIRWISE DISTANCE BIMODALITY TEST + GMM")
print("="*85)

GROUPS_BIM = ['Human', 'Claude', 'Gemini', 'GPT-5.2']
bim_rows, gmm_rows = [], []

fig, axes = plt.subplots(1, len(GROUPS_BIM), figsize=(16, 4), sharey=False)
fig.suptitle("Pairwise Distance Distributions — Bimodality Test (GMM best-k per group)",
             fontsize=12, fontweight='bold')

for ax, g in zip(axes, GROUPS_BIM):
    if g == 'Human':
        dist_vec = human_pairwise
    else:
        dist_vec = model_pairwise.get(g, np.array([]))
    if len(dist_vec) == 0:
        ax.set_title(f"{g}\n(no data)"); continue

    dip_stat, dip_p = diptest.diptest(dist_vec)

    best_bic, best_k_g, best_gmm = np.inf, 1, None
    bic_vals = {}
    for k in [1, 2, 3]:
        gm = GaussianMixture(n_components=k, covariance_type='full',
                             random_state=42, n_init=5)
        gm.fit(dist_vec.reshape(-1,1))
        b = gm.bic(dist_vec.reshape(-1,1))
        bic_vals[k] = b
        if b < best_bic:
            best_bic, best_k_g, best_gmm = b, k, gm

    bim_rows.append({'group':g, 'dip_stat':dip_stat, 'dip_p':dip_p,
                     'best_k_gmm': best_k_g, **{f'BIC_k{k}': bic_vals[k] for k in [1,2,3]}})

    valley_x = None
    if best_k_g >= 2:
        means = np.sort(best_gmm.means_.flatten())
        x_range = np.linspace(means[0], means[-1], 200)
        dens = np.exp(best_gmm.score_samples(x_range.reshape(-1,1)))
        valley_x = x_range[np.argmin(dens)]
    for comp in range(best_k_g):
        w  = float(best_gmm.weights_[comp])
        mu = float(best_gmm.means_[comp][0])
        sg = float(np.sqrt(best_gmm.covariances_[comp][0][0]))
        gmm_rows.append({'group':g,'component':comp,'weight':w,'mean':mu,'std':sg,
                         'best_k':best_k_g})

    x_plot = np.linspace(dist_vec.min(), dist_vec.max(), 300)
    ax.hist(dist_vec, bins=30, density=True, color=colors.get(g,'gray'), alpha=0.35)
    for comp in range(best_k_g):
        w  = best_gmm.weights_[comp]
        mu = best_gmm.means_[comp][0]
        sg = np.sqrt(best_gmm.covariances_[comp][0][0])
        ax.plot(x_plot, w * sp_norm.pdf(x_plot, mu, sg), '--', lw=1.5, alpha=0.8)
    ax.plot(x_plot, np.exp(best_gmm.score_samples(x_plot.reshape(-1,1))),
            '-', lw=2, color='black', label=f'GMM k={best_k_g}')
    if valley_x is not None:
        ax.axvline(valley_x, color='red', ls=':', lw=1.5, alpha=0.7)
    ax.set_title(f"{g}\ndip p={dip_p:.3f}, k={best_k_g}", fontsize=9)
    ax.set_xlabel("Pairwise cosine distance"); ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'pairwise_diversity_bimodality_gmm.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: pairwise_diversity_bimodality_gmm.png")

bim_df = pd.DataFrame(bim_rows)
gmm_df = pd.DataFrame(gmm_rows)
print("\nBimodality test results:")
print(bim_df.to_string(index=False))
bim_df.to_csv(TABLES_DIR / 'diversity_pairwise_bimodality_tests.csv', index=False)
gmm_df.to_csv(TABLES_DIR / 'diversity_pairwise_gmm_summary.csv', index=False)
print("✓ Saved: diversity_pairwise_bimodality_tests.csv, diversity_pairwise_gmm_summary.csv")


## Analysis 2.1c: Cross-Group Topic Space Alignment

For each AI proposal: minimum cosine distance to any human proposal (AI-to-nearest-Human). Tests whether AI models explore the same intellectual territory as humans.

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

print("="*85)
print("ANALYSIS 2.1c: CROSS-GROUP TOPIC SPACE ALIGNMENT (nearest-human distances)")
print("="*85)

hu_idx_c = np.where(proposal_meta['group_binary'] == 'Human')[0]
grp_col_c2 = 'group_model' if 'group_model' in proposal_meta.columns else 'group'
ai_models_c = [g for g in ['Claude','Gemini','GPT-5.2']]

rows_ai2hu, rows_hu2ai, test_rows = [], [], []
for model_g in ai_models_c:
    if model_g == 'Claude':
        ai_idx_c = np.where(proposal_meta[grp_col_c2].str.contains('Claude',case=False,na=False))[0]
    elif model_g == 'Gemini':
        ai_idx_c = np.where(proposal_meta[grp_col_c2].str.contains('Gemini',case=False,na=False))[0]
    else:
        ai_idx_c = np.where(proposal_meta[grp_col_c2].str.contains('GPT',case=False,na=False))[0]
    if len(ai_idx_c) == 0: continue

    cross_block = D_pp[np.ix_(ai_idx_c, hu_idx_c)]
    ai2hu = cross_block.min(axis=1)
    hu2ai = cross_block.min(axis=0)

    for j, (ai_i, dist) in enumerate(zip(ai_idx_c, ai2hu)):
        hu_nearest = hu_idx_c[cross_block[j,:].argmin()]
        rows_ai2hu.append({'model': model_g, 'ai_idx': int(ai_i),
                           'nearest_human_idx': int(hu_nearest),
                           'min_dist_to_human': float(dist)})
    for j, (hu_i, dist) in enumerate(zip(hu_idx_c, hu2ai)):
        rows_hu2ai.append({'model': model_g, 'human_idx': int(hu_i),
                           'min_dist_to_ai': float(dist)})
    test_rows.append({'model': model_g,
                      'ai2human_mean': float(ai2hu.mean()),
                      'ai2human_sd':   float(ai2hu.std()),
                      'human2ai_mean': float(hu2ai.mean()),
                      'human2ai_sd':   float(hu2ai.std())})
    print(f"  {model_g:20s}  AI→Human: {ai2hu.mean():.4f}±{ai2hu.std():.4f}  "
          f"Human→AI: {hu2ai.mean():.4f}±{hu2ai.std():.4f}")

ai2hu_df = pd.DataFrame(rows_ai2hu)
hu2ai_df = pd.DataFrame(rows_hu2ai)
align_df = pd.DataFrame(test_rows)

# MW tests across AI models
model_pairs_c = [(ai_models_c[i], ai_models_c[j])
                 for i in range(len(ai_models_c)) for j in range(i+1,len(ai_models_c))]
mw_rows = []
for m1, m2 in model_pairs_c:
    d1 = ai2hu_df[ai2hu_df['model']==m1]['min_dist_to_human'].values
    d2 = ai2hu_df[ai2hu_df['model']==m2]['min_dist_to_human'].values
    if len(d1)==0 or len(d2)==0: continue
    u, p = mannwhitneyu(d1, d2, alternative='two-sided')
    mw_rows.append({'comparison': f'{m1} vs {m2}', 'U': float(u), 'p_value': float(p)})
if mw_rows:
    mw_df = pd.DataFrame(mw_rows)
    _, mw_df['q_holm'], _, _ = multipletests(mw_df['p_value'], method='holm')
    print("\nMW tests on AI-to-nearest-Human distances (Holm):"); print(mw_df.to_string(index=False))
    mw_df.to_csv(TABLES_DIR / 'diversity_cross_group_alignment_tests.csv', index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Cross-Group Topic Space Alignment (nearest-neighbor distances)", fontsize=12, fontweight='bold')
for model_g in ai_models_c:
    d = ai2hu_df[ai2hu_df['model']==model_g]['min_dist_to_human'].values
    if len(d)==0: continue
    ax1.scatter(np.full(len(d), model_g), d, color=colors.get(model_g,'gray'), alpha=0.5, s=20, zorder=2)
    ax1.plot([model_g]*2, [d.mean()-d.std(), d.mean()+d.std()],
             '-', lw=3, color=colors.get(model_g,'gray'), alpha=0.8)
    ax1.scatter([model_g], [d.mean()], color=colors.get(model_g,'gray'), s=80, zorder=3,
                edgecolors='black', linewidths=1)
ax1.set_ylabel("Min cosine distance to nearest human")
ax1.set_title("AI-to-nearest-Human distance"); ax1.set_xlabel("AI Model"); ax1.grid(axis='y', alpha=0.3)
for model_g in ai_models_c:
    d = hu2ai_df[hu2ai_df['model']==model_g]['min_dist_to_ai'].values
    if len(d)==0: continue
    ax2.scatter(np.full(len(d), model_g), d, color=colors.get(model_g,'gray'), alpha=0.5, s=20, zorder=2)
    ax2.plot([model_g]*2, [d.mean()-d.std(), d.mean()+d.std()],
             '-', lw=3, color=colors.get(model_g,'gray'), alpha=0.8)
    ax2.scatter([model_g], [d.mean()], color=colors.get(model_g,'gray'), s=80, zorder=3,
                edgecolors='black', linewidths=1)
ax2.set_ylabel("Min cosine distance to nearest AI"); ax2.set_title("Human-to-nearest-AI distance")
ax2.set_xlabel("AI Model"); ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'cross_group_topic_alignment.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: cross_group_topic_alignment.png")

ai2hu_df.to_csv(TABLES_DIR / 'diversity_cross_group_nearest_human.csv', index=False)
align_df.to_csv(TABLES_DIR / 'diversity_cross_group_alignment_tests.csv', index=False)
print("✓ Saved: diversity_cross_group_nearest_human.csv, diversity_cross_group_alignment_tests.csv")
nearest_human_idx_by_ai = {row['ai_idx']: row['nearest_human_idx'] for _, row in ai2hu_df.iterrows()}


## Analysis 2.1d: Within-Cluster and Between-Cluster Diversity

Cluster-controlled diversity comparison using Ward labels from Analysis 1.3. Separates within-subfield diversity from cross-subfield gap.

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

print("="*85)
print("ANALYSIS 2.1d: WITHIN-CLUSTER AND BETWEEN-CLUSTER DIVERSITY")
print("="*85)

if 'ward_labels' not in globals():
    _clust_csv = TABLES_DIR / 'diversity_cluster_membership_by_group.csv'
    if _clust_csv.exists():
        _cm = pd.read_csv(_clust_csv)
        ward_labels = _cm['ward_cluster'].values
        best_k_ward = int(ward_labels.max()) + 1
        cluster_label_map = {i: f'Cluster_{chr(65+i)}' for i in range(best_k_ward)}
        print(f"✓ Loaded ward_labels from {_clust_csv}")
    else:
        print("⚠  ward_labels not found — run Analysis 1.3 first."); ward_labels = None

if ward_labels is not None:
    unique_clusters = np.unique(ward_labels)
    cluster_names_d = {k: f'Cluster_{chr(65+k)}' for k in unique_clusters}
    grp_col_d = 'group_model' if 'group_model' in proposal_meta.columns else 'group'
    GROUP_NAMES_D = ['Human', 'Claude', 'Gemini', 'GPT-5.2']

    within_rows, between_rows = [], []
    for g in GROUP_NAMES_D:
        g_mask = proposal_meta['group_binary'] == 'Human' if g == 'Human' \
                 else proposal_meta[grp_col_d].str.contains(g, case=False, na=False)
        g_idx = proposal_meta[g_mask].index.tolist()
        for k in unique_clusters:
            k_idx = [i for i in g_idx if ward_labels[i] == k]
            if len(k_idx) >= 2:
                sub_D = D_pp[np.ix_(k_idx, k_idx)]
                upper = sub_D[np.triu_indices_from(sub_D, k=1)]
                mean_d = float(upper.mean())
            elif len(k_idx) == 1:
                mean_d = 0.0
            else:
                mean_d = float('nan')
            within_rows.append({'group':g,'cluster':cluster_names_d[k],
                                 'n_proposals':len(k_idx),'mean_pairwise_dist':mean_d})
        if len(unique_clusters) >= 2:
            for ki in range(len(unique_clusters)):
                for kj in range(ki+1, len(unique_clusters)):
                    a_idx = [i for i in g_idx if ward_labels[i] == unique_clusters[ki]]
                    b_idx = [i for i in g_idx if ward_labels[i] == unique_clusters[kj]]
                    if a_idx and b_idx:
                        cross_D_d = D_pp[np.ix_(a_idx, b_idx)]
                        between_rows.append({'group':g,
                            'cluster_pair': f"{cluster_names_d[unique_clusters[ki]]}\u2194{cluster_names_d[unique_clusters[kj]]}",
                            'n_a':len(a_idx),'n_b':len(b_idx),'mean_cross_dist':float(cross_D_d.mean())})

    within_df  = pd.DataFrame(within_rows)
    between_df = pd.DataFrame(between_rows)
    print("\nWithin-cluster diversity (mean pairwise cosine distance):")
    print(within_df.to_string(index=False))
    print("\nBetween-cluster gap:"); print(between_df.to_string(index=False))

    # MW tests
    perm_rows_d = []
    for k in unique_clusters:
        k_name = cluster_names_d[k]
        hu_k_idx = [i for i in proposal_meta[proposal_meta['group_binary']=='Human'].index
                    if ward_labels[i]==k]
        if len(hu_k_idx) < 2: continue
        hu_dists_k = D_pp[np.ix_(hu_k_idx,hu_k_idx)][np.triu_indices(len(hu_k_idx),k=1)]
        for g in ['Claude','Gemini','GPT-5.2']:
            g_k_idx = [i for i in proposal_meta[
                proposal_meta[grp_col_d].str.contains(g,case=False,na=False)].index
                       if ward_labels[i]==k]
            if len(g_k_idx) < 2: continue
            g_dists_k = D_pp[np.ix_(g_k_idx,g_k_idx)][np.triu_indices(len(g_k_idx),k=1)]
            u, p = mannwhitneyu(hu_dists_k, g_dists_k, alternative='two-sided')
            perm_rows_d.append({'cluster':k_name,'model':g,
                                  'human_mean':float(hu_dists_k.mean()),
                                  'model_mean':float(g_dists_k.mean()),
                                  'p_value':float(p)})
    if perm_rows_d:
        perm_df_d = pd.DataFrame(perm_rows_d)
        _, perm_df_d['q_holm'], _, _ = multipletests(perm_df_d['p_value'], method='holm')
        print("\nWithin-cluster diversity tests (Human vs AI, Holm):")
        print(perm_df_d.to_string(index=False))

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Cluster-Controlled Diversity Comparison", fontsize=12, fontweight='bold')
    for ki, k in enumerate(list(unique_clusters)[:2]):
        k_name = cluster_names_d[k]
        df_k = within_df[within_df['cluster']==k_name].copy()
        ax = axes[ki]
        bar_c = [colors.get(g, 'gray') for g in df_k['group']]
        x_labels = [f"{g}\n(n={int(row['n_proposals'])})"
                    for g, (_, row) in zip(df_k['group'], df_k.iterrows())]
        ax.bar(range(len(df_k)), df_k['mean_pairwise_dist'], color=bar_c, alpha=0.85)
        ax.set_xticks(range(len(df_k)))
        ax.set_xticklabels(x_labels, fontsize=9)
        ax.set_title(f"Within-{k_name} diversity")
        ax.set_ylabel("Mean pairwise cosine distance"); ax.tick_params(axis='x', rotation=15)
    ax3 = axes[2]
    df_b_grp = between_df.groupby('group')['mean_cross_dist'].mean().reset_index()
    _group_n = within_df.groupby('group')['n_proposals'].sum()
    df_b_grp['n'] = df_b_grp['group'].map(_group_n).fillna(0).astype(int)
    y_labels = [f"{g} (n={n})" for g, n in zip(df_b_grp['group'], df_b_grp['n'])]
    bar_c3 = [colors.get(g, 'gray') for g in df_b_grp['group']]
    ax3.barh(y_labels, df_b_grp['mean_cross_dist'], color=bar_c3, alpha=0.85)
    ax3.set_title("Between-cluster gap"); ax3.set_xlabel("Mean cross-cluster distance")
    hu_gap = df_b_grp[df_b_grp['group']=='Human']['mean_cross_dist'].values
    if len(hu_gap) > 0:
        ax3.axvline(hu_gap[0], ls='--', color='black', lw=1.5, label='Human baseline')
    ax3.legend(fontsize=8); plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'diversity_cluster_aware_comparison.png', dpi=150, bbox_inches='tight')
    plt.show(); print("✓ Saved: diversity_cluster_aware_comparison.png")

    within_df.to_csv(TABLES_DIR / 'diversity_within_cluster_by_group.csv', index=False)
    between_df.to_csv(TABLES_DIR / 'diversity_between_cluster_gap.csv', index=False)
    print("✓ Saved: diversity_within_cluster_by_group.csv, diversity_between_cluster_gap.csv")


## Analysis 2.2: Centroid Dispersion Metric (mean radius + Span-90)

Compute raw centroid distances, leave-one-out centroid distances, and group Span-90.

### 2.2a Within-group Centroid Dispersion

In [ ]:

print('='*85)
print('ANALYSIS 1.2: CENTROID DISPERSION (RAW + LOO + SPAN90)')
print('='*85)

rows = []
group_rows = []

for g, gc in group_cache.items():
    idx = gc['idx']
    Xg = gc['X']
    n = gc['n']
    d_raw = proposal_centroid_distances(Xg, leave_one_out=False)
    d_loo = proposal_centroid_distances(Xg, leave_one_out=True)
    span90 = safe_percentile(d_raw, 90)

    group_rows.append({
        'group': g,
        'n': n,
        'centroid_mean_raw': float(np.mean(d_raw)) if len(d_raw) else np.nan,
        'centroid_median_raw': float(np.median(d_raw)) if len(d_raw) else np.nan,
        'centroid_sd_raw': float(np.std(d_raw)) if len(d_raw) else np.nan,
        'centroid_var_raw': float(np.var(d_raw)) if len(d_raw) else np.nan,
        'centroid_mean_loo': float(np.mean(d_loo)) if len(d_loo) else np.nan,
        'centroid_median_loo': float(np.median(d_loo)) if len(d_loo) else np.nan,
        'centroid_sd_loo': float(np.std(d_loo)) if len(d_loo) else np.nan,
        'span_90': span90,
    })

    for pos, pid in enumerate(idx):
        rows.append({
            'proposal_uid': proposal_meta.loc[pid, 'proposal_uid'],
            'title': proposal_meta.loc[pid, 'title'],
            'group_model': proposal_meta.loc[pid, 'group_model'],
            'group_binary': proposal_meta.loc[pid, 'group_binary'],
            'centroid_dist_raw': float(d_raw[pos]),
            'centroid_dist_loo': float(d_loo[pos]),
        })

centroid_proposal_df = pd.DataFrame(rows)
span90_group_summary_df = pd.DataFrame(group_rows)

# Backward-compatible vars for existing plotting cells
human_centroid_dists = centroid_proposal_df.loc[centroid_proposal_df['group_binary']=='Human', 'centroid_dist_loo'].to_numpy()
ai_centroid_dists = centroid_proposal_df.loc[centroid_proposal_df['group_binary']=='AI', 'centroid_dist_loo'].to_numpy()
model_centroid_dists = {
    m: centroid_proposal_df.loc[centroid_proposal_df['group_model']==m, 'centroid_dist_loo'].to_numpy()
    for m in ai_models
}

# tests on centroid_dist_loo
centroid_tests = []
human_vals = centroid_proposal_df.loc[centroid_proposal_df['group_binary']=='Human', 'centroid_dist_loo'].to_numpy()
for g in ['All AI'] + ai_models:
    g_vals = centroid_proposal_df.loc[
        (centroid_proposal_df['group_binary']=='AI') if g=='All AI' else (centroid_proposal_df['group_model']==g),
        'centroid_dist_loo'
    ].to_numpy()
    if len(g_vals) == 0:
        continue
    r = run_group_comparison(g_vals, human_vals, n_permutations=10000, n_boot=5000, random_state=42)
    r.update({'comparison': f'{g} vs Human', 'group1': g, 'group2': 'Human', 'n_group1': len(g_vals), 'n_group2': len(human_vals)})
    centroid_tests.append(r)

centroid_tests_df = pd.DataFrame(centroid_tests)
if len(centroid_tests_df):
    centroid_tests_df = apply_multiple_testing(centroid_tests_df, p_cols=('p_value_mw', 'p_value_perm'), method='holm')

centroid_export_df = centroid_proposal_df.copy()
centroid_export_df['group'] = centroid_export_df['group_model']
centroid_export_df['centroid_dist'] = centroid_export_df['centroid_dist_loo']
centroid_export_df.to_csv(TABLES_DIR / 'centroid_distances.csv', index=False)
span90_group_summary_df.to_csv(TABLES_DIR / 'diversity_span90_group_summary.csv', index=False)
centroid_tests_df.to_csv(TABLES_DIR / 'diversity_centroid_pairwise_tests.csv', index=False)

print(span90_group_summary_df[['group', 'n', 'centroid_mean_loo', 'span_90']].to_string(index=False))
print('\nSaved centroid_distances.csv, diversity_span90_group_summary.csv, diversity_centroid_pairwise_tests.csv')


In [ ]:
print("="*85)
print("STATISTICAL TESTS: Centroid Dispersion (All Groups vs Human)")
print("="*85)
print("Primary estimand: mean difference in centroid distance (AI - Human).")

comparison_specs = [('All AI', ai_centroid_dists)] + [
    (model, model_centroid_dists[model])
    for model in ai_models
    if model in model_centroid_dists
]

centroid_results = []

for group_name, vals in comparison_specs:
    print()
    print("-"*85)
    print(f"Comparison: {group_name} vs Human")
    print("-"*85)

    res = run_group_comparison(vals, human_centroid_dists, n_permutations=10000, n_boot=5000, random_state=42)
    res['group'] = group_name
    centroid_results.append(res)

    print("Mann-Whitney U Test:")
    print(f"  U-statistic: {res['u_stat']:,.0f}, p-value: {res['p_value_mw']:.4e}")
    print("Cliff's Delta:")
    print(f"  δ = {res['delta']:.4f} ({res['delta_interp']} effect)")
    if res['delta'] > 0:
        print(f"  → {group_name} proposals are MORE dispersed from center")
    else:
        print(f"  → Human proposals are MORE dispersed from center")

    print("Primary mean difference (AI - Human):")
    print(f"  Observed difference: {res['obs_diff_mean']:.4f}")
    print(f"  Bootstrap 95% CI: [{res['ci_low']:.4f}, {res['ci_high']:.4f}]")
    print(f"  Permutation p-value: {res['p_value_perm']:.4f}")

centroid_df = pd.DataFrame(centroid_results)
centroid_df = apply_multiple_testing(centroid_df, p_cols=('p_value_mw', 'p_value_perm'), method='holm')
centroid_comparison_results = centroid_df.to_dict('records')

print()
print("="*85)
print("SUMMARY: Centroid Dispersion with Holm correction")
print("="*85)
print(f"{'Group':<30} {'δ':<10} {'Mean Diff':<12} {'95% CI':<25} {'MW p':<12} {'MW p(Holm)':<12} {'Perm p(Holm)'}")
print("-"*85)
for _, row in centroid_df.iterrows():
    ci_txt = f"[{row['ci_low']:.4f}, {row['ci_high']:.4f}]"
    print(
        f"{row['group']:<30} {row['delta']:<10.4f} {row['obs_diff_mean']:<12.4f} "
        f"{ci_txt:<25} {row['p_value_mw']:<12.4e} {row['p_value_mw_adj_holm']:<12.4e} {row['p_value_perm_adj_holm']:.4e}"
    )
print("="*85)
print("💡 Positive mean-diff/δ = AI more dispersed; Negative = Human more dispersed")


In [ ]:
# Visualization: Centroid dispersion — standardized boxplot + jittered points + mean 95% CI (left) + effect size panel (right)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Build long df from proposal-level rows, preserving proposal_uid for funding/top-rank styling ----
dist_df_c = centroid_proposal_df[['proposal_uid', 'title', 'group_model', 'group_binary', 'centroid_dist_loo']].copy()
dist_df_c = dist_df_c.rename(columns={'group_model': 'group', 'centroid_dist_loo': 'centroid_dist'})
all_ai_c = dist_df_c.loc[dist_df_c['group_binary'] == 'AI'].copy()
all_ai_c['group'] = 'All AI'
dist_df_c = pd.concat([dist_df_c, all_ai_c], ignore_index=True)

plot_specs_c = [('Human', human_centroid_dists)]
for m in ai_models:
    if m in model_centroid_dists and len(model_centroid_dists[m]) > 0:
        plot_specs_c.append((m, model_centroid_dists[m]))
plot_specs_c.append(('All AI', ai_centroid_dists))

group_order_c = [g for g, _ in plot_specs_c]
palette_c = {g: colors.get(g, '#808080') for g in group_order_c}

# ---- Effect-size df with bootstrap CI on Cliff's delta ----
eff_order_c = [g for g in group_order_c if g != "Human"]
eff_df_c = centroid_df.copy()
eff_df_c["group"] = pd.Categorical(eff_df_c["group"], categories=eff_order_c, ordered=True)
eff_df_c = eff_df_c.dropna(subset=["group"]).sort_values("group")

delta_ci_low_c, delta_ci_high_c = [], []
for g in eff_df_c["group"].astype(str).tolist():
    gvals = ai_centroid_dists if g == "All AI" else model_centroid_dists.get(g, np.array([]))
    if len(gvals) == 0:
        delta_ci_low_c.append(np.nan); delta_ci_high_c.append(np.nan)
        continue
    lo, hi = bootstrap_cliffs_delta_ci(gvals, human_centroid_dists, n_boot=2000, random_state=42)
    delta_ci_low_c.append(lo); delta_ci_high_c.append(hi)

eff_df_c["delta_ci_low"] = delta_ci_low_c
eff_df_c["delta_ci_high"] = delta_ci_high_c

# ---- Plot ----
sns.set_theme(style="whitegrid", context="talk")
fig = plt.figure(figsize=(16, 7), dpi=140)
gs = fig.add_gridspec(1, 2, width_ratios=[1.7, 1.0], wspace=0.18)
axL = fig.add_subplot(gs[0, 0])
axR = fig.add_subplot(gs[0, 1])

# Left: standardized boxplot + jittered proposal points + mean CI
styled_boxplot_with_points(
    axL,
    dist_df_c,
    x='group',
    y='centroid_dist',
    order=group_order_c,
    palette=palette_c,
    box_width=0.45,
    jitter=0.15,
    point_size=20,
    point_alpha=0.50,
    random_state=42,
    show_metadata_legend=True,
    metadata_legend_loc='best',
)
axL.set_title("Centroid dispersion (box + mean 95% CI)")
axL.set_xlabel("")
axL.set_ylabel("Distance to group centroid")
axL.tick_params(axis="x", rotation=35, labelsize=10)

# Right: Cliff's delta + bootstrap CI + Holm-adjusted p-values
axR.axvline(0, color="black", linewidth=1.0, alpha=0.8)
ypos = np.arange(len(eff_df_c))
axR.set_yticks(ypos)
axR.set_yticklabels(eff_df_c["group"].astype(str))
axR.invert_yaxis()

for i, row in enumerate(eff_df_c.itertuples(index=False)):
    g = str(row.group)
    c = palette_c.get(g, "#808080")
    axR.errorbar(
        row.delta, i,
        xerr=np.array([[row.delta - row.delta_ci_low], [row.delta_ci_high - row.delta]]),
        fmt="o", color=c, ecolor=c, elinewidth=1.2, capsize=3,
    )
    txt = (
        f"MW(Holm)={fmt_p(getattr(row, 'p_value_mw_adj_holm', np.nan))} | "
        f"Perm(Holm)={fmt_p(getattr(row, 'p_value_perm_adj_holm', np.nan))}"
    )
    axR.text(1.03, i, txt, va="center", ha="left", fontsize=10, clip_on=False)

axR.set_title("Effect size (AI − Human) + significance")
axR.set_xlabel("Cliff's delta (bootstrap 95% CI)")
axR.set_ylabel("")
axR.set_xlim(-1.05, 1.05)

plt.suptitle("Centroid dispersion + effect size + significance", y=1.02, fontsize=16)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'centroid_dispersion_by_model.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure saved to: {FIGURES_DIR / 'centroid_dispersion_by_model.png'}")



### 2.2b: Between-Group Centroid Dispersion

Measure **between-group dispersion** from a **single global centroid** built from all Human and AI proposals.

Groups included:
- **Human proposals**
- **Each AI model's proposals**
- **All AI proposals (combined)**

For each proposal, compute cosine distance to the global centroid. Then compare group-level dispersion distributions with inferential tests.


In [ ]:

print('='*85)
print('ANALYSIS 1.2b: BETWEEN-GROUP GLOBAL-CENTROID DISPERSION')
print('='*85)

global_centroid = X_prop.mean(axis=0, keepdims=True)
global_centroid_dist = cosine_distances(X_prop, global_centroid).ravel()

between_group_global_centroid_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
between_group_global_centroid_df['global_centroid_dist'] = global_centroid_dist

sum_rows = []
for g, idx in GROUPS.items():
    vals = global_centroid_dist[idx]
    sum_rows.append({
        'group': g,
        'n': len(vals),
        'mean': float(np.mean(vals)),
        'median': float(np.median(vals)),
        'sd': float(np.std(vals)),
        'var': float(np.var(vals)),
    })
between_group_global_centroid_summary_df = pd.DataFrame(sum_rows)

# pairwise tests all groups
pairwise = []
order = ['Human'] + ai_models + ['All AI']
for i in range(len(order)):
    for j in range(i+1, len(order)):
        g1, g2 = order[i], order[j]
        m1 = (between_group_global_centroid_df['group_binary']=='AI') if g1=='All AI' else (between_group_global_centroid_df['group_model']==g1)
        m2 = (between_group_global_centroid_df['group_binary']=='AI') if g2=='All AI' else (between_group_global_centroid_df['group_model']==g2)
        v1 = between_group_global_centroid_df.loc[m1, 'global_centroid_dist'].to_numpy()
        v2 = between_group_global_centroid_df.loc[m2, 'global_centroid_dist'].to_numpy()
        if len(v1)==0 or len(v2)==0:
            continue
        r = run_group_comparison(v1, v2, n_permutations=10000, n_boot=5000, random_state=42)
        r.update({'group_1': g1, 'group_2': g2, 'n_1': len(v1), 'n_2': len(v2), 'comparison': f'{g1} vs {g2}'})
        pairwise.append(r)

between_group_global_centroid_tests_df = pd.DataFrame(pairwise)
if len(between_group_global_centroid_tests_df):
    between_group_global_centroid_tests_df = apply_multiple_testing(
        between_group_global_centroid_tests_df,
        p_cols=('p_value_mw', 'p_value_perm'),
        method='holm'
    )

between_group_global_centroid_df.to_csv(TABLES_DIR / 'between_group_global_centroid_distances.csv', index=False)
between_group_global_centroid_summary_df.to_csv(TABLES_DIR / 'between_group_global_centroid_group_summary.csv', index=False)
between_group_global_centroid_tests_df.to_csv(TABLES_DIR / 'between_group_global_centroid_pairwise_tests.csv', index=False)

print('Saved between_group_global_centroid_* exports')


In [ ]:
# Visualization + test summary: Analysis 2.2b — standardized boxplot + jittered points + mean 95% CI (left) + effect size panel (right)
print('='*85)
print('VISUALIZATION: 2.2b BETWEEN-GROUP GLOBAL-CENTROID DISPERSION')
print('='*85)

if 'between_group_global_centroid_df' not in globals():
    raise RuntimeError('between_group_global_centroid_df missing. Run Analysis 2.2b cell first.')

plot_df = between_group_global_centroid_df[['proposal_uid', 'title', 'group_model', 'group_binary', 'global_centroid_dist']].copy()
plot_df['group_plot'] = plot_df['group_model']
all_ai_df = plot_df.loc[plot_df['group_binary'] == 'AI'].copy()
all_ai_df['group_plot'] = 'All AI'
plot_df = pd.concat([plot_df, all_ai_df], ignore_index=True)

group_order_bg = ['Human'] + [m for m in ai_models if m in set(plot_df['group_plot'])] + ['All AI']
palette_bg = {g: (colors.get(g, '#888888') if g != 'All AI' else colors.get('All AI', '#B56576')) for g in group_order_bg}

# ---- Effect-size: recompute delta (AI − Human direction) + bootstrap CI + pull p-values ----
eff_order_bg = [g for g in group_order_bg if g != 'Human']
human_bg_vals = plot_df.loc[plot_df['group_plot'] == 'Human', 'global_centroid_dist'].to_numpy(dtype=float)

eff_rows_bg = []
for g in eff_order_bg:
    gvals = plot_df.loc[plot_df['group_plot'] == g, 'global_centroid_dist'].to_numpy(dtype=float)
    if len(gvals) == 0:
        continue
    delta_val = cliffs_delta(gvals, human_bg_vals)
    lo, hi = bootstrap_cliffs_delta_ci(gvals, human_bg_vals, n_boot=2000, random_state=42)
    mw_holm, perm_holm = np.nan, np.nan
    if 'between_group_global_centroid_tests_df' in globals() and len(between_group_global_centroid_tests_df):
        mask_hg = (
            ((between_group_global_centroid_tests_df['group_1'] == 'Human') & (between_group_global_centroid_tests_df['group_2'] == g)) |
            ((between_group_global_centroid_tests_df['group_1'] == g) & (between_group_global_centroid_tests_df['group_2'] == 'Human'))
        )
        sub = between_group_global_centroid_tests_df.loc[mask_hg]
        if len(sub):
            mw_holm = sub.iloc[0].get('p_value_mw_adj_holm', np.nan)
            perm_holm = sub.iloc[0].get('p_value_perm_adj_holm', np.nan)
    eff_rows_bg.append({
        'group': g, 'delta': delta_val,
        'delta_ci_low': lo, 'delta_ci_high': hi,
        'p_value_mw_adj_holm': mw_holm, 'p_value_perm_adj_holm': perm_holm,
    })

eff_df_bg = pd.DataFrame(eff_rows_bg)

# ---- Plot ----
sns.set_theme(style="whitegrid", context="talk")
fig = plt.figure(figsize=(16, 7), dpi=140)
gs = fig.add_gridspec(1, 2, width_ratios=[1.7, 1.0], wspace=0.18)
axL = fig.add_subplot(gs[0, 0])
axR = fig.add_subplot(gs[0, 1])

# Left: standardized boxplot + jittered proposal points + mean CI
styled_boxplot_with_points(
    axL,
    plot_df,
    x='group_plot',
    y='global_centroid_dist',
    order=group_order_bg,
    palette=palette_bg,
    box_width=0.45,
    jitter=0.15,
    point_size=20,
    point_alpha=0.50,
    random_state=42,
    show_metadata_legend=True,
    metadata_legend_loc='best',
)
axL.set_title('Global-centroid dispersion (box + mean 95% CI)')
axL.set_xlabel('')
axL.set_ylabel('Cosine distance to global centroid')
axL.tick_params(axis='x', rotation=35, labelsize=10)

# Right: Cliff's delta + bootstrap CI + Holm-adjusted p-values
axR.axvline(0, color='black', linewidth=1.0, alpha=0.8)
ypos = np.arange(len(eff_df_bg))
axR.set_yticks(ypos)
axR.set_yticklabels(eff_df_bg['group'].astype(str))
axR.invert_yaxis()

for i, row in enumerate(eff_df_bg.itertuples(index=False)):
    g = str(row.group)
    c = palette_bg.get(g, '#808080')
    axR.errorbar(
        row.delta, i,
        xerr=np.array([[row.delta - row.delta_ci_low], [row.delta_ci_high - row.delta]]),
        fmt='o', color=c, ecolor=c, elinewidth=1.2, capsize=3,
    )
    txt = (
        f"MW(Holm)={fmt_p(getattr(row, 'p_value_mw_adj_holm', np.nan))} | "
        f"Perm(Holm)={fmt_p(getattr(row, 'p_value_perm_adj_holm', np.nan))}"
    )
    axR.text(1.03, i, txt, va='center', ha='left', fontsize=10, clip_on=False)

axR.set_title('Effect size (AI − Human) + significance')
axR.set_xlabel("Cliff's delta (bootstrap 95% CI)")
axR.set_ylabel('')
axR.set_xlim(-1.05, 1.05)

plt.suptitle('Global-centroid dispersion + effect size + significance', y=1.02, fontsize=16)
plt.tight_layout()
fig_path = FIGURES_DIR / 'between_group_global_centroid_dispersion.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved figure: {fig_path}')



## Analysis 2.2c: MST Dispersion

In [ ]:

print('='*85)
print('ANALYSIS 1.2c: MST DISPERSION')
print('='*85)

rows = []
for g, gc in group_cache.items():
    Dg = gc['D']
    mst_val = group_mst_dispersion(Dg)
    boot = bootstrap_group_metric(Dg, group_mst_dispersion, n_boot=5000, random_state=42, metric_type='distance')
    rows.append({
        'group': g,
        'n': gc['n'],
        'mst_dispersion': mst_val,
        'mst_boot_mean': boot['boot_mean'],
        'mst_boot_ci_low': boot['ci_low'],
        'mst_boot_ci_high': boot['ci_high'],
    })

mst_group_summary_df = pd.DataFrame(rows)

perm_rows = []
for g in ai_models + ['All AI']:
    if g not in GROUPS:
        continue
    pt = permutation_test_group_metric(D_pp, proposal_meta['group_model'].to_numpy(), GROUPS[g], GROUPS['Human'], group_mst_dispersion, n_perm=10000, random_state=42, use_distance_submat=True)
    perm_rows.append({'comparison': f'{g} vs Human', 'group1': g, 'group2': 'Human', **pt})

mst_pairwise_perm_df = pd.DataFrame(perm_rows)
if len(mst_pairwise_perm_df):
    mst_pairwise_perm_df = apply_multiple_testing(mst_pairwise_perm_df, p_cols=('perm_p_value',), method='holm')

mst_group_summary_df.to_csv(TABLES_DIR / 'diversity_mst_group_summary.csv', index=False)
mst_pairwise_perm_df.to_csv(TABLES_DIR / 'diversity_mst_pairwise_permutation.csv', index=False)
print('Saved diversity_mst_group_summary.csv and diversity_mst_pairwise_permutation.csv')


In [ ]:
# Visualization + test summary: Analysis 2.2c — bar + bootstrap CI (left) + effect size panel (right)
print('='*85)
print('VISUALIZATION: 2.2c MST DISPERSION')
print('='*85)

if 'mst_group_summary_df' not in globals():
    raise RuntimeError('mst_group_summary_df missing. Run Analysis 2.2c cell first.')

order = ['Human'] + [m for m in ai_models if m in set(mst_group_summary_df['group'])] + ['All AI']
plot_df = mst_group_summary_df.set_index('group').reindex(order).reset_index()
palette_mst = {g: (colors.get(g, '#888888') if g != 'All AI' else colors.get('All AI', '#B56576')) for g in order}
colors_plot = [palette_mst[g] for g in plot_df['group'].tolist()]

# ---- Effect-size df: obs_diff (AI − Human) with propagated CI from per-group bootstrap ----
eff_order_mst = [g for g in order if g != 'Human']
human_mst = float(plot_df.loc[plot_df['group'] == 'Human', 'mst_dispersion'].iloc[0])
human_lo  = float(plot_df.loc[plot_df['group'] == 'Human', 'mst_boot_ci_low'].iloc[0])
human_hi  = float(plot_df.loc[plot_df['group'] == 'Human', 'mst_boot_ci_high'].iloc[0])
sigma_h   = (human_hi - human_lo) / 2

eff_rows_mst = []
for g in eff_order_mst:
    sub = plot_df.loc[plot_df['group'] == g]
    if len(sub) == 0:
        continue
    g_mst = float(sub['mst_dispersion'].iloc[0])
    g_lo  = float(sub['mst_boot_ci_low'].iloc[0])
    g_hi  = float(sub['mst_boot_ci_high'].iloc[0])
    obs_diff  = g_mst - human_mst
    sigma_g   = (g_hi - g_lo) / 2
    sigma_diff = np.sqrt(sigma_g**2 + sigma_h**2)
    perm_holm = np.nan
    if 'mst_pairwise_perm_df' in globals() and len(mst_pairwise_perm_df):
        q_col = 'perm_p_value_adj_holm' if 'perm_p_value_adj_holm' in mst_pairwise_perm_df.columns else 'perm_p_value'
        r = mst_pairwise_perm_df.loc[mst_pairwise_perm_df['group1'] == g]
        if len(r):
            perm_holm = float(r.iloc[0][q_col])
    eff_rows_mst.append({
        'group': g, 'obs_diff': obs_diff,
        'diff_ci_low': obs_diff - sigma_diff,
        'diff_ci_high': obs_diff + sigma_diff,
        'perm_holm': perm_holm,
    })

eff_df_mst = pd.DataFrame(eff_rows_mst)

# ---- Plot ----
sns.set_theme(style='whitegrid', context='talk')
fig = plt.figure(figsize=(16, 7), dpi=140)
gs = fig.add_gridspec(1, 2, width_ratios=[1.7, 1.0], wspace=0.18)
axL = fig.add_subplot(gs[0, 0])
axR = fig.add_subplot(gs[0, 1])

# Left: bar chart with bootstrap CI
y    = plot_df['mst_dispersion'].to_numpy(dtype=float)
lo   = plot_df['mst_boot_ci_low'].to_numpy(dtype=float)
hi   = plot_df['mst_boot_ci_high'].to_numpy(dtype=float)
yerr = np.vstack([y - lo, hi - y])

axL.bar(
    np.arange(len(plot_df)), y,
    yerr=yerr, capsize=6,
    color=colors_plot, edgecolor='black', alpha=0.85,
)
axL.set_xticks(np.arange(len(plot_df)))
axL.set_xticklabels(plot_df['group'].tolist(), rotation=20, fontsize=10)
axL.set_ylabel('MST Dispersion')
axL.set_title('MST Dispersion by Group (95% bootstrap CI)')
axL.grid(alpha=0.3, axis='y')

# Right: obs_diff (AI − Human) with propagated CI + perm Holm p-value
axR.axvline(0, color='black', linewidth=1.0, alpha=0.8)
ypos = np.arange(len(eff_df_mst))
axR.set_yticks(ypos)
axR.set_yticklabels(eff_df_mst['group'].astype(str))
axR.invert_yaxis()

x_vals   = eff_df_mst['obs_diff'].to_numpy(dtype=float)
ci_spans = np.concatenate([
    np.abs(eff_df_mst['diff_ci_low'].to_numpy(dtype=float)),
    np.abs(eff_df_mst['diff_ci_high'].to_numpy(dtype=float)),
])
x_abs_max = float(np.nanmax(np.abs(ci_spans))) * 1.2 if len(ci_spans) else 0.05

for i, row in enumerate(eff_df_mst.itertuples(index=False)):
    g = str(row.group)
    c = palette_mst.get(g, '#808080')
    xerr = np.array([[row.obs_diff - row.diff_ci_low], [row.diff_ci_high - row.obs_diff]])
    axR.errorbar(
        row.obs_diff, i, xerr=xerr,
        fmt='o', color=c, ecolor=c, elinewidth=1.2, capsize=3,
    )
    txt = f"Perm(Holm)={fmt_p(row.perm_holm)}"
    axR.text(x_abs_max * 1.08, i, txt, va='center', ha='left', fontsize=10, clip_on=False)

axR.set_title('Effect size (AI − Human) + significance')
axR.set_xlabel('MST difference (AI − Human, propagated 95% CI)')
axR.set_ylabel('')
axR.set_xlim(-x_abs_max, x_abs_max)

plt.suptitle('MST dispersion + effect size + significance', y=1.02, fontsize=16)
plt.tight_layout()
fig_path = FIGURES_DIR / 'diversity_mst_dispersion.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved figure: {fig_path}')

if 'mst_pairwise_perm_df' in globals() and len(mst_pairwise_perm_df):
    disp_cols = ['comparison', 'obs_diff', 'perm_p_value']
    if 'perm_p_value_adj_holm' in mst_pairwise_perm_df.columns:
        disp_cols.append('perm_p_value_adj_holm')
    print('\nPermutation tests (MST, Human contrasts):')
    print(mst_pairwise_perm_df[disp_cols].sort_values(disp_cols[-1]).to_string(index=False))


## Analysis 2.2d: Sparseness (Medoid-Based Dispersion)

In [ ]:

print('='*85)
print('ANALYSIS 1.2d: SPARSENESS (MEDOID-BASED)')
print('='*85)

rows = []
sum_rows = []
for g, gc in group_cache.items():
    idx = gc['idx']
    Dg = gc['D']
    md = proposal_medoid_distances(Dg)
    sp = group_sparseness(Dg)
    for pos, pid in enumerate(idx):
        rows.append({
            'proposal_uid': proposal_meta.loc[pid, 'proposal_uid'],
            'title': proposal_meta.loc[pid, 'title'],
            'group_model': proposal_meta.loc[pid, 'group_model'],
            'group_binary': proposal_meta.loc[pid, 'group_binary'],
            'medoid_dist': float(md[pos]),
        })
    sum_rows.append({'group': g, 'n': gc['n'], 'sparseness': sp})

medoid_proposal_df = pd.DataFrame(rows)
sparseness_group_summary_df = pd.DataFrame(sum_rows)

tests = []
human_vals = medoid_proposal_df.loc[medoid_proposal_df['group_binary']=='Human', 'medoid_dist'].to_numpy()
for g in ['All AI'] + ai_models:
    vals = medoid_proposal_df.loc[(medoid_proposal_df['group_binary']=='AI') if g=='All AI' else (medoid_proposal_df['group_model']==g), 'medoid_dist'].to_numpy()
    if len(vals)==0:
        continue
    r = run_group_comparison(vals, human_vals, n_permutations=10000, n_boot=5000, random_state=42)
    r.update({'comparison': f'{g} vs Human', 'group1': g, 'group2': 'Human', 'n_group1': len(vals), 'n_group2': len(human_vals)})
    tests.append(r)

sparseness_tests_df = pd.DataFrame(tests)
if len(sparseness_tests_df):
    sparseness_tests_df = apply_multiple_testing(sparseness_tests_df, p_cols=('p_value_mw', 'p_value_perm'), method='holm')

medoid_proposal_df.to_csv(TABLES_DIR / 'diversity_medoid_distances.csv', index=False)
sparseness_group_summary_df.to_csv(TABLES_DIR / 'diversity_sparseness_group_summary.csv', index=False)
sparseness_tests_df.to_csv(TABLES_DIR / 'diversity_sparseness_pairwise_tests.csv', index=False)
print('Saved diversity_medoid_distances.csv, diversity_sparseness_group_summary.csv, diversity_sparseness_pairwise_tests.csv')


In [ ]:
# Visualization + test summary: Analysis 2.2d — mean + bootstrap CI (left) + effect size panel (right)
print('='*85)
print('VISUALIZATION: 2.2d SPARSENESS (MEDOID-BASED)')
print('='*85)

if 'medoid_proposal_df' not in globals():
    raise RuntimeError('medoid_proposal_df missing. Run Analysis 2.2d cell first.')

plot_df = medoid_proposal_df[['group_model', 'group_binary', 'medoid_dist']].copy()
plot_df['group_plot'] = plot_df['group_model']
all_ai_df = plot_df.loc[plot_df['group_binary'] == 'AI'].copy()
all_ai_df['group_plot'] = 'All AI'
plot_df = pd.concat([plot_df, all_ai_df], ignore_index=True)

group_order_sp = ['Human'] + [m for m in ai_models if m in set(plot_df['group_plot'])] + ['All AI']
palette_sp = {g: (colors.get(g, '#888888') if g != 'All AI' else colors.get('All AI', '#B56576')) for g in group_order_sp}
colors_sp  = [palette_sp[g] for g in group_order_sp]

# ---- Bootstrap CI for group means (left panel) ----
group_means, ci_lo_arr, ci_hi_arr = [], [], []
for g in group_order_sp:
    vals = plot_df.loc[plot_df['group_plot'] == g, 'medoid_dist'].to_numpy(dtype=float)
    mean, lo, hi = bootstrap_mean_ci(vals, n_boot=2000, random_state=42)
    group_means.append(mean); ci_lo_arr.append(lo); ci_hi_arr.append(hi)

group_means = np.array(group_means)
yerr = np.vstack([group_means - np.array(ci_lo_arr), np.array(ci_hi_arr) - group_means])

# ---- Effect-size df: Cliff's delta (AI − Human) + bootstrap CI + p-values ----
eff_order_sp = [g for g in group_order_sp if g != 'Human']
human_sp_vals = plot_df.loc[plot_df['group_plot'] == 'Human', 'medoid_dist'].to_numpy(dtype=float)

# Pull p-values from pre-computed sparseness_tests_df; recompute delta in AI-Human direction
eff_rows_sp = []
for g in eff_order_sp:
    gvals = plot_df.loc[plot_df['group_plot'] == g, 'medoid_dist'].to_numpy(dtype=float)
    if len(gvals) == 0:
        continue
    delta_val = cliffs_delta(gvals, human_sp_vals)
    lo_d, hi_d = bootstrap_cliffs_delta_ci(gvals, human_sp_vals, n_boot=2000, random_state=42)
    mw_holm, perm_holm = np.nan, np.nan
    if 'sparseness_tests_df' in globals() and len(sparseness_tests_df):
        # sparseness_tests_df uses group1 = AI group; group2 = Human (from cell 68 comparison_specs)
        mask = sparseness_tests_df['group1'] == g if 'group1' in sparseness_tests_df.columns else pd.Series(False, index=sparseness_tests_df.index)
        # fallback: match via comparison string
        if not mask.any() and 'comparison' in sparseness_tests_df.columns:
            mask = sparseness_tests_df['comparison'].str.startswith(g + ' vs')
        r = sparseness_tests_df.loc[mask]
        if len(r):
            mw_holm   = r.iloc[0].get('p_value_mw_adj_holm',   np.nan)
            perm_holm = r.iloc[0].get('p_value_perm_adj_holm', np.nan)
    eff_rows_sp.append({
        'group': g, 'delta': delta_val,
        'delta_ci_low': lo_d, 'delta_ci_high': hi_d,
        'p_value_mw_adj_holm': mw_holm, 'p_value_perm_adj_holm': perm_holm,
    })

eff_df_sp = pd.DataFrame(eff_rows_sp)

# ---- Plot ----
sns.set_theme(style='whitegrid', context='talk')
fig = plt.figure(figsize=(16, 7), dpi=140)
gs = fig.add_gridspec(1, 2, width_ratios=[1.7, 1.0], wspace=0.18)
axL = fig.add_subplot(gs[0, 0])
axR = fig.add_subplot(gs[0, 1])

# Left: group mean ± bootstrap 95% CI
axL.bar(
    np.arange(len(group_order_sp)), group_means,
    yerr=yerr, capsize=6,
    color=colors_sp, edgecolor='black', alpha=0.85,
)
axL.set_xticks(np.arange(len(group_order_sp)))
axL.set_xticklabels(group_order_sp, rotation=20, fontsize=10)
axL.set_ylabel('Mean Distance to Group Medoid')
axL.set_title('Medoid Sparseness — Group Mean (95% bootstrap CI)')
axL.grid(alpha=0.3, axis='y')

# Right: Cliff's delta + bootstrap CI + Holm-adjusted p-values
axR.axvline(0, color='black', linewidth=1.0, alpha=0.8)
ypos = np.arange(len(eff_df_sp))
axR.set_yticks(ypos)
axR.set_yticklabels(eff_df_sp['group'].astype(str))
axR.invert_yaxis()

for i, row in enumerate(eff_df_sp.itertuples(index=False)):
    g = str(row.group)
    c = palette_sp.get(g, '#808080')
    axR.errorbar(
        row.delta, i,
        xerr=np.array([[row.delta - row.delta_ci_low], [row.delta_ci_high - row.delta]]),
        fmt='o', color=c, ecolor=c, elinewidth=1.2, capsize=3,
    )
    txt = (
        f"MW(Holm)={fmt_p(getattr(row, 'p_value_mw_adj_holm', np.nan))} | "
        f"Perm(Holm)={fmt_p(getattr(row, 'p_value_perm_adj_holm', np.nan))}"
    )
    axR.text(1.03, i, txt, va='center', ha='left', fontsize=10, clip_on=False)

axR.set_title('Effect size (AI − Human) + significance')
axR.set_xlabel("Cliff's delta (bootstrap 95% CI)")
axR.set_ylabel('')
axR.set_xlim(-1.05, 1.05)

plt.suptitle('Medoid sparseness + effect size + significance', y=1.02, fontsize=16)
plt.tight_layout()
fig_path = FIGURES_DIR / 'diversity_sparseness_medoid.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved figure: {fig_path}')

if 'sparseness_tests_df' in globals() and len(sparseness_tests_df):
    cols = ['comparison', 'obs_diff_mean', 'delta',
            'p_value_mw', 'p_value_mw_adj_holm', 'p_value_perm', 'p_value_perm_adj_holm']
    for c in cols:
        if c not in sparseness_tests_df.columns:
            sparseness_tests_df[c] = np.nan
    print('\nPairwise tests vs Human (Holm-adjusted):')
    print(sparseness_tests_df[cols].sort_values('p_value_mw_adj_holm').to_string(index=False))


## Analysis 2.3: Nearest-Neighbor Isolation and Outlier Detection (Chamfer / NN)

Compute global 1-NN isolation, pooled outlier flags, and exact within-group Chamfer summaries.

In [ ]:

print('='*85)
print('ANALYSIS 1.3: GLOBAL NN ISOLATION + CHAMFER')
print('='*85)

n_human = int((proposal_meta['group_binary'] == 'Human').sum())
n_ai = int((proposal_meta['group_binary'] == 'AI').sum())
labels = proposal_meta['group_model'].to_numpy()
ai_models_local = sorted(proposal_meta.loc[proposal_meta['group_binary']=='AI', 'group_model'].unique().tolist())

all_distances = D_pp_infdiag.copy()
nn_idx = np.argmin(all_distances, axis=1)
nn_dist = np.min(all_distances, axis=1)

nn_distances = nn_dist
nn_dist_global = nn_dist
nn_indices = nn_idx
nn_labels = labels[nn_idx]

outliers, nn_outlier_threshold = flag_top_percentile(nn_distances, pct=90, strict=True)
threshold = nn_outlier_threshold  # backward compatibility

# global mean-5NN for unified output/backward compatibility (no separate section)
k5 = min(5, D_pp.shape[0]-1)
mean_knn_dist = np.partition(D_pp_infdiag, kth=k5-1, axis=1)[:, :k5].mean(axis=1)
outliers_mean_knn, mean5nn_threshold = flag_top_percentile(mean_knn_dist, pct=90, strict=True)

human_nn_dists = nn_distances[proposal_meta['group_binary'].to_numpy() == 'Human']
ai_nn_dists = nn_distances[proposal_meta['group_binary'].to_numpy() == 'AI']
model_nn_dists = {m: nn_distances[labels == m] for m in ai_models_local}

# proposal-level table
nn_proposal_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
nn_proposal_df['nn_dist_global'] = nn_distances
nn_proposal_df['nn_neighbor_proposal_uid'] = proposal_meta.loc[nn_indices, 'proposal_uid'].to_numpy()
nn_proposal_df['nn_neighbor_group_model'] = proposal_meta.loc[nn_indices, 'group_model'].to_numpy()
nn_proposal_df['is_nn_outlier'] = outliers.astype(bool)

# chamfer summary
chamfer_rows = []
for g, gc in group_cache.items():
    chamfer_rows.append({'group': g, 'n': gc['n'], 'chamfer': group_chamfer(gc['D'])})
chamfer_group_summary_df = pd.DataFrame(chamfer_rows)

# tests on proposal-level nn_dist_global
tests = []
human_vals = nn_proposal_df.loc[nn_proposal_df['group_binary']=='Human', 'nn_dist_global'].to_numpy()
for g in ['All AI'] + ai_models_local:
    vals = nn_proposal_df.loc[(nn_proposal_df['group_binary']=='AI') if g=='All AI' else (nn_proposal_df['group_model']==g), 'nn_dist_global'].to_numpy()
    if len(vals)==0:
        continue
    r = run_group_comparison(vals, human_vals, n_permutations=10000, n_boot=5000, random_state=42)
    r.update({'comparison': f'{g} vs Human', 'group1': g, 'group2': 'Human', 'n_group1': len(vals), 'n_group2': len(human_vals)})
    tests.append(r)

nn_tests_df = pd.DataFrame(tests)
if len(nn_tests_df):
    nn_tests_df = apply_multiple_testing(nn_tests_df, p_cols=('p_value_mw', 'p_value_perm'), method='holm')

# source composition
rows = []
for g in ['Human'] + ai_models_local + ['All AI']:
    mask = (nn_proposal_df['group_binary']=='AI').to_numpy() if g=='All AI' else (labels==g)
    if mask.sum()==0:
        continue
    neigh = nn_proposal_df.loc[mask, 'nn_neighbor_group_model'].to_numpy()
    rows.append({
        'group': g,
        'n': int(mask.sum()),
        'nn_from_human': int(np.sum(neigh == 'Human')),
        'nn_from_same_group': int(np.sum(neigh == g)) if g != 'All AI' else int(np.sum(neigh != 'Human')),
        'nn_from_other_ai': int(np.sum((neigh != 'Human') & (neigh != g))) if g not in ['Human','All AI'] else np.nan,
    })
nn_source_composition_df = pd.DataFrame(rows)

# legacy export with threshold column
nn_export_df = nn_proposal_df.rename(columns={'group_model': 'group', 'nn_dist_global': 'nn_dist', 'is_nn_outlier': 'is_outlier'})
nn_export_df['threshold'] = nn_outlier_threshold
nn_export_df.to_csv(TABLES_DIR / 'nn_distances.csv', index=False)

# mean-5NN export
mean5_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
mean5_df['mean_5nn_dist_global'] = mean_knn_dist
mean5_df['is_mean5nn_outlier'] = outliers_mean_knn.astype(bool)
mean5_df['threshold_mean5nn'] = mean5nn_threshold
mean5_df.to_csv(TABLES_DIR / 'mean_knn_distances_k5.csv', index=False)

chamfer_group_summary_df.to_csv(TABLES_DIR / 'diversity_chamfer_group_summary.csv', index=False)
nn_source_composition_df.to_csv(TABLES_DIR / 'nearest_neighbor_source_composition.csv', index=False)
nn_tests_df.to_csv(TABLES_DIR / 'diversity_nn_pairwise_tests.csv', index=False)

print('Saved nn_distances.csv, mean_knn_distances_k5.csv, diversity_chamfer_group_summary.csv, nearest_neighbor_source_composition.csv, diversity_nn_pairwise_tests.csv')

# Within-cluster NN isolation (cluster-conditioned outlier detection)
# Corrects for the cross-cluster gap: proposals in the remote cluster are far from
# the main cluster but not necessarily isolated among their same-subfield peers.
if 'ward_labels' in globals() and len(ward_labels) == len(nn_distances):
    _wl = np.asarray(ward_labels)
    nn_dist_within = np.full(len(nn_distances), np.nan)
    nn_idx_within = np.full(len(nn_distances), -1, dtype=int)

    for _cl in np.unique(_wl):
        _cl_idx = np.where(_wl == _cl)[0]
        if len(_cl_idx) < 2:
            continue
        _D_sub = D_pp_infdiag[np.ix_(_cl_idx, _cl_idx)]
        _nn_sub_pos = np.argmin(_D_sub, axis=1)
        nn_dist_within[_cl_idx] = _D_sub[np.arange(len(_cl_idx)), _nn_sub_pos]
        nn_idx_within[_cl_idx] = _cl_idx[_nn_sub_pos]

    # flag outliers per cluster at 90th percentile within that cluster
    outliers_within_cluster = np.zeros(len(nn_distances), dtype=bool)
    nn_outlier_threshold_within = {}
    for _cl in np.unique(_wl):
        _cl_idx = np.where(_wl == _cl)[0]
        _dists = nn_dist_within[_cl_idx]
        _valid = _dists[np.isfinite(_dists)]
        if len(_valid) > 1:
            _thresh = np.percentile(_valid, 90)
            nn_outlier_threshold_within[int(_cl)] = _thresh
            outliers_within_cluster[_cl_idx] = np.isfinite(_dists) & (_dists > _thresh)

    nn_proposal_df['nn_dist_within_cluster'] = nn_dist_within
    nn_proposal_df['is_nn_outlier_within_cluster'] = outliers_within_cluster

    # Re-export nn_distances.csv with within-cluster columns
    nn_export_df2 = nn_proposal_df.rename(columns={
        'group_model': 'group', 'nn_dist_global': 'nn_dist', 'is_nn_outlier': 'is_outlier'
    })
    nn_export_df2['threshold'] = nn_outlier_threshold
    nn_export_df2.to_csv(TABLES_DIR / 'nn_distances.csv', index=False)

    print(f'\nWithin-cluster NN outliers (global threshold inflated by cross-cluster gap):')
    print(f'  Global outliers: {int(outliers.sum())}  vs  Within-cluster outliers: {int(outliers_within_cluster.sum())}')
    print(f'\nPer-group within-cluster outlier counts:')
    for _g in ['Human'] + ai_models_local:
        _mask = (labels == _g)
        _n_out_global = int(outliers[_mask].sum())
        _n_out_within = int(outliers_within_cluster[_mask].sum())
        _n = int(_mask.sum())
        print(f'  {_g:<12}: global={_n_out_global}/{_n}  within-cluster={_n_out_within}/{_n}')
else:
    print('⚠️  ward_labels not available (Analysis 1.3 not run). Skipping within-cluster NN.')
    outliers_within_cluster = outliers  # fall back to global
    nn_proposal_df['nn_dist_within_cluster'] = np.nan
    nn_proposal_df['is_nn_outlier_within_cluster'] = outliers



In [ ]:
import numpy as np
import pandas as pd

print("\n" + "="*85)
print("UNADJUSTED NN OUTLIERS: TITLES + AUTHOR")
print("="*85)

required = ['outliers', 'threshold', 'n_human']
missing = [v for v in required if v not in globals()]
if missing:
    print(f"⚠️ Missing variables: {missing}")
    print("Run the unadjusted NN + outlier detection cells first.")
else:
    outlier_indices = np.where(outliers)[0]
    n_total = int(len(outliers))
    n_ai = int(n_total - n_human)

    # NN distances (unadjusted)
    nn_d = np.asarray(nn_distances) if 'nn_distances' in globals() else None

    # Prefer metadata saved alongside embeddings (aligned with embedding order)
    use_meta = ('ai_metadata' in globals()) and ('human_metadata' in globals())
    if use_meta:
        if len(human_metadata) != n_human or len(ai_metadata) != n_ai:
            print("⚠️ Metadata lengths do not match embedding counts; falling back to df lookup.")
            use_meta = False

    rows = []
    for idx in outlier_indices:
        if idx < n_human:
            who = 'Human'
            model = 'Human'
            if use_meta:
                rec = human_metadata[idx]
                title = rec.get('proposal_title', rec.get('title', ''))
            else:
                title = human_df.iloc[idx].get('proposal_title', human_df.iloc[idx].get('title', '')) if 'human_df' in globals() else ''
        else:
            ai_idx = int(idx - n_human)
            who = 'AI'
            if use_meta:
                rec = ai_metadata[ai_idx]
                model = str(rec.get('model', 'AI'))
                title = str(rec.get('title', ''))
            else:
                model = str(ai_df.iloc[ai_idx].get('model', 'AI')) if 'ai_df' in globals() else 'AI'
                title = str(ai_df.iloc[ai_idx].get('title', '')) if 'ai_df' in globals() else ''

        rows.append({
            'global_index': int(idx),
            'who': who,
            'model': model,
            'nn_distance': float(nn_d[idx]) if nn_d is not None else np.nan,
            'title': title
        })

    out_df = pd.DataFrame(rows).sort_values('nn_distance', ascending=False)

    print(f"Outlier threshold (90th percentile): {threshold:.4f}")
    print(f"Total outliers: {len(out_df)} / {n_total} ({len(out_df)/n_total*100:.1f}%)")
    print("Outliers by source (model):")
    print(out_df['model'].value_counts().to_string())

    with pd.option_context('display.max_colwidth', 140):
        display(out_df.reset_index(drop=True))


In [ ]:
# Visualization: Nearest-neighbor analysis by group (self-contained)
# This cell rebuilds required summary variables from nn_proposal_df/proposal_meta
# so it does not depend on legacy intermediate variable names.

if 'nn_proposal_df' not in globals():
    raise RuntimeError('nn_proposal_df is missing. Run Analysis 2.3 NN cell first.')
if 'proposal_meta' not in globals():
    raise RuntimeError('proposal_meta is missing. Run shared precompute cells first.')

# Canonical group ordering for plotting
ai_models_plot = sorted(proposal_meta.loc[proposal_meta['group_binary']=='AI', 'group_model'].dropna().unique().tolist())
group_order = ['Human'] + [m for m in ai_models_plot if m in set(nn_proposal_df['group_model'])]

# Robust threshold fallback
if 'nn_outlier_threshold' in globals():
    threshold_plot = float(nn_outlier_threshold)
elif 'threshold' in globals():
    threshold_plot = float(threshold)
else:
    threshold_plot = float(np.percentile(nn_proposal_df['nn_dist_global'].to_numpy(), 90))

# Recompute all variables this visualization needs
nn_work = nn_proposal_df.copy()
nn_work['is_nn_outlier'] = nn_work['is_nn_outlier'].astype(bool)

n_human = int((proposal_meta['group_binary'] == 'Human').sum())
human_outliers = int(nn_work.loc[nn_work['group_binary']=='Human', 'is_nn_outlier'].sum())

model_outliers = {
    m: int(nn_work.loc[nn_work['group_model']==m, 'is_nn_outlier'].sum())
    for m in ai_models_plot
}

human_nn_same_group = int((nn_work.loc[nn_work['group_binary']=='Human', 'nn_neighbor_group_model'] == 'Human').sum())
human_nn_diff_group = int((nn_work['group_binary']=='Human').sum() - human_nn_same_group)

model_nn_analysis = {}
for model in ai_models_plot:
    dfm = nn_work.loc[nn_work['group_model'] == model]
    total = int(len(dfm))
    if total == 0:
        continue
    from_human = int((dfm['nn_neighbor_group_model'] == 'Human').sum())
    from_same_model = int((dfm['nn_neighbor_group_model'] == model).sum())
    from_other_ai = int(total - from_human - from_same_model)
    model_nn_analysis[model] = {
        'from_human': from_human,
        'from_same_model': from_same_model,
        'from_other_ai': from_other_ai,
        'total': total,
    }

# NN distributions used in panel 1
human_nn_dists = nn_work.loc[nn_work['group_binary']=='Human', 'nn_dist_global'].to_numpy()
model_nn_dists = {
    m: nn_work.loc[nn_work['group_model']==m, 'nn_dist_global'].to_numpy()
    for m in ai_models_plot
}

fig = plt.figure(figsize=(18, 5))
gs = fig.add_gridspec(1, 3, hspace=0.3, wspace=0.3)

# 1. NN distance distributions by group
ax1 = fig.add_subplot(gs[0, 0])
nn_df = nn_work[['proposal_uid', 'title', 'group_model', 'group_binary', 'nn_dist_global']].copy()
nn_df = nn_df.rename(columns={'group_model': 'Group', 'nn_dist_global': 'NN Distance'})
palette_nn = {g: colors.get(g, 'gray') for g in group_order}
styled_boxplot_with_points(
    ax1,
    nn_df,
    x='Group',
    y='NN Distance',
    order=group_order,
    palette=palette_nn,
    box_width=0.45,
    jitter=0.15,
    point_size=20,
    point_alpha=0.50,
    random_state=42,
    show_metadata_legend=True,
    metadata_legend_loc='best',
)
threshold_line = ax1.axhline(threshold_plot, color='red', linestyle='--', linewidth=2, alpha=0.7,
                             label=f'Outlier threshold (90%): {threshold_plot:.3f}')
metadata_legend = ax1.get_legend()
if metadata_legend is not None:
    ax1.add_artist(metadata_legend)
ax1.legend(handles=[threshold_line], fontsize=8, loc='upper right', framealpha=0.88)
ax1.set_ylabel('Nearest-Neighbor Distance', fontsize=11)
ax1.set_xlabel('Group', fontsize=11)
ax1.set_title('NN Distance Distributions', fontsize=12, fontweight='bold')
ax1.tick_params(axis='x', rotation=20)
ax1.grid(alpha=0.3, axis='y')

# 2. Outlier counts by group
ax2 = fig.add_subplot(gs[0, 1])
outlier_data = []
outlier_data.append({'Group': 'Human', 'Outliers': human_outliers, 'Total': n_human, 'Percentage': human_outliers/max(n_human,1)*100})

for model in ai_models_plot:
    if model in model_outliers:
        n_model = int((proposal_meta['group_model'] == model).sum())
        outlier_data.append({
            'Group': model,
            'Outliers': model_outliers[model],
            'Total': n_model,
            'Percentage': model_outliers[model]/max(n_model,1)*100,
        })

outlier_df = pd.DataFrame(outlier_data)
bars = ax2.bar(
    outlier_df['Group'],
    outlier_df['Percentage'],
    color=[colors.get(g, 'gray') for g in outlier_df['Group']],
    edgecolor='black',
    linewidth=1,
)
ax2.axhline(10, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Expected 10%')
ax2.set_ylabel('Outliers (%)', fontsize=11)
ax2.set_xlabel('Group', fontsize=11)
ax2.set_title('Outlier Percentages by Group', fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', rotation=20)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3, axis='y')

for bar, pct in zip(bars, outlier_df['Percentage']):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.5, f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

# 3. NN Group Analysis - Stacked bar chart
ax3 = fig.add_subplot(gs[0, 2])

groups = ['Human'] + [m for m in ai_models_plot if m in model_nn_analysis]
nn_from_human_pcts = [human_nn_diff_group/max(n_human,1)*100]   # For human, NN from AI
nn_from_same_pcts = [human_nn_same_group/max(n_human,1)*100]     # For human, NN from human
nn_from_other_ai_pcts = [0]

for model in [m for m in ai_models_plot if m in model_nn_analysis]:
    data = model_nn_analysis[model]
    nn_from_human_pcts.append(data['from_human']/max(data['total'],1)*100)
    nn_from_same_pcts.append(data['from_same_model']/max(data['total'],1)*100)
    nn_from_other_ai_pcts.append(data['from_other_ai']/max(data['total'],1)*100)

x_pos = np.arange(len(groups))
width = 0.6

ax3.bar(x_pos, nn_from_same_pcts, width, label='NN from same group', color='#3498db', edgecolor='black', linewidth=0.5)
ax3.bar(x_pos, nn_from_other_ai_pcts, width, bottom=nn_from_same_pcts, label='NN from other AI', color='#95a5a6', edgecolor='black', linewidth=0.5)
bottom = np.array(nn_from_same_pcts) + np.array(nn_from_other_ai_pcts)
ax3.bar(x_pos, nn_from_human_pcts, width, bottom=bottom, label='NN from different group', color='#e74c3c', edgecolor='black', linewidth=0.5)

ax3.set_ylabel('Percentage (%)', fontsize=11)
ax3.set_xlabel('Group', fontsize=11)
ax3.set_title('Nearest Neighbor Origins', fontsize=12, fontweight='bold')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(groups, rotation=20, fontsize=9)
ax3.legend(fontsize=8, loc='upper right')
ax3.grid(alpha=0.3, axis='y')
ax3.set_ylim([0, 100])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nearest_neighbor_by_model.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ Figure saved to: {FIGURES_DIR / "nearest_neighbor_by_model.png"}')


## 2.4 Visualize proposals in Embedding Space V

Visualize all proposals in 2D space using UMAP dimensionality reduction to see:
- **Clustering patterns**: How proposals group together
- **Outliers**: Unique proposals far from others (highlighted in magenta)
- **Group separation**: How Human vs AI proposals distribute in semantic space
- **Model characteristics**: Whether different AI models produce distinctly clustered proposals

**Color coding**: Human proposals in bright red for easy identification, AI models in muted colors. A complementary t-SNE view follows in the next cell.


In [ ]:
print("\n" + "="*85)
print("ANALYSIS 2.4: VISUALIZING PROPOSALS IN 2D EMBEDDING SPACE")
print("="*85)

import umap as umap_lib

# Load cached UMAP from Analysis 1.3; recompute only if cache missing
umap_cache_path = TABLES_DIR / 'cached' / 'proposal_umap2d.npy'
if umap_cache_path.exists():
    embeddings_2d = np.load(umap_cache_path)
    print(f"✓ Loaded cached UMAP from {umap_cache_path}  shape={embeddings_2d.shape}")
else:
    print("⚠  Cache not found — computing UMAP (run Analysis 1.3 first for consistent coordinates)")
    reducer_2_4 = umap_lib.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
                                  metric='cosine', random_state=42)
    embeddings_2d = reducer_2_4.fit_transform(X_prop)
    umap_cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(umap_cache_path, embeddings_2d)
    print(f"✓ UMAP computed and saved to {umap_cache_path}")

grp_col_24 = 'group_model' if 'group_model' in proposal_meta.columns else 'group'
group_order_umap = ['Human', 'Claude', 'Gemini', 'GPT-5.2']
if 'viz_marker_size' in proposal_meta.columns:
    marker_sizes_24 = proposal_meta['viz_marker_size'].to_numpy()
elif 'ranking' in proposal_meta.columns:
    marker_sizes_24 = np.where(proposal_meta['funding'].fillna(0)==1, 120, 55)
else:
    marker_sizes_24 = np.full(len(proposal_meta), 55)

fig, ax = plt.subplots(figsize=(10, 8))
for g in group_order_umap:
    mask_g = proposal_meta['group_binary'] == 'Human' if g == 'Human' \
             else proposal_meta[grp_col_24].str.contains(g, case=False, na=False)
    idx_g = proposal_meta[mask_g].index
    ax.scatter(embeddings_2d[idx_g, 0], embeddings_2d[idx_g, 1],
               c=[colors.get(g,'gray')], s=float(marker_sizes_24[idx_g].mean()),
               alpha=0.75, label=g, edgecolors='white', linewidths=0.4)
ax.set_title("Proposal Embedding Space — UMAP 2D", fontsize=12, fontweight='bold')
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.legend(fontsize=9); ax.grid(alpha=0.2)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'embedding_space_umap_2d.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: embedding_space_umap_2d.png")


### Analysis 2.4b: Per-Cluster Zoom — UMAP Detail View

In [ ]:
print("="*85)
print("ANALYSIS 2.4b: PER-CLUSTER ZOOM — UMAP DETAIL VIEW")
print("="*85)

if 'ward_labels' not in globals() or 'embeddings_2d' not in globals():
    print("⚠  ward_labels or embeddings_2d not found — run Analysis 1.3 and 2.4 first.")
else:
    _unique_k_24b = np.unique(ward_labels)
    _n_panels = min(len(_unique_k_24b), 4)
    _grp_col_24b = 'group_model' if 'group_model' in proposal_meta.columns else 'group'
    _model_order_24b = (['Human']
                        + sorted(proposal_meta.loc[proposal_meta['group_binary'] == 'AI',
                                                   _grp_col_24b].unique().tolist()))
    _clmap_24b = (cluster_label_map if 'cluster_label_map' in globals()
                  else {k: f'Cluster_{chr(65+k)}' for k in _unique_k_24b})

    _cluster_topic_labels_24b = {}
    if 'cluster_display_label_map' in globals():
        _cluster_topic_labels_24b = {
            int(k): str(cluster_display_label_map.get(int(k), _clmap_24b.get(int(k), f'Cluster {int(k)}')))
            for k in _unique_k_24b
        }

    # Build cluster → discriminative LDA topic label
    # Lift = (topic freq in this cluster) / (topic freq in all other clusters).
    # Pass 1 picks the highest-lift topic per cluster.
    # Pass 2 strips words shared with rival clusters' best topics so each panel
    # shows words that are distinctively overrepresented in that cluster.
    if not _cluster_topic_labels_24b and 'doc_topic_df' in globals() and 'topics' in globals() and 'title_norm' in proposal_meta.columns:
        _dtdf_z = doc_topic_df.copy()
        if 'title' in _dtdf_z.columns:
            _dtdf_z['_tnorm'] = (_dtdf_z['title'].astype(str).str.strip().str.lower()
                                  .str.replace(r'[^a-z0-9 ]', '', regex=True)
                                  .str.replace(r'\s+', ' ', regex=True).str.strip())
        else:
            _dtdf_z['_tnorm'] = ''
        _pm_z = proposal_meta[['title_norm']].copy()
        _pm_z['_ward_k'] = ward_labels
        _aligned_z = _pm_z.merge(_dtdf_z[['_tnorm', 'dominant_topic']],
                                  left_on='title_norm', right_on='_tnorm', how='left')

        # Per-cluster topic frequency distributions
        _all_topics_z = sorted(_aligned_z['dominant_topic'].dropna().unique())
        _cluster_sizes_z = {}
        _topic_counts_z = {}
        for _kk in _unique_k_24b:
            _rows = _aligned_z.loc[_aligned_z['_ward_k'] == _kk, 'dominant_topic'].dropna()
            _cluster_sizes_z[_kk] = max(len(_rows), 1)
            _topic_counts_z[_kk] = _rows.value_counts().to_dict()

        # Pass 1: find each cluster's highest-lift topic vs. its complement
        _best_topic_per_k = {}
        for _kk in _unique_k_24b:
            _other_rows = _aligned_z.loc[_aligned_z['_ward_k'] != _kk, 'dominant_topic'].dropna()
            _other_counts = _other_rows.value_counts().to_dict()
            _other_total = max(len(_other_rows), 1)
            _best_topic = None; _best_lift = -1
            for _t in _all_topics_z:
                _frac_this  = _topic_counts_z[_kk].get(_t, 0) / _cluster_sizes_z[_kk]
                _frac_other = _other_counts.get(_t, 0) / _other_total
                _lift = _frac_this / (_frac_other + 1e-6)
                if _lift > _best_lift:
                    _best_lift = _lift; _best_topic = _t
            _best_topic_per_k[_kk] = _best_topic

        # Pass 2: build label; prefer words not shared with rival clusters' best topics
        for _kk in _unique_k_24b:
            _bt = _best_topic_per_k.get(_kk)
            if not _bt:
                continue
            _t_idx = int(_bt.split('_')[1]) - 1
            if not (0 <= _t_idx < len(topics)):
                continue
            _rival_words = set()
            for _jj, _jt in _best_topic_per_k.items():
                if _jj != _kk and _jt:
                    _jt_idx = int(_jt.split('_')[1]) - 1
                    if 0 <= _jt_idx < len(topics):
                        _rival_words.update(topics[_jt_idx][:8])
            _distinct = [w for w in topics[_t_idx] if w not in _rival_words]
            _show = _distinct[:5] if len(_distinct) >= 3 else topics[_t_idx][:6]
            _cluster_topic_labels_24b[_kk] = f"{_bt}: {', '.join(_show)}"


    # Top-5 and funding masks for highlight outlines
    _top5_24b = proposal_meta['is_top5_ranked'].fillna(False).to_numpy(dtype=bool) \
                if 'is_top5_ranked' in proposal_meta.columns else np.zeros(len(proposal_meta), dtype=bool)
    _funding_24b = pd.to_numeric(proposal_meta.get('funding', pd.Series([np.nan]*len(proposal_meta))),
                                 errors='coerce').to_numpy()

    def _scatter_24b(ax_z, pts_idx, color, label, top5_arr, s=75, alpha=0.88, zorder=3):
        """Draw points split into regular (no outline) and top-5 (black outline)."""
        _reg = pts_idx[~top5_arr[pts_idx]]
        _top = pts_idx[top5_arr[pts_idx]]
        if len(_reg):
            ax_z.scatter(embeddings_2d[_reg, 0], embeddings_2d[_reg, 1],
                         c=color, s=s, alpha=alpha, edgecolors='none', linewidths=0,
                         label=f'{label} (n={len(pts_idx)})', zorder=zorder)
        if len(_top):
            ax_z.scatter(embeddings_2d[_top, 0], embeddings_2d[_top, 1],
                         c=color, s=s + 20, alpha=alpha,
                         edgecolors='black', linewidths=1.8,
                         label=(f'{label} ★top5' if not len(_reg) else None), zorder=zorder + 0.1)

    fig_z, axes_z = plt.subplots(1, _n_panels, figsize=(7 * _n_panels, 7), squeeze=False)
    axes_z = axes_z[0]

    for _pi, _k in enumerate(_unique_k_24b[:_n_panels]):
        ax_z = axes_z[_pi]
        _k_name = _clmap_24b.get(_k, f'Cluster_{chr(65+_k)}')
        _k_mask = ward_labels == _k
        _other  = ~_k_mask

        # AI author groups (colored by model, black outline for top-5)
        for _g in [m for m in _model_order_24b if m != 'Human']:
            _g_mask = proposal_meta[_grp_col_24b].eq(_g).values
            _idx = np.where(_k_mask & _g_mask)[0]
            if len(_idx) == 0:
                continue
            _scatter_24b(ax_z, _idx, colors.get(_g, 'gray'), _g, _top5_24b, zorder=3)

        # Human: split by funding shade, top-5 outlined
        _hu_mask = proposal_meta['group_binary'].eq('Human').values
        _hu_funded    = np.where(_k_mask & _hu_mask & np.isfinite(_funding_24b) & (_funding_24b == 1))[0]
        _hu_nonfunded = np.where(_k_mask & _hu_mask & ~(np.isfinite(_funding_24b) & (_funding_24b == 1)))[0]
        if len(_hu_nonfunded):
            _scatter_24b(ax_z, _hu_nonfunded, '#F08080', 'Human (funding=0/NA)', _top5_24b,
                         alpha=0.90, zorder=5)
        if len(_hu_funded):
            _scatter_24b(ax_z, _hu_funded, '#8B0000', 'Human (funding=1)', _top5_24b,
                         alpha=0.95, zorder=6)
        
        # Magenta outlier rings (within-cluster outliers preferred over global)
        _outlier_mask_24b = (outliers_within_cluster if 'outliers_within_cluster' in globals()
                             else outliers)
        _outlier_idx_24b = np.where(_k_mask & _outlier_mask_24b)[0]
        if len(_outlier_idx_24b):
            _ms_24b = (proposal_meta['viz_marker_size'].to_numpy()
                       if 'viz_marker_size' in proposal_meta.columns
                       else np.full(len(proposal_meta), 75.0))
            ax_z.scatter(embeddings_2d[_outlier_idx_24b, 0], embeddings_2d[_outlier_idx_24b, 1],
                         s=np.maximum(_ms_24b[_outlier_idx_24b] + 80, 220),
                         facecolors='none', edgecolors='magenta', linewidth=2.5,
                         alpha=0.75, label=f'Outliers (n={len(_outlier_idx_24b)})', zorder=8)

        # Zoom to cluster bounding box with padding
        _xs = embeddings_2d[_k_mask, 0]
        _ys = embeddings_2d[_k_mask, 1]
        _px = max((_xs.max() - _xs.min()) * 0.25, 0.5)
        _py = max((_ys.max() - _ys.min()) * 0.25, 0.5)
        ax_z.set_xlim(_xs.min() - _px, _xs.max() + _px)
        ax_z.set_ylim(_ys.min() - _py, _ys.max() + _py)
        _topic_hint_24b = _cluster_topic_labels_24b.get(_k, '')
        _panel_title_24b = f'{_k_name}  (n={int(_k_mask.sum())} total)'
        if _topic_hint_24b:
            _panel_title_24b += f'\n{_topic_hint_24b}'
        ax_z.set_title(_panel_title_24b, fontsize=11, fontweight='bold')
        ax_z.set_xlabel("UMAP-1", fontsize=10)
        ax_z.set_ylabel("UMAP-2", fontsize=10)
        ax_z.legend(fontsize=9, loc='best', framealpha=0.88)
        ax_z.grid(alpha=0.2, linestyle='--')

    fig_z.suptitle("Per-Cluster Zoom UMAP Projection for All Proposals\n"
                   "(black outline = top-5 ranked proposals within each author group; dark red = funded human)",
                   fontsize=12, fontweight='bold')
    plt.tight_layout()
    _out_z = FIGURES_DIR / 'embedding_space_umap_per_cluster_zoom.png'
    fig_z.savefig(_out_z, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {_out_z}")


In [ ]:
# UMAP projection — review-aware, consistent with all other 2D visualizations in this notebook
import umap as umap_lib

print("\n" + "="*85)
print("UMAP Projection (metadata-aligned, review-aware)")
print("="*85)

if 'proposal_meta' not in globals():
    raise RuntimeError('proposal_meta missing. Run shared precompute cell first.')

# ── Load / reuse cached UMAP (same coords as Analysis 2.4) ───────────────────
if 'embeddings_2d' in globals() and len(embeddings_2d) == len(proposal_meta):
    embeddings_2d_umap = embeddings_2d
    print(f"✓ Reusing cached UMAP  shape={embeddings_2d_umap.shape}")
else:
    umap_cache_path = TABLES_DIR / 'cached' / 'proposal_umap2d.npy'
    if umap_cache_path.exists():
        embeddings_2d_umap = np.load(umap_cache_path)
        print(f"✓ Loaded cached UMAP from {umap_cache_path}  shape={embeddings_2d_umap.shape}")
    else:
        print("⚠  Cache not found — computing UMAP (run Analysis 2.4 first for consistent coordinates)")
        reducer_umap = umap_lib.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
                                     metric='cosine', random_state=42)
        embeddings_2d_umap = reducer_umap.fit_transform(X_prop)
        umap_cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(umap_cache_path, embeddings_2d_umap)
        print(f"✓ UMAP computed and saved to {umap_cache_path}")

if len(embeddings_2d_umap) != len(proposal_meta):
    raise RuntimeError(f'Count mismatch: UMAP ({len(embeddings_2d_umap)}) vs proposal_meta ({len(proposal_meta)}).')

labels_prop    = proposal_meta['group_model'].to_numpy()
funding        = pd.to_numeric(proposal_meta['funding'], errors='coerce').to_numpy()
rank_vals      = pd.to_numeric(proposal_meta['viz_rank'], errors='coerce').to_numpy()
top5_mask      = proposal_meta['is_top5_ranked'].fillna(False).to_numpy(dtype=bool)
marker_sizes   = proposal_meta['viz_marker_size'].to_numpy()
ai_models_local = sorted(proposal_meta.loc[proposal_meta['group_binary'] == 'AI', 'group_model'].dropna().unique().tolist())

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(14, 10))


def _scatter_with_top5_outline(mask, color, label, alpha=0.6, zorder=3):
    mask = np.asarray(mask, dtype=bool)
    if not mask.any():
        return
    m_other = mask & (~top5_mask)
    m_top   = mask & top5_mask
    if m_other.any():
        pts = embeddings_2d_umap[m_other]
        ax.scatter(pts[:, 0], pts[:, 1], c=color, label=label,
                   s=marker_sizes[m_other], alpha=alpha,
                   edgecolors='none', linewidth=0, zorder=zorder)
    if m_top.any():
        pts = embeddings_2d_umap[m_top]
        ax.scatter(pts[:, 0], pts[:, 1], c=color,
                   label=(label if not m_other.any() else None),
                   s=marker_sizes[m_top], alpha=alpha,
                   edgecolors='black', linewidth=1.6, zorder=zorder + 0.1)


# AI proposals by model
for model in ai_models_local:
    _scatter_with_top5_outline(labels_prop == model,
                               colors.get(model, '#808080'), model, alpha=0.55, zorder=3)

# Human proposals (shade by funding)
human_mask    = proposal_meta['group_binary'].eq('Human').to_numpy()
human_funded  = human_mask & np.isfinite(funding) & (funding == 1)
human_nonfund = human_mask & (~human_funded)
_scatter_with_top5_outline(human_nonfund, '#F08080', 'Human (funding=0/NA)', alpha=0.82, zorder=10)
_scatter_with_top5_outline(human_funded,  '#8B0000', 'Human (funding=1)',    alpha=0.92, zorder=11)

# Outlier ring
_outliers = outliers_within_cluster if 'outliers_within_cluster' in globals() else outliers
outlier_indices = np.where(_outliers)[0]
outlier_coords  = embeddings_2d_umap[outlier_indices]
ax.scatter(outlier_coords[:, 0], outlier_coords[:, 1],
           s=np.maximum(marker_sizes[outlier_indices] + 160, 320),
           facecolors='none', edgecolors='magenta', linewidth=2.5, alpha=0.7,
           label=f'Outliers (n={len(outlier_indices)})', zorder=12)

# Title lookup
all_titles_umap = []
for r in (human_metadata if 'human_metadata' in globals() else []):
    all_titles_umap.append(r.get('proposal_title', r.get('title', '')))
for r in (ai_metadata if 'ai_metadata' in globals() else []):
    all_titles_umap.append(r.get('title', r.get('proposal_title', '')))

# Stacked, non-overlapping outlier labels with arrows
all_labels_umap = np.array(
    (['Human'] * int(proposal_meta['group_binary'].eq('Human').sum())) +
    labels_prop[proposal_meta['group_binary'].eq('AI').to_numpy()].tolist()
)

outlier_info = sorted(
    [(embeddings_2d_umap[idx][0], embeddings_2d_umap[idx][1],
      all_titles_umap[idx] if idx < len(all_titles_umap) else '',
      labels_prop[idx],
      rank_vals[idx] if idx < len(rank_vals) else np.nan,
      funding[idx]   if idx < len(funding)   else np.nan)
     for idx in outlier_indices],
    key=lambda p: -p[1],
)

x_data_max  = embeddings_2d_umap[:, 0].max()
x_data_min  = embeddings_2d_umap[:, 0].min()
x_label_col = x_data_min - (x_data_max - x_data_min) * 0.18

y_data_max = embeddings_2d_umap[:, 1].max()
y_data_min = embeddings_2d_umap[:, 1].min()
n_out = len(outlier_info)
y_label_positions = [
    y_data_max - i * (y_data_max - y_data_min) / max(n_out - 1, 1)
    for i in range(n_out)
]


def wrap_title(title, max_chars=32):
    words = title.split()
    lines, line = [], []
    for w in words:
        line.append(w)
        if len(' '.join(line)) > max_chars:
            lines.append(' '.join(line[:-1]))
            line = [w]
    if line:
        lines.append(' '.join(line))
    return '\n'.join(lines[:3])


def format_meta(rank, funding_value):
    rank_txt    = f'rank={int(rank)}'    if np.isfinite(rank)           else 'rank=NA'
    funding_txt = f'funding={int(funding_value)}' if np.isfinite(funding_value) else 'funding=NA'
    return f'{rank_txt}; {funding_txt}'


for (px, py, title, grp, rank, funding_value), y_lab in zip(outlier_info, y_label_positions):
    grp_color = colors.get(grp, '#808080')
    ax.annotate(
        f"{wrap_title(title)}\n{format_meta(rank, funding_value)}",
        xy=(px, py), xytext=(x_label_col, y_lab),
        fontsize=7, verticalalignment='center', zorder=20,
        arrowprops=dict(arrowstyle='->', color=grp_color, lw=1.3,
                        connectionstyle='arc3,rad=0.15'),
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor=grp_color, linewidth=1.2, alpha=0.92),
    )

# Group centroids
if (proposal_meta['group_binary'] == 'Human').any():
    mask_h = proposal_meta['group_binary'].eq('Human').to_numpy()
    c_h = embeddings_2d_umap[mask_h].mean(axis=0)
    ax.scatter(c_h[0], c_h[1], c='#8B0000', s=400, marker='X',
               edgecolors='black', linewidth=2, alpha=1.0, zorder=15)

for model in ai_models_local:
    mask_m = labels_prop == model
    if mask_m.any():
        c_m = embeddings_2d_umap[mask_m].mean(axis=0)
        ax.scatter(c_m[0], c_m[1], c=colors.get(model, '#808080'), s=350, marker='X',
                   edgecolors='black', linewidth=1.5, alpha=0.9, zorder=14)

ax.set_xlabel('UMAP Dimension 1', fontsize=12, fontweight='bold')
ax.set_ylabel('UMAP Dimension 2', fontsize=12, fontweight='bold')
ax.set_title('Proposal Embedding Space: UMAP Projection\nBlack outline = top-5 rank; dark red Human = funding=1',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='best', fontsize=9, framealpha=0.9, edgecolor='black')

plt.tight_layout()
out_path = FIGURES_DIR / 'embedding_space_umap_reviewaware.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Figure saved to: {out_path}")
print("✓ Outlier labels include title + rank + funding status")
print("="*85)


## Analysis 2.5: Grid Entropy of Proposal Occupancy

In [ ]:

print('='*85)
print('ANALYSIS 1.5: GRID ENTROPY OF PROPOSAL OCCUPANCY')
print('='*85)

rows = []
for g, idx in GROUPS.items():
    ent_raw = group_grid_entropy(pca_2d, idx, bins=5, normalize=False)
    ent_norm = group_grid_entropy(pca_2d, idx, bins=5, normalize=True)
    rows.append({
        'group': g,
        'n': len(idx),
        'grid_entropy': ent_raw,
        'grid_entropy_normalized': ent_norm,
        'bins': 5,
        'projection_method': 'PCA-2D',
    })

entropy_group_summary_df = pd.DataFrame(rows)
entropy_group_summary_df.to_csv(TABLES_DIR / 'diversity_entropy_group_summary.csv', index=False)
print(entropy_group_summary_df.to_string(index=False))
print('Saved diversity_entropy_group_summary.csv')


In [ ]:

# Visualization + permutation-test summary: Analysis 2.5
print('='*85)
print('VISUALIZATION: 2.5 GRID ENTROPY OF PROPOSAL OCCUPANCY')
print('='*85)

if 'entropy_group_summary_df' not in globals():
    raise RuntimeError('entropy_group_summary_df missing. Run Analysis 2.5 entropy cell first.')

order = ['Human'] + [m for m in ai_models if m in set(entropy_group_summary_df['group'])] + ['All AI']
plot_df = entropy_group_summary_df.set_index('group').reindex(order).reset_index()
palette = [colors.get(g, '#888888') if g != 'All AI' else '#B56576' for g in plot_df['group']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(
    np.arange(len(plot_df)),
    plot_df['grid_entropy_normalized'].to_numpy(dtype=float),
    color=palette,
    edgecolor='black',
    alpha=0.9,
)
axes[0].set_xticks(np.arange(len(plot_df)))
axes[0].set_xticklabels(plot_df['group'].tolist(), rotation=20)
axes[0].set_ylabel('Normalized Grid Entropy')
axes[0].set_title('Grid Entropy (normalized) by Group', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3, axis='y')
for i,v in enumerate(plot_df['grid_entropy_normalized'].to_numpy(dtype=float)):
    if np.isfinite(v):
        axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=8)

axes[1].bar(
    np.arange(len(plot_df)),
    plot_df['grid_entropy'].to_numpy(dtype=float),
    color=palette,
    edgecolor='black',
    alpha=0.9,
)
axes[1].set_xticks(np.arange(len(plot_df)))
axes[1].set_xticklabels(plot_df['group'].tolist(), rotation=20)
axes[1].set_ylabel('Raw Grid Entropy')
axes[1].set_title('Grid Entropy (raw) by Group', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')
for i,v in enumerate(plot_df['grid_entropy'].to_numpy(dtype=float)):
    if np.isfinite(v):
        axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
fig_path = FIGURES_DIR / 'diversity_entropy_group_summary.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved figure: {fig_path}')

# Permutation tests: group-level entropy contrasts vs Human

def _entropy_norm_metric(coords_subset):
    return group_grid_entropy(coords_subset, np.arange(len(coords_subset)), bins=5, normalize=True)

rows = []
for g in ai_models + ['All AI']:
    if g not in GROUPS:
        continue
    pt = permutation_test_group_metric(
        pca_2d,
        proposal_meta['group_model'].to_numpy(),
        GROUPS[g],
        GROUPS['Human'],
        _entropy_norm_metric,
        n_perm=10000,
        random_state=42,
        use_distance_submat=False,
    )
    rows.append({
        'comparison': f'{g} vs Human',
        'group1': g,
        'group2': 'Human',
        **pt,
    })

entropy_pairwise_perm_df = pd.DataFrame(rows)
if len(entropy_pairwise_perm_df):
    entropy_pairwise_perm_df = apply_multiple_testing(entropy_pairwise_perm_df, p_cols=('perm_p_value',), method='holm')
entropy_pairwise_perm_df.to_csv(TABLES_DIR / 'diversity_entropy_pairwise_permutation.csv', index=False)

if len(entropy_pairwise_perm_df):
    print('\nPermutation tests (normalized entropy, vs Human):')
    disp_cols = ['comparison', 'obs_diff', 'perm_p_value']
    if 'perm_p_value_adj_holm' in entropy_pairwise_perm_df.columns:
        disp_cols.append('perm_p_value_adj_holm')
    print(entropy_pairwise_perm_df[disp_cols].sort_values(disp_cols[-1]).to_string(index=False))
print('Saved diversity_entropy_pairwise_permutation.csv')


# PART III: NOVELTY

Can AI create more novel proposals than teams of human scientists?

We'll compare how far proposals are from existing literature (PubMed corpus).

## Step 1: Load Prepared Literature Corpus

In [ ]:
import json

# Load prepared literature corpus (generated in prepare_data_for_analysis.ipynb)
print('='*85)
print('LOADING PREPARED LITERATURE CORPUS')
print('='*85)

literature_corpus_path = PREPARED_LITERATURE_CORPUS_PATH
if not literature_corpus_path.exists():
    raise FileNotFoundError(
        f'Prepared literature corpus not found at {literature_corpus_path}. '
        'Run prepare_data_for_analysis.ipynb first.'
    )

with open(literature_corpus_path, 'r') as f:
    corpus_data = json.load(f)

articles = corpus_data.get('articles', [])
print(f'\n✓ Loaded {len(articles)} PubMed articles')
print(f'  Search queries: {len(corpus_data.get("search_queries", []))}')
print(f'  Source file: {literature_corpus_path}')

corpus_texts = []
for article in articles:
    title = article.get('title', '')
    abstract = article.get('abstract', '')
    text = f'Title: {title}\n\nAbstract: {abstract}'
    corpus_texts.append(text)

print(f'\n✓ Prepared {len(corpus_texts)} literature texts')
print(f'  Average length: {np.mean([len(t) for t in corpus_texts]):.0f} characters')
print('='*85)


In [ ]:
# Visualize literature corpus: articles per query + publication year distribution
import re

fig, axes = plt.subplots(2, 1, figsize=(12, 9))

# --- Panel 1: Articles per search query ---
ax1 = axes[0]
queries = corpus_data['search_queries']
labels = [q['label'] for q in queries]
counts = [q['new_unique_added'] for q in queries]
colors_bar = plt.cm.viridis(np.linspace(0.2, 0.8, len(labels)))
bars = ax1.barh(range(len(labels)), counts, color=colors_bar, edgecolor='black', linewidth=0.8)
ax1.set_yticks(range(len(labels)))
ax1.set_yticklabels(labels, fontsize=8, ha='right')
ax1.set_xlabel('Number of articles', fontsize=12, fontweight='bold')
ax1.set_title('Articles per search query', fontsize=14, fontweight='bold')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)
for bar, c in zip(bars, counts):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, str(c), va='center', fontsize='10')

# --- Panel 2: Publication year distribution ---
ax2 = axes[1]
def parse_year(pub_date):
    if not pub_date or not isinstance(pub_date, str):
        return None
    m = re.match(r'(\d{4})', pub_date.strip())
    return int(m.group(1)) if m else None

years = [parse_year(a.get('publication_date')) for a in articles]
years = [y for y in years if y is not None]
year_counts = pd.Series(years).value_counts().sort_index()

ax2.bar(year_counts.index, year_counts.values, color='steelblue', edgecolor='black', linewidth=0.8, alpha=0.85)
ax2.set_xlabel('Publication year', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of articles', fontsize=12, fontweight='bold')
ax2.set_title('Publication date distribution (year only)', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
if years:
    ax2.text(0.98, 0.98, f'Total: {len(years)} articles\nRange: {min(years)}–{max(years)}', transform=ax2.transAxes,
             fontsize=10, verticalalignment='top', horizontalalignment='left',
             bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'literature_corpus_overview.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Figure saved to: {FIGURES_DIR / 'literature_corpus_overview.png'}")

# Print search terms (queries) used for each category
print('\n' + '='*85)
print('SEARCH TERMS (QUERIES) USED PER CATEGORY')
print('='*85)
for q in queries:
    print(f"\n• {q['label']} (n={q['new_unique_added']} articles):")
    print(f"  {q['query']}")
print('='*85)


## Step 2: Embed Literature Corpus 

Embed all 350 abstracts using BioLinkBERT-Large (same model used for proposals).

In [ ]:
# Embed literature corpus using the same approach (truncated to 512 tokens)
print("="*85)
print("EMBEDDING LITERATURE CORPUS")
print("="*85)
print("Using BioLinkBERT-Large (same as proposals, truncated to 512 tokens)")
print()

# Check if embeddings already exist
literature_embeddings_file = LITERATURE_EMBEDDINGS_FILE

if literature_embeddings_file.exists():
    print(f"✓ Loading cached literature embeddings from {literature_embeddings_file}")
    with open(literature_embeddings_file, 'rb') as f:
        literature_data = pickle.load(f)
        literature_embeddings = literature_data['embeddings']
    print(f"✓ Loaded {len(literature_embeddings)} embeddings")
else:
    print("Embedding literature corpus (this will take a few minutes)...")
    literature_embeddings = get_embeddings(corpus_texts)
    print(f"✓ Literature embeddings shape: {literature_embeddings.shape}")
    
    # Cache for future use
    literature_embeddings_file.parent.mkdir(parents=True, exist_ok=True)
    with open(literature_embeddings_file, 'wb') as f:
        pickle.dump({
            'embeddings': literature_embeddings,
            'model_name': model_name,
            'timestamp': datetime.now().isoformat()
        }, f)
    print(f"✓ Cached embeddings to: {literature_embeddings_file}")

print("="*85)

# Load abstract-only proposal embeddings for all PubMed/literature comparisons.
if ABSTRACT_EMBEDDINGS_FILE.exists():
    print(f"Loading prepared abstract-only proposal embeddings from {ABSTRACT_EMBEDDINGS_FILE}")
    with open(ABSTRACT_EMBEDDINGS_FILE, 'rb') as f:
        abstract_embeddings_data = pickle.load(f)
    ai_literature_embeddings = np.asarray(abstract_embeddings_data['ai_embeddings'])
    human_literature_embeddings = np.asarray(abstract_embeddings_data['human_embeddings'])
else:
    print('No abstract-only proposal embedding cache found - generating fallback now.')
    ai_literature_embeddings = get_embeddings(ai_df['abstract_text'].fillna('').tolist())
    human_literature_embeddings = get_embeddings(human_df['abstract_text'].fillna('').tolist())
    abstract_embeddings_data = {
        'ai_embeddings': ai_literature_embeddings,
        'human_embeddings': human_literature_embeddings,
        'ai_metadata': ai_metadata,
        'human_metadata': human_metadata,
        'model_name': model_name,
        'text_field': 'rephrased_abstract',
        'timestamp': datetime.now().isoformat(),
    }
    with open(ABSTRACT_EMBEDDINGS_FILE, 'wb') as f:
        pickle.dump(abstract_embeddings_data, f)

print(f'Abstract-only proposal embeddings for literature comparisons: Human {human_literature_embeddings.shape}, AI {ai_literature_embeddings.shape}')


## Shared Novelty Precomputation (CAREFUL, computationally expensive)

This block computes reusable proposal-to-literature distance objects and literature local-density baselines used by all later novelty analyses.

In [ ]:

# proposal/literature normalized matrices
# Literature comparisons use Section-1/abstract-only proposal embeddings so the
# proposal text representation matches literature abstracts.
X_lit = np.asarray(literature_embeddings, dtype=np.float32)
X_lit = X_lit / np.clip(np.linalg.norm(X_lit, axis=1, keepdims=True), 1e-12, None)

X_prop_lit = np.vstack([human_literature_embeddings, ai_literature_embeddings]).astype(np.float32)
X_prop_lit = X_prop_lit / np.clip(np.linalg.norm(X_prop_lit, axis=1, keepdims=True), 1e-12, None)
if 'proposal_meta' in globals() and len(X_prop_lit) != len(proposal_meta):
    raise RuntimeError(f'Section-1 proposal embeddings ({len(X_prop_lit)}) do not align with proposal_meta ({len(proposal_meta)}).')

S_pl = X_prop_lit @ X_lit.T
D_pl = 1.0 - S_pl

D_pl_sorted_idx = np.argsort(D_pl, axis=1)
D_pl_sorted_dist = np.take_along_axis(D_pl, D_pl_sorted_idx, axis=1)

# literature self-kNN cache to k=50 — load from disk if available, compute once
_lit_knn_cache = cache_dir / 'lit_knn_distances_50.npy'
_lit_n_cache   = cache_dir / 'lit_knn_n.npy'

if (_lit_knn_cache.exists() and _lit_n_cache.exists() and
        int(np.load(_lit_n_cache)) == len(X_lit)):
    lit_knn_distances_50 = np.load(_lit_knn_cache)
    print(f'✓ Loaded lit kNN cache ({len(X_lit)} papers) from {_lit_knn_cache}')
else:
    print(f'Computing lit-lit kNN ({len(X_lit):,} × {len(X_lit):,}) — one-time cost...')
    S_ll = X_lit @ X_lit.T
    D_ll = 1.0 - S_ll
    np.fill_diagonal(D_ll, np.inf)
    ll_sorted = np.sort(D_ll, axis=1)
    lit_knn_distances_50 = ll_sorted[:, :50]
    np.save(_lit_knn_cache, lit_knn_distances_50)
    np.save(_lit_n_cache, np.array([len(X_lit)]))
    print(f'✓ Saved lit kNN cache to {_lit_knn_cache}')

lit_mean_knn_5 = lit_knn_distances_50[:, :5].mean(axis=1)
lit_mean_knn_10 = lit_knn_distances_50[:, :10].mean(axis=1)
lit_mean_knn_20 = lit_knn_distances_50[:, :20].mean(axis=1)
lit_mean_knn_50 = lit_knn_distances_50[:, :50].mean(axis=1)

print('D_pl shape:', D_pl.shape)
print('Computed reusable literature kNN baselines up to k=50')


## Step 2.5: Element Novelty Percentiles

Compute proposal-level element novelty percentiles from proposal-to-literature cosine distances.

In [ ]:

print('='*85)
print('STEP 2.5: ELEMENT NOVELTY PERCENTILES')
print('='*85)

el = compute_element_novel_percentiles(D_pl, q_list=[0,1,5,10])

element_novelty_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
for c in ['element_novel_0', 'element_novel_1', 'element_novel_5', 'element_novel_10']:
    element_novelty_df[c] = el[c]

# Tests for each element metric (AI vs Human)
rows = []
for metric in ['element_novel_0', 'element_novel_1', 'element_novel_5', 'element_novel_10']:
    hv = element_novelty_df.loc[element_novelty_df['group_binary']=='Human', metric].to_numpy()
    for g in ai_models:
        gv = element_novelty_df.loc[element_novelty_df['group_model']==g, metric].to_numpy()
        if len(gv) == 0:
            continue
        r = run_group_comparison(gv, hv, n_permutations=10000, n_boot=5000, random_state=42)
        r.update({'metric_name': metric, 'comparison': f'{g} vs Human', 'group1': g, 'group2': 'Human', 'n_group1': len(gv), 'n_group2': len(hv)})
        rows.append(r)

element_novelty_tests_df = pd.DataFrame(rows)
if len(element_novelty_tests_df):
    by_metric = []
    for metric, d in element_novelty_tests_df.groupby('metric_name', sort=False):
        by_metric.append(apply_multiple_testing(d, p_cols=('p_value_mw','p_value_perm'), method='holm'))
    element_novelty_tests_df = pd.concat(by_metric, ignore_index=True)

element_novelty_df.to_csv(TABLES_DIR / 'novelty_element_percentiles.csv', index=False)
element_novelty_tests_df.to_csv(TABLES_DIR / 'novelty_element_percentiles_pairwise_tests.csv', index=False)

print('Saved novelty_element_percentiles.csv and novelty_element_percentiles_pairwise_tests.csv')


## Step 3: Raw Novelty Scores (Mean k-NN to Literature)

In [ ]:

print('='*85)
print('STEP 3: RAW NOVELTY SCORES (MEAN k-NN TO LITERATURE)')
print('='*85)

knn = compute_mean_knn_novelty(D_pl_sorted_dist, ks=[5,10,20,50])
mean_knn_novelty_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
for c in ['mean_knn_5', 'mean_knn_10', 'mean_knn_20', 'mean_knn_50']:
    mean_knn_novelty_df[c] = knn[c]

# normalized novelty from local literature density around nearest 10 papers
proposal_top10_lit_idx = D_pl_sorted_idx[:, :10]
novelty_ratio, novelty_z, proposal_lit_local_density10 = compute_local_density_normalized_novelty(
    mean_knn_novelty_df['mean_knn_10'].to_numpy(),
    proposal_top10_lit_idx,
    lit_mean_knn_10,
)

novelty_local_density_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
novelty_local_density_df['mean_knn_10'] = mean_knn_novelty_df['mean_knn_10'].to_numpy()
novelty_local_density_df['lit_local_density_10'] = proposal_lit_local_density10
novelty_local_density_df['novelty_ratio'] = novelty_ratio
novelty_local_density_df['novelty_z'] = novelty_z

# legacy variables used later
all_mean_knn_10 = mean_knn_novelty_df['mean_knn_10'].to_numpy()
human_novelty_scores = mean_knn_novelty_df.loc[mean_knn_novelty_df['group_binary']=='Human', 'mean_knn_10'].to_numpy()
ai_novelty_scores = mean_knn_novelty_df.loc[mean_knn_novelty_df['group_binary']=='AI', 'mean_knn_10'].to_numpy()
model_novelty_scores = {m: mean_knn_novelty_df.loc[mean_knn_novelty_df['group_model']==m, 'mean_knn_10'].to_numpy() for m in ai_models}

# tests for MeanKNN family
rows_knn = []
for metric in ['mean_knn_5', 'mean_knn_10', 'mean_knn_20', 'mean_knn_50']:
    hv = mean_knn_novelty_df.loc[mean_knn_novelty_df['group_binary']=='Human', metric].to_numpy()
    for g in ai_models:
        gv = mean_knn_novelty_df.loc[mean_knn_novelty_df['group_model']==g, metric].to_numpy()
        if len(gv)==0:
            continue
        r = run_group_comparison(gv, hv, n_permutations=10000, n_boot=5000, random_state=42)
        r.update({'metric_name': metric, 'comparison': f'{g} vs Human', 'group1': g, 'group2': 'Human', 'n_group1': len(gv), 'n_group2': len(hv)})
        rows_knn.append(r)

mean_knn_novelty_tests_df = pd.DataFrame(rows_knn)
if len(mean_knn_novelty_tests_df):
    by_metric = []
    for metric, d in mean_knn_novelty_tests_df.groupby('metric_name', sort=False):
        by_metric.append(apply_multiple_testing(d, p_cols=('p_value_mw','p_value_perm'), method='holm'))
    mean_knn_novelty_tests_df = pd.concat(by_metric, ignore_index=True)

# tests for normalized novelty
rows_norm = []
for metric in ['novelty_ratio', 'novelty_z']:
    hv = novelty_local_density_df.loc[novelty_local_density_df['group_binary']=='Human', metric].dropna().to_numpy()
    for g in ai_models:
        gv = novelty_local_density_df.loc[novelty_local_density_df['group_model']==g, metric].dropna().to_numpy()
        if len(gv)==0 or len(hv)==0:
            continue
        r = run_group_comparison(gv, hv, n_permutations=10000, n_boot=5000, random_state=42)
        r.update({'metric_name': metric, 'comparison': f'{g} vs Human', 'group1': g, 'group2': 'Human', 'n_group1': len(gv), 'n_group2': len(hv)})
        rows_norm.append(r)

novelty_local_density_tests_df = pd.DataFrame(rows_norm)
if len(novelty_local_density_tests_df):
    by_metric = []
    for metric, d in novelty_local_density_tests_df.groupby('metric_name', sort=False):
        by_metric.append(apply_multiple_testing(d, p_cols=('p_value_mw','p_value_perm'), method='holm'))
    novelty_local_density_tests_df = pd.concat(by_metric, ignore_index=True)

mean_knn_novelty_df.to_csv(TABLES_DIR / 'novelty_mean_knn_scores.csv', index=False)
mean_knn_novelty_tests_df.to_csv(TABLES_DIR / 'novelty_mean_knn_pairwise_tests.csv', index=False)
novelty_local_density_df.to_csv(TABLES_DIR / 'novelty_local_density_normalized.csv', index=False)
novelty_local_density_tests_df.to_csv(TABLES_DIR / 'novelty_local_density_pairwise_tests.csv', index=False)

print('Saved novelty_mean_knn_scores.csv, novelty_mean_knn_pairwise_tests.csv, novelty_local_density_normalized.csv, novelty_local_density_pairwise_tests.csv')


## Step 5: Statistical Tests for Novelty Metrics

Combined inferential report for ElementNovel, MeanKNN, and local-density normalized novelty metrics.

In [ ]:

print('='*85)
print('STEP 5: COMBINED STATISTICAL TESTS FOR NOVELTY METRICS')
print('='*85)

frames = []
if 'element_novelty_tests_df' in globals() and len(element_novelty_tests_df):
    tmp = element_novelty_tests_df.copy(); tmp['metric_family'] = 'ElementNovel'; frames.append(tmp)
if 'mean_knn_novelty_tests_df' in globals() and len(mean_knn_novelty_tests_df):
    tmp = mean_knn_novelty_tests_df.copy(); tmp['metric_family'] = 'MeanKNN'; frames.append(tmp)
if 'novelty_local_density_tests_df' in globals() and len(novelty_local_density_tests_df):
    tmp = novelty_local_density_tests_df.copy(); tmp['metric_family'] = 'NormalizedNovelty'; frames.append(tmp)

if frames:
    novelty_all_pairwise_tests_df = pd.concat(frames, ignore_index=True)
else:
    novelty_all_pairwise_tests_df = pd.DataFrame()

if len(novelty_all_pairwise_tests_df):
    novelty_all_pairwise_tests_df['metric_name'] = novelty_all_pairwise_tests_df['metric_name'].fillna('mean_knn_10')
    novelty_all_pairwise_tests_df['q_value_holm'] = novelty_all_pairwise_tests_df.get('p_value_mw_adj_holm', np.nan)
    novelty_all_pairwise_tests_df['perm_q_value_holm'] = novelty_all_pairwise_tests_df.get('p_value_perm_adj_holm', np.nan)
    novelty_all_pairwise_tests_df['cliffs_delta'] = novelty_all_pairwise_tests_df.get('delta', np.nan)
    novelty_all_pairwise_tests_df['delta_magnitude'] = novelty_all_pairwise_tests_df.get('delta_interp', None)

    keep_cols = [
        'metric_family','metric_name','comparison','group1','group2','n_group1','n_group2',
        'u_stat','p_value_mw','q_value_holm','cliffs_delta','delta_magnitude',
        'p_value_perm','perm_q_value_holm','obs_diff_mean','obs_diff_median','ci_low','ci_high'
    ]
    for c in keep_cols:
        if c not in novelty_all_pairwise_tests_df.columns:
            novelty_all_pairwise_tests_df[c] = np.nan
    novelty_all_pairwise_tests_df = novelty_all_pairwise_tests_df[keep_cols].rename(columns={
        'p_value_mw': 'p_value',
        'p_value_perm': 'perm_p_value',
        'obs_diff_mean': 'mean_diff',
        'obs_diff_median': 'median_diff',
        'ci_low': 'boot_ci_low',
        'ci_high': 'boot_ci_high',
    })

novelty_all_pairwise_tests_df.to_csv(TABLES_DIR / 'novelty_all_pairwise_tests.csv', index=False)
print('Saved novelty_all_pairwise_tests.csv')
# Display results
if len(novelty_all_pairwise_tests_df):
    fmt_cols = ['p_value', 'perm_p_value', 'q_value_holm', 'perm_q_value_holm',
                'cliffs_delta', 'mean_diff', 'boot_ci_low', 'boot_ci_high']
    display_df = novelty_all_pairwise_tests_df.copy()
    for c in fmt_cols:
        if c in display_df.columns:
            display_df[c] = display_df[c].map(lambda x: f'{x:.4f}' if pd.notna(x) else '')
    for family, grp in display_df.groupby('metric_family', sort=False):
        print(f'\n--- {family} ---')
        print(grp[['metric_name','comparison','n_group1','n_group2',
                    'cliffs_delta','delta_magnitude',
                    'p_value','q_value_holm',
                    'perm_p_value','perm_q_value_holm',
                    'mean_diff','boot_ci_low','boot_ci_high']].to_string(index=False))


# compatibility with older visualization variable
novelty_comparison_results = []
if 'mean_knn_novelty_tests_df' in globals() and len(mean_knn_novelty_tests_df):
    sel = mean_knn_novelty_tests_df[mean_knn_novelty_tests_df['metric_name']=='mean_knn_10'].copy()
    sel = sel.rename(columns={'group1': 'group'})
    novelty_comparison_results = sel.to_dict('records')


## Step 6: Visualize Novelty Results

In [ ]:
print('='*85)
print('STEP 6: NOVELTY VISUALIZATIONS (ELEMENT, MEANKNN, NORMALIZED)')
print('='*85)

def _novelty_forest_panel(ax, metrics, metric_labels, ai_models_list,
                           source_df, tests_df, colors_dict, metric_col='metric_name'):
    """
    Grouped forest plot: rows = metric sections × AI model, x = Cliff's delta.
    Section headers are rendered as labeled rows inside the plot to avoid
    overlapping the model-name y-tick labels.
    """
    human_mask = source_df['group_binary'] == 'Human'
    ROW_GAP = 0.6

    # Each element: (y, delta, lo, hi, color, model, mw_h, perm_h, is_header, header_label)
    rows = []
    y = 0.0

    for metric, mlabel in zip(metrics, metric_labels):
        h_vals = source_df.loc[human_mask, metric].dropna().to_numpy(dtype=float)
        # header row for this metric section
        rows.append((y, None, None, None, None, None, None, None, True, mlabel))
        y += 1
        for model in ai_models_list:
            g_vals = source_df.loc[source_df['group_model'] == model, metric].dropna().to_numpy(dtype=float)
            if len(g_vals) == 0:
                y += 1; continue
            delta_val = cliffs_delta(g_vals, h_vals)
            lo, hi = bootstrap_cliffs_delta_ci(g_vals, h_vals, n_boot=2000, random_state=42)
            mw_holm, perm_holm = np.nan, np.nan
            if tests_df is not None and len(tests_df):
                sub = tests_df.loc[
                    (tests_df[metric_col] == metric) & (tests_df['group1'] == model)
                ]
                if len(sub):
                    mw_holm   = sub.iloc[0].get('p_value_mw_adj_holm',   np.nan)
                    perm_holm = sub.iloc[0].get('p_value_perm_adj_holm', np.nan)
            rows.append((y, delta_val, lo, hi, colors_dict.get(model, '#808080'),
                         model, mw_holm, perm_holm, False, None))
            y += 1
        y += ROW_GAP

    ax.axvline(0, color='black', linewidth=1.0, alpha=0.8)
    ytick_pos, ytick_labels = [], []
    prev_header_y = None

    for row in rows:
        ypos, dv, lo, hi, c, model, mw_h, perm_h, is_header, hlabel = row
        if is_header:
            # separator line above every header except the first
            if prev_header_y is not None:
                ax.axhline(ypos - 0.45, color='gray', linewidth=0.6, linestyle='--', alpha=0.4)
            ax.text(0.0, ypos, hlabel, va='center', ha='center',
                    fontsize=9.5, fontweight='bold', color='#333333',
                    bbox=dict(boxstyle='round,pad=0.15', fc='#f0f0f0', ec='#cccccc', lw=0.8),
                    clip_on=False)
            ytick_pos.append(ypos)
            ytick_labels.append('')
            prev_header_y = ypos
        else:
            ax.errorbar(dv, ypos,
                        xerr=np.array([[dv - lo], [hi - dv]]),
                        fmt='o', color=c, ecolor=c, elinewidth=1.2, capsize=3, markersize=7)
            txt = f"MW(Holm)={fmt_p(mw_h)} | Perm(Holm)={fmt_p(perm_h)}"
            ax.text(1.04, ypos, txt, va='center', ha='left', fontsize=8.5, clip_on=False)
            ytick_pos.append(ypos)
            ytick_labels.append(model)

    ax.set_yticks(ytick_pos)
    ax.set_yticklabels(ytick_labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(-1.05, 1.05)
    ax.set_xlabel("Cliff's δ (bootstrap 95% CI)", fontsize=10)
    ax.set_title("Effect size (AI − Human) + significance", fontsize=11)
    ax.grid(alpha=0.2, axis='x')


ai_models_list = list(ai_models)
n_models = len(ai_models_list)

# ── Panel A: Element novelty percentiles ─────────────────────────────────────
metrics_a  = ['element_novel_0', 'element_novel_1', 'element_novel_5', 'element_novel_10']
mlabels_a  = ['el_novel_0', 'el_novel_1', 'el_novel_5', 'el_novel_10']
n_rows_a   = len(metrics_a) * n_models + (len(metrics_a) - 1)
fig_h_a    = max(7, n_rows_a * 0.55 + 2)

sns.set_theme(style='whitegrid', context='talk')
fig_a = plt.figure(figsize=(24, fig_h_a), dpi=140)
gs_a  = fig_a.add_gridspec(1, 2, width_ratios=[1.4, 1.0], wspace=0.22)
axL_a = fig_a.add_subplot(gs_a[0, 0])
axR_a = fig_a.add_subplot(gs_a[0, 1])

plot_df_a = element_novelty_df.melt(
    id_vars=['proposal_uid', 'group_model', 'group_binary'],
    value_vars=metrics_a, var_name='metric', value_name='value',
)
styled_grouped_boxplot_with_points(
    axL_a,
    plot_df_a,
    x='metric',
    y='value',
    hue='group_model',
    order=metrics_a,
    hue_order=['Human'] + ai_models_list,
    palette=colors,
    width=0.82,
    jitter=0.15,
    point_size=20,
    point_alpha=0.50,
    random_state=42,
    show_group_legend=True,
    show_metadata_legend=True,
    legend_loc='best',
)
axL_a.set_title('Element Novelty Percentiles: Human vs Each AI Model\nDiamond/error bar = mean bootstrap 95% CI')
axL_a.set_xlabel('Metric'); axL_a.set_ylabel('Cosine Distance')
axL_a.tick_params(axis='x', rotation=20); axL_a.grid(alpha=0.3)

_novelty_forest_panel(
    axR_a, metrics_a, mlabels_a, ai_models_list,
    element_novelty_df,
    element_novelty_tests_df if 'element_novelty_tests_df' in globals() else None,
    colors,
)
fig_a.suptitle('Element Novelty + Effect Sizes', y=1.01, fontsize=14, fontweight='bold')
fig_a.tight_layout()
fig_a.savefig(FIGURES_DIR / 'novelty_analysis_element_percentiles.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Panel B: MeanKNN k-sensitivity ───────────────────────────────────────────
metrics_b = ['mean_knn_5', 'mean_knn_10', 'mean_knn_20', 'mean_knn_50']
mlabels_b = ['kNN-5', 'kNN-10', 'kNN-20', 'kNN-50']
n_rows_b  = len(metrics_b) * n_models + (len(metrics_b) - 1)
fig_h_b   = max(7, n_rows_b * 0.55 + 2)

fig_b = plt.figure(figsize=(24, fig_h_b), dpi=140)
gs_b  = fig_b.add_gridspec(1, 2, width_ratios=[1.4, 1.0], wspace=0.22)
axL_b = fig_b.add_subplot(gs_b[0, 0])
axR_b = fig_b.add_subplot(gs_b[0, 1])

plot_df_b = mean_knn_novelty_df.melt(
    id_vars=['proposal_uid', 'group_model', 'group_binary'],
    value_vars=metrics_b, var_name='metric', value_name='value',
)
styled_grouped_boxplot_with_points(
    axL_b,
    plot_df_b,
    x='metric',
    y='value',
    hue='group_model',
    order=metrics_b,
    hue_order=['Human'] + ai_models_list,
    palette=colors,
    width=0.82,
    jitter=0.15,
    point_size=20,
    point_alpha=0.50,
    random_state=42,
    show_group_legend=True,
    show_metadata_legend=True,
    legend_loc='best',
)
axL_b.set_title('Mean-kNN Novelty Sensitivity by k: Human vs Each AI Model\nDiamond/error bar = mean bootstrap 95% CI')
axL_b.set_xlabel('Metric'); axL_b.set_ylabel('Cosine Distance')
axL_b.tick_params(axis='x', rotation=20); axL_b.grid(alpha=0.3)

_novelty_forest_panel(
    axR_b, metrics_b, mlabels_b, ai_models_list,
    mean_knn_novelty_df,
    mean_knn_novelty_tests_df if 'mean_knn_novelty_tests_df' in globals() else None,
    colors,
)
fig_b.suptitle('Mean-kNN Novelty + Effect Sizes', y=1.01, fontsize=14, fontweight='bold')
fig_b.tight_layout()
fig_b.savefig(FIGURES_DIR / 'novelty_analysis_mean_knn.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Panel C: Normalized novelty ───────────────────────────────────────────────
metrics_c = ['novelty_z', 'novelty_ratio']
mlabels_c = ['novelty_z', 'novelty_ratio']
n_rows_c  = len(metrics_c) * n_models + (len(metrics_c) - 1)
fig_h_c   = max(6, n_rows_c * 0.55 + 2)

fig_c = plt.figure(figsize=(24, fig_h_c), dpi=140)
gs_c  = fig_c.add_gridspec(1, 2, width_ratios=[1.4, 1.0], wspace=0.22)
axL_c = fig_c.add_subplot(gs_c[0, 0])
axR_c = fig_c.add_subplot(gs_c[0, 1])

plot_df_c = novelty_local_density_df.melt(
    id_vars=['proposal_uid', 'group_model', 'group_binary'],
    value_vars=metrics_c, var_name='metric', value_name='value',
)
styled_grouped_boxplot_with_points(
    axL_c,
    plot_df_c,
    x='metric',
    y='value',
    hue='group_model',
    order=metrics_c,
    hue_order=['Human'] + ai_models_list,
    palette=colors,
    width=0.82,
    jitter=0.15,
    point_size=20,
    point_alpha=0.50,
    random_state=42,
    show_group_legend=True,
    show_metadata_legend=True,
    legend_loc='best',
)
axL_c.set_title('Local-Density Normalized Novelty: Human vs Each AI Model\nDiamond/error bar = mean bootstrap 95% CI')
axL_c.set_xlabel('Metric'); axL_c.set_ylabel('Value')
axL_c.tick_params(axis='x', rotation=20); axL_c.grid(alpha=0.3)

_novelty_forest_panel(
    axR_c, metrics_c, mlabels_c, ai_models_list,
    novelty_local_density_df,
    novelty_local_density_tests_df if 'novelty_local_density_tests_df' in globals() else None,
    colors,
)
fig_c.suptitle('Normalized Novelty + Effect Sizes', y=1.01, fontsize=14, fontweight='bold')
fig_c.tight_layout()
fig_c.savefig(FIGURES_DIR / 'novelty_analysis_local_density.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved novelty_analysis_element_percentiles.png, novelty_analysis_mean_knn.png, novelty_analysis_local_density.png')




## Step 7: Visualize Proposals in Literature Embedding Space

Visualize proposals and literature in shared 2D embedding spaces.Its fits **UMAP on literature only** and projects proposals into that space.

Plots are now shown as **single full-canvas views** (not left/right midpoint splits) to avoid implying an artificial boundary in the embedding.
The publication-year view reuses the UMAP coordinates from the UMAP cell directly below.


In [ ]:

import umap

print("="*85)
print("VISUALIZING SECTION-1 PROPOSALS IN LITERATURE EMBEDDING SPACE (UMAP, REVIEW-AWARE)")
print("="*85)

# Literature-space comparisons use Section-1/abstract-only proposal embeddings.
# This keeps the proposal representation comparable to literature abstracts and
# aligned with the novelty precomputation (`X_prop_lit @ X_lit.T`).
if 'human_literature_embeddings' not in globals() or 'ai_literature_embeddings' not in globals():
    raise RuntimeError('Section-1 proposal embeddings are unavailable. Run Step 2 first.')
if 'proposal_meta' not in globals():
    raise RuntimeError('proposal_meta is unavailable. Run the proposal metadata setup cells first.')

n_human_abs = len(human_literature_embeddings)
n_ai_abs = len(ai_literature_embeddings)
n_prop_abs = n_human_abs + n_ai_abs
if len(proposal_meta) != n_prop_abs:
    raise RuntimeError(
        f'Section-1 proposal embeddings ({n_prop_abs}) do not align with proposal_meta ({len(proposal_meta)}).'
    )

_lit_meta_cols = [
    c for c in [
        'proposal_uid', 'title', 'group_binary', 'group_model', 'viz_marker_size',
        'is_top5_ranked', 'funding', 'viz_rank', 'ranking', 'ranking_AI_reviews'
    ]
    if c in proposal_meta.columns
]
proposal_meta_lit = proposal_meta[_lit_meta_cols].copy().reset_index(drop=True)
proposal_meta_lit['group_model'] = proposal_meta_lit['group_model'].map(lambda x: MODEL_NAME_MAP.get(x, x))
proposal_meta_lit['viz_marker_size'] = pd.to_numeric(
    proposal_meta_lit.get('viz_marker_size', pd.Series([145.0] * len(proposal_meta_lit))), errors='coerce'
).fillna(145.0)
proposal_meta_lit['is_top5_ranked'] = proposal_meta_lit.get('is_top5_ranked', False)
proposal_meta_lit['funding'] = pd.to_numeric(
    proposal_meta_lit.get('funding', pd.Series([np.nan] * len(proposal_meta_lit))), errors='coerce'
)

labels_prop   = proposal_meta_lit['group_model'].to_numpy()
marker_sizes  = pd.to_numeric(proposal_meta_lit['viz_marker_size'], errors='coerce').fillna(145.0).to_numpy()
top5_mask     = proposal_meta_lit['is_top5_ranked'].fillna(False).to_numpy(dtype=bool)
funding       = pd.to_numeric(proposal_meta_lit['funding'], errors='coerce').to_numpy()
human_mask    = proposal_meta_lit['group_binary'].eq('Human').to_numpy()
ai_models_nov = sorted(proposal_meta_lit.loc[proposal_meta_lit['group_binary']=='AI', 'group_model'].dropna().unique().tolist())

print('Literature-space proposal counts by group_model:')
print(proposal_meta_lit['group_model'].value_counts().to_string())

proposals_section1_umap = np.vstack([human_literature_embeddings, ai_literature_embeddings]).astype(np.float32)
proposals_section1_umap = proposals_section1_umap / np.clip(
    np.linalg.norm(proposals_section1_umap, axis=1, keepdims=True), 1e-12, None)

_lit_umap2d_path        = LITERATURE_EMBEDDINGS_FILE.parent / 'lit_umap2d.npy'
_lit_reducer_path       = LITERATURE_EMBEDDINGS_FILE.parent / 'lit_umap_reducer.pkl'
_prop_section1_cache    = LITERATURE_EMBEDDINGS_FILE.parent / 'proposals_section1_lit_umap2d.npy'

if 'literature_2d_umap' in globals() and len(literature_2d_umap) > 0:
    print(f"✓ Reusing literature_2d_umap already in globals  shape={literature_2d_umap.shape}")
elif _lit_umap2d_path.exists():
    literature_2d_umap = np.load(_lit_umap2d_path)
    print(f"✓ Loaded literature 2D UMAP from {_lit_umap2d_path}  shape={literature_2d_umap.shape}")
else:
    raise RuntimeError(
        f"Literature UMAP 2D cache not found at {_lit_umap2d_path}. "
        "Run prepare_data_for_analysis.ipynb Section 13 first."
    )

# Use an explicit Section-1 cache/variable. Do not reuse a generic `proposals_2d_umap`
# because older notebook runs may have stored full-proposal coordinates there.
if 'prop_section1_2d_35' in globals() and len(prop_section1_2d_35) == len(proposal_meta_lit):
    proposals_2d_umap = np.asarray(prop_section1_2d_35)
    print(f"✓ Reusing prop_section1_2d_35 from Analysis 3.5  shape={proposals_2d_umap.shape}")
elif 'proposals_section1_2d_umap' in globals() and len(proposals_section1_2d_umap) == len(proposal_meta_lit):
    proposals_2d_umap = np.asarray(proposals_section1_2d_umap)
    print(f"✓ Reusing proposals_section1_2d_umap already in globals  shape={proposals_2d_umap.shape}")
elif _prop_section1_cache.exists():
    proposals_2d_umap = np.load(_prop_section1_cache)
    print(f"✓ Loaded cached Section-1 proposal projections from {_prop_section1_cache}  shape={proposals_2d_umap.shape}")
elif 'lit_reducer_35' in globals():
    proposals_2d_umap = lit_reducer_35.transform(proposals_section1_umap)
    np.save(_prop_section1_cache, proposals_2d_umap)
    print(f"✓ Projected Section-1 proposals with lit_reducer_35 and cached → {_prop_section1_cache}")
elif _lit_reducer_path.exists():
    print(f"Loading literature UMAP reducer to project Section-1 proposals...")
    with open(_lit_reducer_path, 'rb') as f:
        reducer_nov_umap = pickle.load(f)
    proposals_2d_umap = reducer_nov_umap.transform(proposals_section1_umap)
    np.save(_prop_section1_cache, proposals_2d_umap)
    print(f"✓ Projected Section-1 proposals and cached → {_prop_section1_cache}")
else:
    raise RuntimeError(
        f"Fixed literature UMAP reducer not found at {_lit_reducer_path}. "
        "Cannot project Section-1 proposals into the fixed literature map. "
        "Rerun prepare_data_for_analysis.ipynb Section 13 so lit_umap_reducer.pkl is saved, "
        "or run Analysis 3.5 first in this kernel."
    )

proposals_section1_2d_umap = proposals_2d_umap

if len(proposals_2d_umap) != len(proposal_meta_lit):
    raise RuntimeError(
        f'Proposal count mismatch: UMAP ({len(proposals_2d_umap)}) vs proposal_meta_lit ({len(proposal_meta_lit)}).')

print(f"SUCCESS: Literature: {len(literature_2d_umap)} points | Proposals: {len(proposals_2d_umap)} points")

fig, ax = plt.subplots(1, 1, figsize=(13, 10))
ax.scatter(
    literature_2d_umap[:, 0], literature_2d_umap[:, 1],
    c='#AAAAAA', s=24, alpha=0.35, linewidths=0,
    label=f'Literature ({len(literature_2d_umap)} articles)', zorder=1,
)

def _scatter_with_top5_outline(mask, color, label, alpha=0.75, zorder=3):
    mask = np.asarray(mask, dtype=bool)
    if not mask.any():
        return
    m_other = mask & (~top5_mask)
    m_top   = mask & top5_mask
    if m_other.any():
        pts = proposals_2d_umap[m_other]
        ax.scatter(pts[:,0], pts[:,1], c=color, label=label, s=marker_sizes[m_other],
                   alpha=alpha, edgecolors='none', linewidth=0, zorder=zorder)
    if m_top.any():
        pts = proposals_2d_umap[m_top]
        ax.scatter(pts[:,0], pts[:,1], c=color,
                   label=(label if not m_other.any() else None),
                   s=marker_sizes[m_top], alpha=alpha,
                   edgecolors='black', linewidth=1.8, zorder=zorder+0.1)

for model in ai_models_nov:
    _scatter_with_top5_outline(labels_prop == model, colors.get(model, '#808080'), model, alpha=0.78, zorder=4)

human_funded  = human_mask & np.isfinite(funding) & (funding == 1)
human_nonfund = human_mask & (~human_funded)
_scatter_with_top5_outline(human_nonfund, '#F08080', 'Human (funding=0/NA)', alpha=0.90, zorder=6)
_scatter_with_top5_outline(human_funded,  '#8B0000', 'Human (funding=1)',    alpha=0.95, zorder=7)

if 'outliers' in globals() and len(outliers) == len(proposals_2d_umap):
    outlier_coords_umap = proposals_2d_umap[outliers]
    ax.scatter(outlier_coords_umap[:, 0], outlier_coords_umap[:, 1],
               s=np.maximum(marker_sizes[outliers] + 160, 320),
               facecolors='none', edgecolors='magenta', linewidth=2.2, alpha=0.75,
               label=f'Outliers (n={int(outliers.sum())})', zorder=9)

ax.set_xlabel('Literature UMAP Dim 1', fontsize=12, fontweight='bold')
ax.set_ylabel('Literature UMAP Dim 2', fontsize=12, fontweight='bold')
ax.set_title('Section-1 Proposals in Literature Embedding Space (UMAP)\nBlack outline = top-5 rank; dark red Human = funding=1',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='best', fontsize=8.8, framealpha=0.9, edgecolor='black')

plt.tight_layout()
out_path_umap = FIGURES_DIR / 'proposals_in_literature_space_umap.png'
plt.savefig(out_path_umap, dpi=300, bbox_inches='tight')
plt.show()
print(f"\nSUCCESS: Figure saved to: {out_path_umap}")
print("="*85)


### Step 7B: Literature-Space Outliers and High-Novelty Flags

Flags top 10% proposals for mean_knn_10, element_novel_0, and novelty_z, then runs group prevalence tests.

In [ ]:

print('='*85)
print('STEP 7B: LITERATURE-SPACE OUTLIERS AND HIGH-NOVELTY FLAGS')
print('='*85)

lit_outlier_flags_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
lit_outlier_flags_df['mean_knn_10'] = mean_knn_novelty_df['mean_knn_10'].to_numpy()
lit_outlier_flags_df['element_novel_0'] = element_novelty_df['element_novel_0'].to_numpy()
lit_outlier_flags_df['novelty_z'] = novelty_local_density_df['novelty_z'].to_numpy()

flag_mean10, thr_mean10 = flag_top_percentile(lit_outlier_flags_df['mean_knn_10'].to_numpy(), pct=90, strict=True)
flag_element0, thr_element0 = flag_top_percentile(lit_outlier_flags_df['element_novel_0'].to_numpy(), pct=90, strict=True)
valid_z = lit_outlier_flags_df['novelty_z'].to_numpy()
mask_valid_z = np.isfinite(valid_z)
thr_z = np.percentile(valid_z[mask_valid_z], 90) if mask_valid_z.any() else np.nan
flag_z = np.zeros(len(valid_z), dtype=bool)
flag_z[mask_valid_z] = valid_z[mask_valid_z] > thr_z

lit_outlier_flags_df['is_lit_outlier_mean10'] = flag_mean10
lit_outlier_flags_df['is_lit_outlier_element0'] = flag_element0
lit_outlier_flags_df['is_lit_outlier_z'] = flag_z

# Backward compatibility aliases used in downstream/legacy cells
mean_lit_knn_dist = lit_outlier_flags_df['mean_knn_10'].to_numpy()
lit_mknn_threshold = thr_mean10
is_literature_outlier = flag_mean10

# prevalence tests per flag family
test_frames = []
for flag_col in ['is_lit_outlier_mean10', 'is_lit_outlier_element0', 'is_lit_outlier_z']:
    t = fisher_group_prevalence_tests(lit_outlier_flags_df[flag_col].to_numpy(), lit_outlier_flags_df['group_model'].to_numpy(), reference_group='Human')
    if len(t):
        t['flag_name'] = flag_col
        t = apply_multiple_testing(t, p_cols=('p_value',), method='holm')
    test_frames.append(t)

lit_outlier_prevalence_tests_df = pd.concat(test_frames, ignore_index=True) if test_frames else pd.DataFrame()

# Required exports
lit_outlier_flags_df.to_csv(TABLES_DIR / 'literature_space_outliers_mean_knn_k10.csv', index=False)
lit_outlier_flags_df.to_csv(TABLES_DIR / 'literature_space_outliers_element0.csv', index=False)
lit_outlier_flags_df.to_csv(TABLES_DIR / 'literature_space_outliers_z.csv', index=False)
lit_outlier_prevalence_tests_df.to_csv(TABLES_DIR / 'literature_space_outlier_prevalence_tests.csv', index=False)

print('Saved literature_space_outliers_mean_knn_k10.csv, literature_space_outliers_element0.csv, literature_space_outliers_z.csv, literature_space_outlier_prevalence_tests.csv')



# Outlier-comparison UMAP overlay (proposal-space outliers vs literature-space mean-k10 outliers)
if 'proposals_2d_umap' in globals() and 'lit_outlier_flags_df' in globals():
    print('\n' + '='*85)
    print('STEP 7B VIS: UMAP outlier comparison (proposal-space vs literature-space k10)')
    print('='*85)

    # Rebuild plotting metadata from canonical proposal_meta every time, so this cell
    # does not inherit an older generic Human/AI proposal_meta_lit from the kernel.
    if 'proposal_meta' in globals() and len(proposal_meta) == len(proposals_2d_umap):
        _meta_cols_7b = [
            c for c in [
                'proposal_uid', 'title', 'group_binary', 'group_model', 'viz_marker_size',
                'is_top5_ranked', 'funding', 'viz_rank', 'ranking', 'ranking_AI_reviews'
            ]
            if c in proposal_meta.columns
        ]
        proposal_meta_lit = proposal_meta[_meta_cols_7b].copy().reset_index(drop=True)
    elif 'proposal_meta_lit' not in globals() or len(proposal_meta_lit) != len(proposals_2d_umap):
        raise RuntimeError('proposal_meta_lit is unavailable or misaligned; run the Section-1 Step 7 UMAP cell first.')

    _name_map_7b = MODEL_NAME_MAP if 'MODEL_NAME_MAP' in globals() else {}
    proposal_meta_lit['group_model'] = proposal_meta_lit['group_model'].map(lambda x: _name_map_7b.get(x, x))
    proposal_meta_lit['viz_marker_size'] = pd.to_numeric(
        proposal_meta_lit.get('viz_marker_size', pd.Series([145.0] * len(proposal_meta_lit))), errors='coerce'
    ).fillna(145.0)
    proposal_meta_lit['is_top5_ranked'] = proposal_meta_lit.get('is_top5_ranked', False)
    proposal_meta_lit['funding'] = pd.to_numeric(
        proposal_meta_lit.get('funding', pd.Series([np.nan] * len(proposal_meta_lit))), errors='coerce'
    )

    print('Outlier-comparison proposal counts by group_model:')
    print(proposal_meta_lit['group_model'].value_counts().to_string())

    # Align lit-space outlier flags to literature-space embedding order. Prefer
    # proposal_uid; fall back to normalized title/group only for legacy metadata.
    src = lit_outlier_flags_df.copy()
    if 'proposal_uid' in proposal_meta_lit.columns and 'proposal_uid' in src.columns:
        lit_flags = proposal_meta_lit[['proposal_uid']].copy()
        keep = ['proposal_uid', 'is_lit_outlier_mean10']
        lit_flags = lit_flags.merge(src[keep].drop_duplicates('proposal_uid'), on='proposal_uid', how='left')
    else:
        lit_flags = proposal_meta_lit[['title','group_binary','group_model']].copy()
        lit_flags['title_norm'] = lit_flags['title'].astype(str).str.strip().str.lower()
        src['title_norm'] = src['title'].astype(str).str.strip().str.lower()
        keep = ['title_norm','group_binary','group_model','is_lit_outlier_mean10']
        lit_flags = lit_flags.merge(src[keep].drop_duplicates(subset=['title_norm','group_binary','group_model']),
                                    on=['title_norm','group_binary','group_model'], how='left')
    lit_out_mask = lit_flags['is_lit_outlier_mean10'].fillna(False).to_numpy(dtype=bool)

    labels_7b = proposal_meta_lit['group_model'].to_numpy()
    top5_7b = proposal_meta_lit['is_top5_ranked'].fillna(False).to_numpy(dtype=bool)
    marker_sizes_7b = pd.to_numeric(proposal_meta_lit['viz_marker_size'], errors='coerce').fillna(145.0).to_numpy()

    fig, ax = plt.subplots(1, 1, figsize=(13, 10))
    ax.scatter(literature_2d_umap[:,0], literature_2d_umap[:,1], c='#AAAAAA', s=24, alpha=0.35, linewidths=0,
               label=f'Literature ({len(literature_embeddings)} articles)', zorder=1)

    # Draw proposals with the same metadata encoding as the main literature-space UMAP:
    # group color, funded-human dark red, and black outline only for top-ranked proposals.
    def _draw(mask, color, label, alpha=0.78, zorder=4):
        mask = np.asarray(mask, dtype=bool)
        if not mask.any():
            return
        m_other = mask & (~top5_7b)
        m_top = mask & top5_7b
        if m_other.any():
            pts = proposals_2d_umap[m_other]
            ax.scatter(pts[:,0], pts[:,1], c=color, s=marker_sizes_7b[m_other], alpha=alpha,
                       edgecolors='none', linewidth=0, label=label, zorder=zorder)
        if m_top.any():
            pts = proposals_2d_umap[m_top]
            ax.scatter(pts[:,0], pts[:,1], c=color, s=marker_sizes_7b[m_top], alpha=alpha,
                       edgecolors='black', linewidth=1.8,
                       label=(label if not m_other.any() else None), zorder=zorder + 0.1)

    for model in sorted(proposal_meta_lit.loc[proposal_meta_lit['group_binary']=='AI','group_model'].dropna().unique().tolist()):
        _draw(labels_7b == model, colors.get(model, '#808080'), model, alpha=0.78, zorder=4)

    hmask = proposal_meta_lit['group_binary'].eq('Human').to_numpy()
    hfund = pd.to_numeric(proposal_meta_lit['funding'], errors='coerce').to_numpy()
    _draw(hmask & (~(np.isfinite(hfund) & (hfund==1))), '#F08080', 'Human (funding=0/NA)', alpha=0.90, zorder=6)
    _draw(hmask & (np.isfinite(hfund) & (hfund==1)), '#8B0000', 'Human (funding=1)', alpha=0.95, zorder=7)

    if 'outliers' in globals() and len(outliers)==len(proposals_2d_umap):
        pts = proposals_2d_umap[outliers]
        ax.scatter(pts[:,0], pts[:,1], s=np.maximum(marker_sizes_7b[outliers] + 160, 320),
                   facecolors='none', edgecolors='magenta', linewidth=2.2,
                   alpha=0.75, label=f'Proposal-space outliers (1-NN; n={int(outliers.sum())})', zorder=9)

    if len(lit_out_mask)==len(proposals_2d_umap):
        pts = proposals_2d_umap[lit_out_mask]
        ax.scatter(pts[:,0], pts[:,1], s=np.maximum(marker_sizes_7b[lit_out_mask] + 220, 380),
                   facecolors='none', edgecolors='cyan', linewidth=2.0,
                   alpha=0.75, label=f'Literature-space outliers (mean-10NN; n={int(lit_out_mask.sum())})', zorder=10)

    ax.set_xlabel('Literature UMAP Dim 1', fontsize=12, fontweight='bold')
    ax.set_ylabel('Literature UMAP Dim 2', fontsize=12, fontweight='bold')
    ax.set_title('Section-1 Proposals in Literature Space (UMAP) | Outlier Comparison\nBlack outline = top-5 rank; dark red Human = funding=1',
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='best', fontsize=8.4, framealpha=0.9, edgecolor='black')

    out_cmp = FIGURES_DIR / 'proposals_in_literature_space_umap_outliers_comparison_k10.png'
    plt.tight_layout()
    plt.savefig(out_cmp, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'SUCCESS: Figure saved to: {out_cmp}')


## Additional Analysis: Nearest Neighbors in Literature for Every Proposal

For **each** proposal (human and AI), retrieve the 3 nearest literature abstracts by cosine distance and save the result as a **CSV table** (`nearest_literature_neighbors_top3.csv`).
Each row corresponds to one proposal-neighbor pair and includes proposal metadata, neighbor rank, literature metadata, and distance.

In [ ]:

print('='*85)
print('ADDITIONAL ANALYSIS: TOP-3 LITERATURE NEIGHBORS FOR EACH PROPOSAL')
print('='*85)

rows = []
for i in range(len(proposal_meta)):
    top3_idx = D_pl_sorted_idx[i, :3]
    top3_dist = D_pl_sorted_dist[i, :3]
    for rnk, (lid, dist) in enumerate(zip(top3_idx, top3_dist), start=1):
        art = articles[int(lid)]
        rows.append({
            'proposal_uid': proposal_meta.loc[i, 'proposal_uid'],
            'proposal_title': proposal_meta.loc[i, 'title'],
            'proposal_group_model': proposal_meta.loc[i, 'group_model'],
            'neighbor_rank': rnk,
            'lit_doc_id': art.get('pmid', lid),
            'lit_title': art.get('title', ''),
            'lit_year': str(art.get('publication_date', ''))[:4],
            'lit_distance': float(dist),
            'lit_query_category': art.get('query_category', ''),
            'lit_abstract_preview': str(art.get('abstract', ''))[:240],
        })

nearest_literature_neighbors_df = pd.DataFrame(rows)
nearest_literature_neighbors_df.to_csv(TABLES_DIR / 'nearest_literature_neighbors_top3.csv', index=False)
print('Saved nearest_literature_neighbors_top3.csv')

# print required top tables
m = mean_knn_novelty_df[['proposal_uid','title','group_model','mean_knn_10']].copy()
e0 = element_novelty_df[['proposal_uid','title','group_model','element_novel_0']].copy()
print('\nTop 10 largest mean_knn_10')
print(m.sort_values('mean_knn_10', ascending=False).head(10).to_string(index=False))
print('\nTop 10 smallest mean_knn_10')
print(m.sort_values('mean_knn_10', ascending=True).head(10).to_string(index=False))
print('\nTop 10 largest element_novel_0')
print(e0.sort_values('element_novel_0', ascending=False).head(10).to_string(index=False))


In [ ]:

# Unified novelty export
novelty_scores_from_literature_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()

for col in ['element_novel_0','element_novel_1','element_novel_5','element_novel_10']:
    novelty_scores_from_literature_df[col] = element_novelty_df[col].to_numpy()
for col in ['mean_knn_5','mean_knn_10','mean_knn_20','mean_knn_50']:
    novelty_scores_from_literature_df[col] = mean_knn_novelty_df[col].to_numpy()
for col in ['novelty_ratio','novelty_z']:
    novelty_scores_from_literature_df[col] = novelty_local_density_df[col].to_numpy()
for col in ['is_lit_outlier_mean10','is_lit_outlier_element0','is_lit_outlier_z']:
    novelty_scores_from_literature_df[col] = lit_outlier_flags_df[col].to_numpy()

novelty_scores_from_literature_df.to_csv(TABLES_DIR / 'novelty_scores_from_literature.csv', index=False)
print('Saved novelty_scores_from_literature.csv')


## Analysis 3.5: Literature-Anchored UMAP with Embedding-Native Topic Regions 

[UPDATE] Primary visualization: color the fixed literature UMAP by BERTopic embedding-region labels, not LDA. LDA-colored maps are retained only as supplementary lexical diagnostics.


In [ ]:
print('='*85)
print('ANALYSIS 3.5: LITERATURE-ANCHORED UMAP - BERTOPIC EMBEDDING REGIONS')
print('='*85)

_lit_umap2d_path_35 = LITERATURE_EMBEDDINGS_FILE.parent / 'lit_umap2d.npy'
_lit_umap_reducer_path_35 = LITERATURE_EMBEDDINGS_FILE.parent / 'lit_umap_reducer.pkl'
_prop_section1_cache_35 = LITERATURE_EMBEDDINGS_FILE.parent / 'proposals_section1_lit_umap2d.npy'
_bt_assign_path_35 = PREPARED_DIR / 'lit_bertopic_assignments.csv'
_bt_info_path_35 = PREPARED_DIR / 'lit_bertopic_topic_info.csv'
_lit_topic_path_35 = PREPARED_DIR / 'lit_topic_assignments.csv'
_lda_bundle_path_35 = LITERATURE_EMBEDDINGS_FILE.parent / 'lit_lda_model.pkl'

_required_35 = [_lit_umap2d_path_35, _bt_assign_path_35, _bt_info_path_35]
_missing_35 = [p for p in _required_35 if not p.exists()]
_missing_projection_35 = (not _prop_section1_cache_35.exists()) and (not _lit_umap_reducer_path_35.exists())
if _missing_35 or _missing_projection_35:
    if _missing_projection_35:
        _missing_35.append(_lit_umap_reducer_path_35)
    print(f'WARNING: Missing BERTopic literature-region outputs: {[str(p) for p in _missing_35]}')
    print('Run prepare_data_for_analysis.ipynb Sections 12 and 13 first. Skipping primary BERTopic Analysis 3.5.')
else:
    lit_2d_35 = literature_2d_umap if 'literature_2d_umap' in globals() else np.load(_lit_umap2d_path_35)
    # Proposal projection into the literature map uses Section-1/abstract-only embeddings
    # so proposal-to-literature comparisons match literature abstracts. If the projection
    # cache already exists, the reducer is not required for rendering.
    if _prop_section1_cache_35.exists():
        prop_section1_2d_35 = np.load(_prop_section1_cache_35)
        print(f'Loaded cached Section-1 proposal projection: {_prop_section1_cache_35}')
    else:
        with open(_lit_umap_reducer_path_35, 'rb') as f:
            lit_reducer_35 = pickle.load(f)
        X_prop_lit_35 = np.vstack([human_literature_embeddings, ai_literature_embeddings]).astype(np.float32)
        X_prop_lit_35 = X_prop_lit_35 / np.clip(np.linalg.norm(X_prop_lit_35, axis=1, keepdims=True), 1e-12, None)
        prop_section1_2d_35 = lit_reducer_35.transform(X_prop_lit_35)
        np.save(_prop_section1_cache_35, prop_section1_2d_35)
        print(f'Saved Section-1 proposal projection: {_prop_section1_cache_35}')
    proposals_section1_2d_umap = prop_section1_2d_35
    proposals_2d_umap = prop_section1_2d_35
    bt_df_35 = pd.read_csv(_bt_assign_path_35)
    bt_info_35 = pd.read_csv(_bt_info_path_35)

    # Align BERTopic labels to the literature embedding/UMAP row order.
    if 'article_idx' in bt_df_35.columns:
        bt_df_35 = (
            bt_df_35.set_index('article_idx')
            .reindex(np.arange(len(lit_2d_35)))
            .reset_index()
        )
        if bt_df_35['bertopic_topic'].isna().any():
            missing_n = int(bt_df_35['bertopic_topic'].isna().sum())
            raise RuntimeError(f'BERTopic assignments are missing {missing_n} article_idx rows needed to align with lit_umap2d.npy.')

    bt_topic_arr_35 = bt_df_35['bertopic_topic'].to_numpy(dtype=int)

    if len(lit_2d_35) != len(bt_topic_arr_35):
        raise RuntimeError(f'Literature UMAP rows ({len(lit_2d_35)}) do not match BERTopic assignments ({len(bt_topic_arr_35)}).')

    label_strategy_35 = str(bt_info_35.get('display_label_strategy', pd.Series(['unknown'])).dropna().iloc[0]) if len(bt_info_35) else 'unknown'
    print(f'BERTopic display-label strategy loaded from prepare_data: {label_strategy_35}')
    if label_strategy_35 != 'contrastive_phrase_v4':
        print('WARNING: BERTopic topic-info labels do not appear to be the latest contrastive_phrase_v4 labels. Rerun prepare_data Section 12.')
    if len(prop_section1_2d_35) != len(proposal_meta):
        raise RuntimeError(f'Section-1 proposal UMAP rows ({len(prop_section1_2d_35)}) do not match proposal_meta ({len(proposal_meta)}).')

    import textwrap
    import matplotlib.patheffects as path_effects
    from matplotlib.lines import Line2D

    bt_labels_35 = dict(zip(bt_info_35['Topic'].astype(int), bt_info_35.get('display_label', bt_info_35.get('Name', bt_info_35['Topic'].astype(str)))))
    bt_counts_35 = bt_df_35['bertopic_topic'].value_counts().to_dict()
    region_ids_35 = sorted([int(t) for t in np.unique(bt_topic_arr_35) if int(t) >= 0])
    topic_rank_35 = sorted(region_ids_35, key=lambda t: bt_counts_35.get(t, 0), reverse=True)
    label_region_ids_35 = set(topic_rank_35[:25])

    cmap_35 = plt.cm.get_cmap('tab20', max(20, len(region_ids_35)))
    region_color_35 = {t: cmap_35(i % 20) for i, t in enumerate(region_ids_35)}
    markers_35 = {'Human': 'o', 'Claude': 's', 'Gemini': '^', 'GPT-5.2': 'D'}
    MODEL_NAMES_35 = ['Human', 'Claude', 'Gemini', 'GPT-5.2']

    def _style_literature_umap_axes_35(ax):
        x_pad = 0.03 * (float(np.nanmax(lit_2d_35[:, 0])) - float(np.nanmin(lit_2d_35[:, 0])))
        y_pad = 0.03 * (float(np.nanmax(lit_2d_35[:, 1])) - float(np.nanmin(lit_2d_35[:, 1])))
        ax.set_xlim(float(np.nanmin(lit_2d_35[:, 0])) - x_pad, float(np.nanmax(lit_2d_35[:, 0])) + x_pad)
        ax.set_ylim(float(np.nanmin(lit_2d_35[:, 1])) - y_pad, float(np.nanmax(lit_2d_35[:, 1])) + y_pad)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)

    def _draw_lit_background_35(ax, alpha=0.42, s=4, label_top_regions=True, label_fontsize=8, show_region_legend=False):
        # Match the literature-only prepare_data visualization so proposal overlays sit on the same visible topic map.
        outlier_mask = bt_topic_arr_35 == -1
        if outlier_mask.any():
            ax.scatter(
                lit_2d_35[outlier_mask, 0],
                lit_2d_35[outlier_mask, 1],
                c='#d9d9d9',
                s=max(2, s - 1),
                alpha=0.18,
                linewidths=0,
                rasterized=True,
                zorder=1,
            )
        for t in region_ids_35:
            mask_t = bt_topic_arr_35 == t
            if mask_t.any():
                ax.scatter(
                    lit_2d_35[mask_t, 0],
                    lit_2d_35[mask_t, 1],
                    c=[region_color_35[t]],
                    s=s,
                    alpha=alpha,
                    linewidths=0,
                    rasterized=True,
                    zorder=2,
                )
        if label_top_regions:
            for t in topic_rank_35:
                if t not in label_region_ids_35:
                    continue
                mask_t = bt_topic_arr_35 == t
                if mask_t.sum() < 20:
                    continue
                centroid_x, centroid_y = np.median(lit_2d_35[mask_t], axis=0)
                label = str(bt_labels_35.get(t, f'Region {t}'))
                label = ', '.join(label.split(',')[:4]).strip()
                label_text = textwrap.fill(f'{t}: {label}', width=34)
                txt = ax.text(
                    centroid_x,
                    centroid_y,
                    label_text,
                    fontsize=label_fontsize,
                    fontweight='bold',
                    ha='center',
                    va='center',
                    color='black',
                    bbox={
                        'boxstyle': 'round,pad=0.25',
                        'facecolor': 'white',
                        'edgecolor': region_color_35[t],
                        'linewidth': 1.2,
                        'alpha': 0.86,
                    },
                    zorder=6,
                )
                txt.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground='white')])
        if show_region_legend:
            region_handles = [
                Line2D(
                    [0], [0],
                    marker='o',
                    color='w',
                    markerfacecolor=region_color_35[t],
                    markersize=7,
                    label=f'{t}: n={int(bt_counts_35.get(t, 0)):,}',
                )
                for t in region_ids_35
            ]
            region_legend = ax.legend(
                handles=region_handles,
                title='BERTopic regions',
                loc='center left',
                bbox_to_anchor=(1.01, 0.5),
                frameon=False,
                fontsize=8,
                title_fontsize=9,
            )
            ax.add_artist(region_legend)
        _style_literature_umap_axes_35(ax)

    def _draw_proposals_35(ax, group_filter=None, legend=True, legend_loc='upper right'):
        proposal_handles = []
        for gm in MODEL_NAMES_35:
            gm_mask = proposal_meta['group_model'].eq(gm).values
            if group_filter is not None:
                gm_mask &= proposal_meta['group_model'].eq(group_filter).values
            if not gm_mask.any():
                continue
            pts = prop_section1_2d_35[gm_mask]
            artist = ax.scatter(
                pts[:, 0],
                pts[:, 1],
                c=colors.get(gm, '#808080'),
                s=86,
                marker=markers_35.get(gm, 'o'),
                alpha=0.90,
                edgecolors='black',
                linewidth=0.70,
                label=f'{gm} (n={int(gm_mask.sum())})',
                zorder=10,
            )
            proposal_handles.append(artist)
        if legend and proposal_handles:
            ax.legend(handles=proposal_handles, loc=legend_loc, fontsize=8.5, framealpha=0.92, title='Proposals')

    fig35, ax35 = plt.subplots(1, 1, figsize=(14, 10))
    _draw_lit_background_35(ax35, alpha=0.42, s=4, label_top_regions=True, label_fontsize=8, show_region_legend=True)
    _draw_proposals_35(ax35, legend=True, legend_loc='upper right')
    ax35.set_title('Proposals in Fixed Literature Map - BERTopic Embedding Regions\nDots = literature colored by BioLinkBERT-derived region; markers = proposals', fontsize=13, fontweight='bold')
    ax35.set_xlabel('Literature UMAP Dim 1')
    ax35.set_ylabel('Literature UMAP Dim 2')
    plt.tight_layout()
    out_35 = FIGURES_DIR / 'literature_umap_with_bertopic_regions.png'
    fig35.savefig(out_35, dpi=220, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_35}')

    # Split zoom: same proposal markers, but only show local BERTopic regions that proposals occupy.
    def _split_proposal_sides_35():
        prop_x = prop_section1_2d_35[:, 0]
        if len(prop_x) < 2:
            split_x = float(np.median(prop_x)) if len(prop_x) else 0.0
        else:
            sorted_x = np.sort(prop_x)
            gaps = np.diff(sorted_x)
            split_x = float((sorted_x[int(np.argmax(gaps))] + sorted_x[int(np.argmax(gaps)) + 1]) / 2.0) if len(gaps) else float(np.median(prop_x))
        left_mask = prop_x <= split_x
        right_mask = prop_x > split_x
        if left_mask.sum() == 0 or right_mask.sum() == 0:
            split_x = float(np.median(prop_x))
            left_mask = prop_x <= split_x
            right_mask = prop_x > split_x
        return split_x, left_mask, right_mask

    def _zoom_limits_around_proposals_35(prop_mask, min_x_pad=0.95, min_y_pad=0.85):
        pts = prop_section1_2d_35[prop_mask]
        if len(pts) == 0:
            return None
        x_min, y_min = np.min(pts, axis=0)
        x_max, y_max = np.max(pts, axis=0)
        lit_x_span = float(np.ptp(lit_2d_35[:, 0])) or 1.0
        lit_y_span = float(np.ptp(lit_2d_35[:, 1])) or 1.0
        x_pad = max(min_x_pad, 0.55 * max(float(x_max - x_min), 1e-6), 0.035 * lit_x_span)
        y_pad = max(min_y_pad, 0.65 * max(float(y_max - y_min), 1e-6), 0.045 * lit_y_span)
        return (float(x_min - x_pad), float(x_max + x_pad)), (float(y_min - y_pad), float(y_max + y_pad))

    def _proposal_local_topic_ids_35(prop_mask, k_neighbors=35, min_topic_votes=1):
        """Topic ids for literature regions actually occupied by proposals in the pane, using 2D local neighbors for visual alignment."""
        pts = prop_section1_2d_35[prop_mask]
        if len(pts) == 0:
            return []
        k_neighbors = max(1, min(int(k_neighbors), len(lit_2d_35)))
        topic_votes = []
        for pt in pts:
            d2 = np.sum((lit_2d_35 - pt) ** 2, axis=1)
            nn = np.argpartition(d2, k_neighbors - 1)[:k_neighbors]
            nn_topics = bt_topic_arr_35[nn]
            nn_topics = nn_topics[nn_topics >= 0]
            if len(nn_topics) == 0:
                continue
            counts = pd.Series(nn_topics).value_counts()
            top_count = int(counts.iloc[0])
            for topic_id, count in counts.items():
                if int(count) >= max(min_topic_votes, int(0.25 * top_count)):
                    topic_votes.append(int(topic_id))
        if not topic_votes:
            return []
        topic_order = pd.Series(topic_votes).value_counts().index.astype(int).tolist()
        return [int(t) for t in topic_order]

    def _draw_lit_background_zoom_35(ax, xlim, ylim, prop_topic_ids, alpha=0.78, s=8, label_fontsize=7.2):
        visible_mask = (
            (lit_2d_35[:, 0] >= xlim[0]) & (lit_2d_35[:, 0] <= xlim[1]) &
            (lit_2d_35[:, 1] >= ylim[0]) & (lit_2d_35[:, 1] <= ylim[1])
        )
        prop_topic_ids = [int(t) for t in prop_topic_ids if int(t) >= 0]
        if not prop_topic_ids:
            local_counts = pd.Series(bt_topic_arr_35[visible_mask & (bt_topic_arr_35 >= 0)]).value_counts()
            prop_topic_ids = [int(t) for t in local_counts.head(4).index]
        for t in prop_topic_ids:
            mask_t = visible_mask & (bt_topic_arr_35 == int(t))
            if mask_t.any():
                ax.scatter(
                    lit_2d_35[mask_t, 0], lit_2d_35[mask_t, 1],
                    c=[region_color_35.get(int(t), '#808080')],
                    s=s,
                    alpha=alpha,
                    linewidths=0,
                    rasterized=True,
                    zorder=2,
                )
        for t in prop_topic_ids:
            mask_t = visible_mask & (bt_topic_arr_35 == int(t))
            if mask_t.sum() < 15:
                continue
            centroid_x, centroid_y = np.median(lit_2d_35[mask_t], axis=0)
            label = str(bt_labels_35.get(int(t), f'Region {int(t)}'))
            label = ', '.join(label.split(',')[:4]).strip()
            label_text = textwrap.fill(f'{int(t)}: {label}', width=28)
            txt = ax.text(
                centroid_x,
                centroid_y,
                label_text,
                fontsize=label_fontsize,
                fontweight='bold',
                ha='center',
                va='center',
                color='black',
                bbox={
                    'boxstyle': 'round,pad=0.22',
                    'facecolor': 'white',
                    'edgecolor': region_color_35.get(int(t), '#808080'),
                    'linewidth': 1.1,
                    'alpha': 0.86,
                },
                zorder=6,
                clip_on=True,
            )
            txt.set_path_effects([path_effects.withStroke(linewidth=2.2, foreground='white')])
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

    split_x_35, left_prop_mask_35, right_prop_mask_35 = _split_proposal_sides_35()
    zoom_specs_35 = [
        ('Left proposal-bearing literature regions', left_prop_mask_35),
        ('Right proposal-bearing literature regions', right_prop_mask_35),
    ]
    fig35_zoom, axes35_zoom = plt.subplots(1, 2, figsize=(17, 7.2), sharex=False, sharey=False)
    for ax_zoom, (pane_title_35, prop_mask_35) in zip(axes35_zoom, zoom_specs_35):
        zoom_lims_35 = _zoom_limits_around_proposals_35(prop_mask_35)
        prop_topic_ids_35 = _proposal_local_topic_ids_35(prop_mask_35)
        if zoom_lims_35 is not None:
            _draw_lit_background_zoom_35(
                ax_zoom,
                zoom_lims_35[0],
                zoom_lims_35[1],
                prop_topic_ids_35,
                alpha=0.78,
                s=8,
                label_fontsize=7.2,
            )
        _draw_proposals_35(ax_zoom, legend=False)
        if zoom_lims_35 is not None:
            ax_zoom.set_xlim(*zoom_lims_35[0])
            ax_zoom.set_ylim(*zoom_lims_35[1])
        topic_txt_35 = ', '.join([str(t) for t in prop_topic_ids_35]) if prop_topic_ids_35 else 'local'
        ax_zoom.set_title(f'{pane_title_35}\nproposal-region labels shown: {topic_txt_35}; n proposals={int(prop_mask_35.sum())}', fontsize=10.5, fontweight='bold')
        ax_zoom.set_xlabel('Literature UMAP Dim 1')
        ax_zoom.set_ylabel('Literature UMAP Dim 2')
        ax_zoom.grid(True, alpha=0.12, linestyle='--')
        for spine in ax_zoom.spines.values():
            spine.set_visible(True)
            spine.set_alpha(0.25)

    proposal_handles_35, proposal_labels_35 = axes35_zoom[-1].get_legend_handles_labels()
    if proposal_handles_35:
        fig35_zoom.legend(
            proposal_handles_35,
            proposal_labels_35,
            title='Proposals',
            loc='center right',
            bbox_to_anchor=(1.01, 0.5),
            fontsize=9,
            title_fontsize=10,
            framealpha=0.92,
        )
    fig35_zoom.suptitle(
        'Split Zoom: Proposal-Bearing BERTopic Regions in the Fixed Literature Map\nOnly local literature regions occupied by proposals are shown in each pane',
        fontsize=13,
        fontweight='bold',
    )
    plt.tight_layout(rect=[0, 0, 0.93, 0.91])
    out_35_zoom = FIGURES_DIR / 'literature_umap_with_bertopic_regions_split_zoom.png'
    fig35_zoom.savefig(out_35_zoom, dpi=220, bbox_inches='tight')
    plt.show()
    print(f'Saved split zoom: {out_35_zoom}')
    print(f'Proposal x split for zoom panes: {split_x_35:.3f}; left n={int(left_prop_mask_35.sum())}, right n={int(right_prop_mask_35.sum())}')

    fig35b, axes35b = plt.subplots(2, 2, figsize=(17, 14), sharex=True, sharey=True)
    axes35b = axes35b.flatten()
    for ax, gm in zip(axes35b, MODEL_NAMES_35):
        _draw_lit_background_35(ax, alpha=0.24, s=3, label_top_regions=False)
        _draw_proposals_35(ax, group_filter=gm, legend=True, legend_loc='upper right')
        ax.set_title(gm, fontsize=11, fontweight='bold')
        ax.set_xlabel('Literature UMAP Dim 1')
        ax.set_ylabel('Literature UMAP Dim 2')
    fig35b.suptitle('Section-1 Proposal Locations by Author Group over BERTopic Literature Regions', fontsize=13, fontweight='bold')
    plt.tight_layout()
    out_35b = FIGURES_DIR / 'literature_umap_bertopic_by_author_group.png'
    fig35b.savefig(out_35b, dpi=220, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_35b}')


print('Analysis 3.5 complete.')


## Analysis 3.6: Literature Embedding-Region Coverage per Author Group [UPDATE]

[UPDATE] Primary quantification: assign proposals to BERTopic embedding regions using high-dimensional nearest literature neighbors. LDA coverage is retained as a supplementary lexical comparison.


In [ ]:
print('='*85)
print('ANALYSIS 3.6: LITERATURE EMBEDDING-REGION COVERAGE PER AUTHOR GROUP')
print('='*85)

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

_bt_assign_path_36 = PREPARED_DIR / 'lit_bertopic_assignments.csv'
_bt_info_path_36 = PREPARED_DIR / 'lit_bertopic_topic_info.csv'
MODEL_NAMES_36 = ['Human', 'Claude', 'Gemini', 'GPT-5.2']
K_NEIGHBORS_36 = 20

if not _bt_assign_path_36.exists():
    print('WARNING: lit_bertopic_assignments.csv not found. Run prepare_data Section 12. Skipping primary BERTopic region coverage.')
    prop_region_weights = None
else:
    bt_df_36 = pd.read_csv(_bt_assign_path_36)
    bt_info_36 = pd.read_csv(_bt_info_path_36) if _bt_info_path_36.exists() else pd.DataFrame()

    # D_pl_sorted_idx indexes X_lit/literature_embeddings, so align BERTopic labels to that exact row order.
    n_lit_36 = X_lit.shape[0] if 'X_lit' in globals() else len(literature_embeddings)
    if 'article_idx' in bt_df_36.columns:
        bt_df_36 = (
            bt_df_36.set_index('article_idx')
            .reindex(np.arange(n_lit_36))
            .reset_index()
        )
        if bt_df_36['bertopic_topic'].isna().any():
            missing_n = int(bt_df_36['bertopic_topic'].isna().sum())
            raise RuntimeError(f'BERTopic assignments are missing {missing_n} article_idx rows needed to align with literature embeddings.')
    if len(bt_df_36) != n_lit_36:
        raise RuntimeError(f'BERTopic assignment rows ({len(bt_df_36)}) do not match literature embeddings ({n_lit_36}).')

    label_strategy_36 = str(bt_info_36.get('display_label_strategy', pd.Series(['unknown'])).dropna().iloc[0]) if len(bt_info_36) else 'unknown'
    print(f'BERTopic display-label strategy loaded from prepare_data: {label_strategy_36}')
    if label_strategy_36 != 'contrastive_phrase_v4':
        print('WARNING: BERTopic topic-info labels do not appear to be the latest contrastive_phrase_v4 labels. Rerun prepare_data Section 12.')

    lit_region_arr_36 = bt_df_36['bertopic_topic'].to_numpy(dtype=int)
    region_ids_36 = sorted([int(t) for t in np.unique(lit_region_arr_36) if int(t) >= 0])
    region_to_col_36 = {t: i for i, t in enumerate(region_ids_36)}
    n_regions_36 = len(region_ids_36)
    region_label_36 = dict(zip(bt_info_36.get('Topic', pd.Series(region_ids_36)).astype(int), bt_info_36.get('display_label', pd.Series([f'Region {t}' for t in region_ids_36])))) if len(bt_info_36) else {t: f'Region {t}' for t in region_ids_36}

    prop_region_weights = np.zeros((len(proposal_meta), n_regions_36), dtype=np.float32)
    prop_unassigned_frac_36 = np.zeros(len(proposal_meta), dtype=float)
    for i in range(len(proposal_meta)):
        nbr_idx = D_pl_sorted_idx[i, :K_NEIGHBORS_36]
        nbr_regions = lit_region_arr_36[nbr_idx]
        valid = nbr_regions >= 0
        prop_unassigned_frac_36[i] = 1.0 - (valid.sum() / max(1, len(nbr_regions)))
        if valid.sum() > 0:
            for r in nbr_regions[valid]:
                prop_region_weights[i, region_to_col_36[int(r)]] += 1.0
            prop_region_weights[i] /= valid.sum()

    prop_region_max_weight_36 = prop_region_weights.max(axis=1) if n_regions_36 else np.zeros(len(proposal_meta))
    prop_region_entropy_36 = np.array([
        -np.sum(row[row > 0] * np.log(row[row > 0] + 1e-12)) if row.sum() > 0 else 0.0
        for row in prop_region_weights
    ])
    prop_effective_regions_36 = np.exp(prop_region_entropy_36)
    prop_region_n_covered_36 = (prop_region_weights > 0.05).sum(axis=1)
    prop_region_dominant_36 = np.array([region_ids_36[int(row.argmax())] if row.sum() > 0 and n_regions_36 else -1 for row in prop_region_weights], dtype=int)

    group_distributions_36 = {}
    summary_rows_36 = []
    for gm in MODEL_NAMES_36:
        gm_mask = proposal_meta['group_model'].eq(gm).values
        gm_w = prop_region_weights[gm_mask]
        gd = gm_w.sum(axis=0)
        if gd.sum() > 0:
            gd = gd / gd.sum()
        group_distributions_36[gm] = gd
        entropy_val = float(-np.sum(gd[gd > 0] * np.log(gd[gd > 0] + 1e-12))) if gd.sum() > 0 else 0.0
        hhi_val = float(np.sum(gd ** 2)) if gd.sum() > 0 else np.nan
        row36 = {
            'group': gm,
            'n_proposals': int(gm_mask.sum()),
            'breadth_gt5pct': int((gd > 0.05).sum()),
            'shannon_entropy': round(entropy_val, 4),
            'effective_region_count': round(float(np.exp(entropy_val)), 4),
            'dominant_region_frac': round(float(gd.max()), 4) if len(gd) else np.nan,
            'hhi_concentration': round(hhi_val, 4) if np.isfinite(hhi_val) else np.nan,
            'mean_unassigned_neighbor_frac': round(float(prop_unassigned_frac_36[gm_mask].mean()), 4) if gm_mask.sum() else np.nan,
        }
        for region_id, col in region_to_col_36.items():
            row36[f'region_{region_id}_weight'] = round(float(gd[col]), 4)
        summary_rows_36.append(row36)

    bertopic_region_cov_group_df = pd.DataFrame(summary_rows_36)
    bertopic_region_cov_group_df.to_csv(TABLES_DIR / 'bertopic_region_coverage_per_group.csv', index=False)

    prop_region_cov_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
    prop_region_cov_df['dominant_region'] = prop_region_dominant_36
    prop_region_cov_df['dominant_region_label'] = [region_label_36.get(int(r), 'mixed_or_unassigned') for r in prop_region_dominant_36]
    prop_region_cov_df['max_region_weight'] = prop_region_max_weight_36
    prop_region_cov_df['region_entropy'] = prop_region_entropy_36
    prop_region_cov_df['effective_region_count'] = prop_effective_regions_36
    prop_region_cov_df['n_regions_gt5pct'] = prop_region_n_covered_36
    prop_region_cov_df['unassigned_neighbor_frac'] = prop_unassigned_frac_36
    for region_id, col in region_to_col_36.items():
        prop_region_cov_df[f'region_{region_id}_weight'] = prop_region_weights[:, col]
    prop_region_cov_df.to_csv(TABLES_DIR / 'bertopic_region_coverage_per_proposal.csv', index=False)

    test_rows_36 = []
    metric_specs_36 = {
        'max_region_weight': prop_region_max_weight_36,
        'region_entropy': prop_region_entropy_36,
        'unassigned_neighbor_frac': prop_unassigned_frac_36,
    }
    human_mask_36 = proposal_meta['group_model'].eq('Human').values
    for metric_name, metric_vals in metric_specs_36.items():
        human_vals = metric_vals[human_mask_36]
        for gm in ['Claude', 'Gemini', 'GPT-5.2']:
            gm_mask = proposal_meta['group_model'].eq(gm).values
            gm_vals = metric_vals[gm_mask]
            if len(gm_vals) and len(human_vals):
                stat, p = mannwhitneyu(gm_vals, human_vals, alternative='two-sided')
                test_rows_36.append({
                    'comparison': f'{gm} vs Human',
                    'metric': metric_name,
                    'n_group': int(len(gm_vals)),
                    'n_human': int(len(human_vals)),
                    'mean_group': float(np.mean(gm_vals)),
                    'mean_human': float(np.mean(human_vals)),
                    'u_stat': float(stat),
                    'p_value': float(p),
                })
    bertopic_region_tests_df = pd.DataFrame(test_rows_36)
    if len(bertopic_region_tests_df):
        _, p_holm36, _, _ = multipletests(bertopic_region_tests_df['p_value'], method='holm')
        bertopic_region_tests_df['p_holm'] = p_holm36
    bertopic_region_tests_df.to_csv(TABLES_DIR / 'bertopic_region_coverage_tests.csv', index=False)

    # ── Two-panel figure: stacked bar (left) + MW effect size / significance (right) ─
    represented_regions_36 = [r for r in region_ids_36 if any(group_distributions_36[gm][region_to_col_36[r]] > 0 for gm in MODEL_NAMES_36)]
    cmap36 = plt.cm.get_cmap('tab20', max(20, len(represented_regions_36)))

    sns.set_theme(style='whitegrid', context='talk')
    fig36 = plt.figure(figsize=(22, 8), dpi=140)
    gs36  = fig36.add_gridspec(1, 2, width_ratios=[1.5, 1.0], wspace=0.30)
    ax36  = fig36.add_subplot(gs36[0, 0])
    axR36 = fig36.add_subplot(gs36[0, 1])

    # Left: stacked bar chart — legend placed ABOVE the axis so it doesn't overlap the right panel
    x_pos36  = np.arange(len(MODEL_NAMES_36))
    bottom36 = np.zeros(len(MODEL_NAMES_36))
    bar_handles36 = []
    for j, region_id in enumerate(represented_regions_36):
        vals = np.array([group_distributions_36[gm][region_to_col_36[region_id]] for gm in MODEL_NAMES_36])
        if vals.sum() == 0:
            continue
        label = str(region_label_36.get(region_id, f'Region {region_id}'))
        label = ', '.join(label.split(', ')[:3])
        bar = ax36.bar(x_pos36, vals, bottom=bottom36, color=cmap36(j % 20),
                       edgecolor='white', linewidth=0.4, label=f'R{region_id}: {label}')
        bar_handles36.append(bar[0])
        bottom36 += vals
    ax36.set_xticks(x_pos36)
    ax36.set_xticklabels(MODEL_NAMES_36, fontsize=11)
    ax36.set_ylabel(f'Proportion of BERTopic region weight')
    ax36.set_title(
        f'Literature Embedding-Region Coverage by Author Group\n'
        f'(k={K_NEIGHBORS_36} nearest literature neighbors, BERTopic semantic regions)',
        fontsize=12, fontweight='bold'
    )
    ax36.set_ylim(0, 1.05)
    # Legend above the left panel (outside axes, does not overlap right panel)
    ax36.legend(
        handles=bar_handles36,
        labels=[h.get_label() for h in bar_handles36],
        loc='upper left',
        bbox_to_anchor=(0.0, -0.08),
        ncol=3, fontsize=6.5, framealpha=0.9,
        borderaxespad=0,
    )

    # Right: Cliff's delta forest plot — max_region_weight and region_entropy only
    metric_specs_forest = [
        ('max_region_weight',  prop_region_max_weight_36, 'Max region wt'),
        ('region_entropy',     prop_region_entropy_36,    'Region entropy'),
    ]
    ai_models_36      = [m for m in MODEL_NAMES_36 if m != 'Human']
    human_mask_36_viz = proposal_meta['group_model'].eq('Human').values
    ROW_GAP_36 = 0.5

    # Build rows with a header row per section (avoids left-side label collision)
    rows_36   = []   # (ypos, delta, lo, hi, color, model, p_holm, is_header, label)
    y36 = 0.0
    for metric_name, metric_arr, metric_label in metric_specs_forest:
        h_vals = metric_arr[human_mask_36_viz]
        rows_36.append((y36, None, None, None, None, None, None, True, metric_label))
        y36 += 1
        for model in ai_models_36:
            g_mask = proposal_meta['group_model'].eq(model).values
            g_vals = metric_arr[g_mask]
            if len(g_vals) == 0:
                y36 += 1; continue
            delta_val   = cliffs_delta(g_vals, h_vals)
            lo36, hi36  = bootstrap_cliffs_delta_ci(g_vals, h_vals, n_boot=2000, random_state=42)
            p_holm_val  = np.nan
            if 'bertopic_region_tests_df' in globals() and len(bertopic_region_tests_df):
                sub36 = bertopic_region_tests_df.loc[
                    (bertopic_region_tests_df['metric']      == metric_name) &
                    (bertopic_region_tests_df['comparison'] == f'{model} vs Human')
                ]
                if len(sub36):
                    p_holm_val = sub36.iloc[0].get('p_holm', np.nan)
            rows_36.append((y36, delta_val, lo36, hi36,
                            colors.get(model, '#808080'), model, p_holm_val, False, None))
            y36 += 1
        y36 += ROW_GAP_36

    axR36.axvline(0, color='black', linewidth=1.0, alpha=0.8)
    ytick_pos36, ytick_labels36 = [], []
    for row in rows_36:
        ypos = row[0]
        is_header = row[7]
        if is_header:
            label = row[8]
            axR36.text(0.0, ypos, label, va='center', ha='center',
                       fontsize=10, fontweight='bold', color='#333333',
                       bbox=dict(boxstyle='round,pad=0.15', fc='#f0f0f0', ec='#cccccc', lw=0.8),
                       clip_on=False)
            ytick_pos36.append(ypos)
            ytick_labels36.append('')
            # separator line just above this header
            axR36.axhline(ypos - 0.55, color='gray', linewidth=0.6, linestyle='--', alpha=0.4)
        else:
            _, dv, lo, hi, c, model, p_h, _, _ = row
            axR36.errorbar(dv, ypos,
                           xerr=np.array([[dv - lo], [hi - dv]]),
                           fmt='o', color=c, ecolor=c, elinewidth=1.2, capsize=3, markersize=7)
            axR36.text(1.06, ypos, f'MW(Holm)={fmt_p(p_h)}',
                       va='center', ha='left', fontsize=9, clip_on=False)
            ytick_pos36.append(ypos)
            ytick_labels36.append(model)

    axR36.set_yticks(ytick_pos36)
    axR36.set_yticklabels(ytick_labels36, fontsize=9)
    axR36.invert_yaxis()
    axR36.set_xlim(-1.05, 1.05)
    axR36.set_xlabel("Cliff's δ (bootstrap 95% CI)", fontsize=10)
    axR36.set_title(
        f'Effect size (AI − Human) + MW significance\n(k={K_NEIGHBORS_36} nearest-neighbor region metrics)',
        fontsize=11, fontweight='bold'
    )
    axR36.grid(alpha=0.2, axis='x')

    fig36.subplots_adjust(bottom=0.32)   # room for the multi-row legend below the left panel
    out_36 = FIGURES_DIR / 'bertopic_region_coverage_stacked_bar.png'
    fig36.savefig(out_36, dpi=220, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_36}')
    print('\nGroup-level BERTopic region coverage:')
    print(bertopic_region_cov_group_df[['group', 'breadth_gt5pct', 'shannon_entropy', 'effective_region_count', 'dominant_region_frac', 'mean_unassigned_neighbor_frac']].to_string(index=False))
    print('\nMW tests vs Human:')
    if len(bertopic_region_tests_df):
        print(bertopic_region_tests_df[['comparison', 'metric', 'mean_group', 'mean_human', 'p_value', 'p_holm']].to_string(index=False))

# Supplementary LDA lexical topic coverage; keep existing artifact names for continuity.
_lit_topic_path_36 = PREPARED_DIR / 'lit_topic_assignments.csv'
_lda_bundle_path_36 = LITERATURE_EMBEDDINGS_FILE.parent / 'lit_lda_model.pkl'
if _lit_topic_path_36.exists():
    import pickle as _pkl36
    lit_topic_df_36 = pd.read_csv(_lit_topic_path_36)
    dominant_topic_arr_36 = lit_topic_df_36['dominant_topic'].values
    n_lit_topics_36 = int(lit_topic_df_36['dominant_topic'].max()) + 1
    lda_bundle_36 = None
    if _lda_bundle_path_36.exists():
        with open(_lda_bundle_path_36, 'rb') as _fh36:
            lda_bundle_36 = _pkl36.load(_fh36)

    prop_topic_weights = np.zeros((len(proposal_meta), n_lit_topics_36), dtype=np.float32)
    for i in range(len(proposal_meta)):
        nbr_idx = D_pl_sorted_idx[i, :K_NEIGHBORS_36]
        nbr_topics = dominant_topic_arr_36[nbr_idx]
        valid = nbr_topics >= 0
        if valid.sum() > 0:
            for t in nbr_topics[valid]:
                prop_topic_weights[i, t] += 1.0
            prop_topic_weights[i] /= valid.sum()
    prop_max_weight_36 = prop_topic_weights.max(axis=1)
    prop_n_covered_36 = (prop_topic_weights > 0.05).sum(axis=1)

    group_distributions_lda_36 = {}
    summary_rows_lda_36 = []
    for gm in MODEL_NAMES_36:
        gm_mask = proposal_meta['group_model'].eq(gm).values
        gd = prop_topic_weights[gm_mask].sum(axis=0)
        if gd.sum() > 0:
            gd /= gd.sum()
        group_distributions_lda_36[gm] = gd
        entropy_val = float(-np.sum(gd[gd > 0] * np.log(gd[gd > 0] + 1e-12))) if gd.sum() > 0 else 0.0
        row = {'group': gm, 'n_proposals': int(gm_mask.sum()), 'breadth_gt5pct': int((gd > 0.05).sum()), 'shannon_entropy': round(entropy_val, 4), 'dominant_topic_frac': round(float(gd.max()), 4) if len(gd) else np.nan}
        for t in range(n_lit_topics_36):
            row[f'topic_{t}_weight'] = round(float(gd[t]), 4)
        summary_rows_lda_36.append(row)
    lit_topic_cov_group_df = pd.DataFrame(summary_rows_lda_36)
    lit_topic_cov_group_df.to_csv(TABLES_DIR / 'lit_topic_coverage_per_group.csv', index=False)

    prop_lda_36_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
    prop_lda_36_df['max_topic_weight'] = prop_max_weight_36
    prop_lda_36_df['n_topics_gt5pct'] = prop_n_covered_36
    for t in range(n_lit_topics_36):
        prop_lda_36_df[f'topic_{t}_weight'] = prop_topic_weights[:, t]
    prop_lda_36_df.to_csv(TABLES_DIR / 'lit_topic_coverage_per_proposal.csv', index=False)

    human_conc = prop_max_weight_36[proposal_meta['group_model'].eq('Human').values]
    rows = []
    for gm in ['Claude', 'Gemini', 'GPT-5.2']:
        gm_c = prop_max_weight_36[proposal_meta['group_model'].eq(gm).values]
        stat, p = mannwhitneyu(gm_c, human_conc, alternative='two-sided')
        rows.append({'comparison': f'{gm} vs Human', 'metric': 'max_topic_weight', 'n_group': len(gm_c), 'n_human': len(human_conc), 'mean_group': float(gm_c.mean()), 'mean_human': float(human_conc.mean()), 'u_stat': float(stat), 'p_value': float(p)})
    test_df_lda_36 = pd.DataFrame(rows)
    if len(test_df_lda_36):
        _, p_holm_lda, _, _ = multipletests(test_df_lda_36['p_value'], method='holm')
        test_df_lda_36['p_holm'] = p_holm_lda
    test_df_lda_36.to_csv(TABLES_DIR / 'lit_topic_coverage_tests.csv', index=False)
    print('\nSupplementary LDA lexical topic coverage saved to lit_topic_coverage_*.csv')
else:
    print('Supplementary LDA topic assignments not found; skipped LDA lexical comparison.')

print('Analysis 3.6 complete.')


* max_region_weight — of a proposal's k nearest literature neighbors, what fraction fall in the single dominant topic region. Higher = more concentrated in one topic cluster.
* region_entropy — Shannon entropy across regions. Higher = literature spread across more topics (broader coverage). Lower = concentrated in fewer topics.

Claude and Gemini proposals draw on literature that is strikingly more topically narrow than human proposals. Claude's k nearest literature neighbors are almost entirely (99.8%) in a single BERTopic region, with near-zero entropy — essentially it cites within one topic cluster. Gemini is similar but less extreme. GPT-5.2 is statistically indistinguishable from humans.

Humans show the most diverse literature coverage — they draw from across topic regions (entropy ~0.24), suggesting broader interdisciplinary engagement with the literature.

Research interpretation: Claude and Gemini proposals are more topically "on-rails" — their conceptual grounding maps very tightly to a single literature cluster. This could reflect that they generate ideas that stay within a well-defined topic zone, whereas human researchers (and GPT-5.2) draw from a wider, more heterogeneous set of related literatures when forming their proposals.

## Analysis 3.7: MeSH Term Coverage per Author Group

In [ ]:
print('='*85)
print('ANALYSIS 3.7: MeSH TERM COVERAGE PER AUTHOR GROUP')
print('='*85)

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

K_MESH_37 = 20
MODEL_NAMES_37 = ['Human', 'Claude', 'Gemini', 'GPT-5.2']

prop_mesh_counts_37 = []
prop_mesh_sets_37 = []
for i in range(len(proposal_meta)):
    nbr_idx = D_pl_sorted_idx[i, :K_MESH_37]
    mesh_union = set()
    for nidx in nbr_idx:
        for term in articles[int(nidx)].get('mesh_terms', []):
            major = term.split('/')[0].strip()
            if major:
                mesh_union.add(major)
    prop_mesh_counts_37.append(len(mesh_union))
    prop_mesh_sets_37.append(mesh_union)
prop_mesh_counts_37 = np.array(prop_mesh_counts_37)

prop_37_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
prop_37_df['unique_mesh_count'] = prop_mesh_counts_37
prop_37_df['mesh_union_preview'] = ['; '.join(sorted(s)[:20]) for s in prop_mesh_sets_37]
prop_37_df.to_csv(TABLES_DIR / 'mesh_coverage_per_proposal.csv', index=False)

summary_rows_37 = []
for gm in MODEL_NAMES_37:
    gm_mask = proposal_meta['group_model'].eq(gm).values
    gm_c = prop_mesh_counts_37[gm_mask]
    group_union_37 = set()
    for s in [prop_mesh_sets_37[i] for i in np.where(gm_mask)[0]]:
        group_union_37.update(s)
    summary_rows_37.append({
        'group': gm, 'n_proposals': int(gm_mask.sum()),
        'mean_unique_mesh': round(float(gm_c.mean()), 2),
        'median_unique_mesh': round(float(np.median(gm_c)), 2),
        'std_unique_mesh': round(float(gm_c.std()), 2),
        'group_union_unique_mesh': len(group_union_37),
    })

mesh_group_df_37 = pd.DataFrame(summary_rows_37)
mesh_group_df_37.to_csv(TABLES_DIR / 'mesh_coverage_group_summary.csv', index=False)

human_mesh_37 = prop_mesh_counts_37[proposal_meta['group_model'].eq('Human').values]
test_rows_37 = []
for gm in ['Claude', 'Gemini', 'GPT-5.2']:
    gm_m = prop_mesh_counts_37[proposal_meta['group_model'].eq(gm).values]
    stat37, p37 = mannwhitneyu(gm_m, human_mesh_37, alternative='two-sided')
    test_rows_37.append({'comparison': f'{gm} vs Human', 'n_group': len(gm_m), 'n_human': len(human_mesh_37),
                         'mean_group': round(float(gm_m.mean()), 2), 'mean_human': round(float(human_mesh_37.mean()), 2),
                         'u_stat': float(stat37), 'p_value': float(p37)})
test_df_37 = pd.DataFrame(test_rows_37)
if len(test_df_37):
    _, p_holm37, _, _ = multipletests(test_df_37['p_value'], method='holm')
    test_df_37['p_holm'] = p_holm37
test_df_37.to_csv(TABLES_DIR / 'mesh_coverage_tests.csv', index=False)

fig37, axes37 = plt.subplots(1, 3, figsize=(22, 6))

ax37a = axes37[0]
styled_boxplot_with_points(
    ax37a,
    prop_37_df,
    x='group_model',
    y='unique_mesh_count',
    order=MODEL_NAMES_37,
    palette=colors,
    box_width=0.45,
    jitter=0.15,
    point_size=20,
    point_alpha=0.50,
    random_state=42,
    show_metadata_legend=True,
    metadata_legend_loc='best',
)
ax37a.set_ylabel(f'Unique MeSH descriptors (k={K_MESH_37} nearest lit neighbors)', fontsize=10)
ax37a.set_title('MeSH Term Coverage per Author Group\n(per-proposal unique major MeSH; diamond/error bar = mean 95% CI)',
                fontsize=11, fontweight='bold')
ax37a.grid(True, alpha=0.3, axis='y', linestyle='--')

ax37b = axes37[1]
group_unions_37 = [r['group_union_unique_mesh'] for r in summary_rows_37]
bar_colors37 = [colors.get(gm, '#808080') for gm in MODEL_NAMES_37]
bars37 = ax37b.bar(MODEL_NAMES_37, group_unions_37, color=bar_colors37, edgecolor='black', linewidth=0.8, alpha=0.85)
for bar37, val37 in zip(bars37, group_unions_37):
    ax37b.text(bar37.get_x() + bar37.get_width()/2, bar37.get_height() + 5,
               str(val37), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax37b.set_ylabel('Total unique MeSH descriptors (group-level union)', fontsize=10)
ax37b.set_title('Group-Level MeSH Coverage Breadth\n(total unique MeSH across all proposals in group)',
                fontsize=11, fontweight='bold')
ax37b.grid(True, alpha=0.3, axis='y', linestyle='--')

# Right: Cliff's delta forest plot + MW(Holm) significance
ax37c = axes37[2]
human_vals_37 = prop_mesh_counts_37[proposal_meta['group_model'].eq('Human').values]
ai_models_37  = [m for m in MODEL_NAMES_37 if m != 'Human']
forest_rows_37 = []
for yi, model in enumerate(ai_models_37):
    g_vals = prop_mesh_counts_37[proposal_meta['group_model'].eq(model).values]
    dv    = cliffs_delta(g_vals, human_vals_37)
    lo37, hi37 = bootstrap_cliffs_delta_ci(g_vals, human_vals_37, n_boot=2000, random_state=42)
    row37 = test_df_37[test_df_37['comparison'] == f'{model} vs Human']
    p_h37 = row37.iloc[0]['p_holm'] if len(row37) else np.nan
    forest_rows_37.append((yi, dv, lo37, hi37, colors.get(model, '#808080'), model, p_h37))

ax37c.axvline(0, color='black', linewidth=1.0, alpha=0.8)
for (yi, dv, lo, hi, c, model, p_h) in forest_rows_37:
    ax37c.errorbar(dv, yi,
                   xerr=np.array([[dv - lo], [hi - dv]]),
                   fmt='o', color=c, ecolor=c, elinewidth=1.4, capsize=3, markersize=8)
    ax37c.text(1.06, yi, f'MW(Holm)={fmt_p(p_h)}',
               va='center', ha='left', fontsize=9.5, clip_on=False)

ax37c.set_yticks(range(len(ai_models_37)))
ax37c.set_yticklabels(ai_models_37, fontsize=10)
ax37c.invert_yaxis()
ax37c.set_xlim(-1.05, 1.05)
ax37c.set_xlabel("Cliff's δ  (bootstrap 95% CI)\npositive = AI > Human", fontsize=10)
ax37c.set_title('Effect size + MW significance\n(AI − Human, unique MeSH count)',
                fontsize=11, fontweight='bold')
ax37c.grid(alpha=0.2, axis='x')

plt.tight_layout()
out_37 = FIGURES_DIR / 'mesh_coverage_by_group.png'
plt.savefig(out_37, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {out_37}')
print('\nGroup MeSH summary:')
print(mesh_group_df_37.to_string(index=False))
print('\nMW tests:')
print(test_df_37[['comparison', 'mean_group', 'mean_human', 'p_value', 'p_holm']].to_string(index=False))
print('Analysis 3.7 complete.')



## Analysis 3.8: Publication Year Recency of Nearest Literature (Within Embedding Region) [UPDATE]

[UPDATE] Primary stratification uses BERTopic embedding-region labels from Analysis 3.6. LDA-stratified recency may be retained only as a supplementary lexical sensitivity check.


In [ ]:
print('='*85)
print('ANALYSIS 3.8: PUBLICATION YEAR RECENCY OF NEAREST LITERATURE (WITHIN BERTOPIC REGION)')
print('='*85)

import re as _re38
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

K_YEAR_38 = 20
MODEL_NAMES_38 = ['Human', 'Claude', 'Gemini', 'GPT-5.2']

def _parse_year_38(date_str):
    m = _re38.search(r'\b(\d{4})\b', str(date_str))
    return int(m.group(1)) if m else 0

lit_years_arr_38 = np.array([_parse_year_38(art.get('publication_date', '')) for art in articles])

prop_median_year_38 = []
prop_mean_year_38 = []
for i in range(len(proposal_meta)):
    nbr_idx = D_pl_sorted_idx[i, :K_YEAR_38]
    years = lit_years_arr_38[nbr_idx]
    valid_years = years[years > 0]
    prop_median_year_38.append(float(np.median(valid_years)) if len(valid_years) > 0 else np.nan)
    prop_mean_year_38.append(float(np.mean(valid_years)) if len(valid_years) > 0 else np.nan)
prop_median_year_38 = np.array(prop_median_year_38)
prop_mean_year_38 = np.array(prop_mean_year_38)

# Assign BERTopic embedding-region stratum. Reuse Analysis 3.6 outputs if available.
_bt_assign_path_38 = PREPARED_DIR / 'lit_bertopic_assignments.csv'
_bt_info_path_38 = PREPARED_DIR / 'lit_bertopic_topic_info.csv'
region_label_map_38 = {}
if _bt_info_path_38.exists():
    _bt_info_38 = pd.read_csv(_bt_info_path_38)
    if 'Topic' in _bt_info_38.columns:
        region_label_map_38 = dict(zip(_bt_info_38['Topic'].astype(int), _bt_info_38.get('display_label', _bt_info_38['Topic'].astype(str))))

if 'prop_region_weights' in globals() and prop_region_weights is not None and prop_region_weights.shape[0] == len(proposal_meta):
    _region_cols_38 = [int(c.replace('region_', '').replace('_weight', '')) for c in prop_region_cov_df.columns if c.startswith('region_') and c.endswith('_weight')]
    if len(_region_cols_38) != prop_region_weights.shape[1]:
        _region_cols_38 = list(range(prop_region_weights.shape[1]))
    _dom_col_38 = prop_region_weights.argmax(axis=1)
    _dom_w_38 = prop_region_weights.max(axis=1)
    prop_lit_region_stratum_38 = np.array([_region_cols_38[j] if _dom_w_38[i] >= 0.20 else -1 for i, j in enumerate(_dom_col_38)], dtype=int)
elif _bt_assign_path_38.exists():
    _bt_df_38 = pd.read_csv(_bt_assign_path_38)
    _region_arr_38 = _bt_df_38['bertopic_topic'].to_numpy(dtype=int)
    _region_ids_38 = sorted([int(t) for t in np.unique(_region_arr_38) if int(t) >= 0])
    _region_to_col_38 = {t: i for i, t in enumerate(_region_ids_38)}
    _weights_38 = np.zeros((len(proposal_meta), len(_region_ids_38)), dtype=np.float32)
    for i in range(len(proposal_meta)):
        nbr_idx = D_pl_sorted_idx[i, :K_YEAR_38]
        nbr_regions = _region_arr_38[nbr_idx]
        valid = nbr_regions >= 0
        if valid.sum() > 0:
            for r in nbr_regions[valid]:
                _weights_38[i, _region_to_col_38[int(r)]] += 1.0
            _weights_38[i] /= valid.sum()
    _dom_col_38 = _weights_38.argmax(axis=1) if len(_region_ids_38) else np.zeros(len(proposal_meta), dtype=int)
    _dom_w_38 = _weights_38.max(axis=1) if len(_region_ids_38) else np.zeros(len(proposal_meta), dtype=float)
    prop_lit_region_stratum_38 = np.array([_region_ids_38[j] if len(_region_ids_38) and _dom_w_38[i] >= 0.20 else -1 for i, j in enumerate(_dom_col_38)], dtype=int)
else:
    prop_lit_region_stratum_38 = np.full(len(proposal_meta), -1, dtype=int)
    print('WARNING: BERTopic region assignments not found; all proposals assigned to mixed_or_unassigned stratum.')

prop_38_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary']].copy()
prop_38_df['median_neighbor_year'] = prop_median_year_38
prop_38_df['mean_neighbor_year'] = np.round(prop_mean_year_38, 2)
prop_38_df['lit_region_stratum'] = prop_lit_region_stratum_38
prop_38_df['lit_region_label'] = [region_label_map_38.get(int(r), 'mixed_or_unassigned' if int(r) == -1 else f'Region {int(r)}') for r in prop_lit_region_stratum_38]
prop_38_df.to_csv(TABLES_DIR / 'lit_neighbor_year_per_proposal.csv', index=False)

strata_38 = sorted([int(s) for s in np.unique(prop_lit_region_stratum_38) if int(s) >= 0])
test_rows_38 = []
for stratum in strata_38:
    s_mask = prop_lit_region_stratum_38 == stratum
    h_vals = prop_median_year_38[s_mask & proposal_meta['group_model'].eq('Human').values]
    h_vals = h_vals[~np.isnan(h_vals)]
    for gm in ['Claude', 'Gemini', 'GPT-5.2']:
        gm_vals = prop_median_year_38[s_mask & proposal_meta['group_model'].eq(gm).values]
        gm_vals = gm_vals[~np.isnan(gm_vals)]
        if len(gm_vals) >= 3 and len(h_vals) >= 3:
            stat38, p38 = mannwhitneyu(gm_vals, h_vals, alternative='two-sided')
            test_rows_38.append({
                'stratum': stratum,
                'stratum_label': region_label_map_38.get(int(stratum), f'Region {stratum}'),
                'comparison': f'{gm} vs Human',
                'n_group': len(gm_vals),
                'n_human': len(h_vals),
                'median_year_group': float(np.median(gm_vals)),
                'median_year_human': float(np.median(h_vals)),
                'u_stat': float(stat38),
                'p_value': float(p38),
            })

test_df_38 = pd.DataFrame(test_rows_38) if test_rows_38 else pd.DataFrame()
if len(test_df_38) > 0:
    _, p_holm38, _, _ = multipletests(test_df_38['p_value'], method='holm')
    test_df_38['p_holm'] = p_holm38
test_df_38.to_csv(TABLES_DIR / 'lit_neighbor_year_within_region_tests.csv', index=False)

summary_rows_38 = []
all_strata_38 = strata_38 + ([-1] if (-1 in prop_lit_region_stratum_38) else [])
for gm in MODEL_NAMES_38:
    gm_mask = proposal_meta['group_model'].eq(gm).values
    for stratum in all_strata_38:
        s_mask = prop_lit_region_stratum_38 == stratum
        vals = prop_median_year_38[gm_mask & s_mask]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0:
            summary_rows_38.append({
                'group': gm,
                'stratum': stratum if stratum >= 0 else 'mixed_or_unassigned',
                'stratum_label': region_label_map_38.get(int(stratum), 'mixed_or_unassigned' if stratum == -1 else f'Region {stratum}'),
                'n': len(vals),
                'mean_year': round(float(np.mean(vals)), 1),
                'median_year': float(np.median(vals)),
            })
pd.DataFrame(summary_rows_38).to_csv(TABLES_DIR / 'lit_neighbor_year_region_group_summary.csv', index=False)

# Visualization: cap panels to the largest strata plus mixed to keep the figure readable.
_stratum_counts_38 = pd.Series(prop_lit_region_stratum_38).value_counts()
plot_strata_38 = [int(s) for s in _stratum_counts_38.index if int(s) >= 0][:8]
if -1 in _stratum_counts_38.index:
    plot_strata_38.append(-1)
if not plot_strata_38:
    plot_strata_38 = [-1]

n_cols_38 = min(len(plot_strata_38), 4)
n_rows_38 = max(1, -(-len(plot_strata_38) // n_cols_38))
fig38, axes38 = plt.subplots(n_rows_38, n_cols_38, figsize=(5 * n_cols_38, 5 * n_rows_38), squeeze=False)
for ax in axes38.flat:
    ax.set_visible(False)

for panel_i, stratum in enumerate(plot_strata_38):
    ax38 = axes38.flat[panel_i]
    ax38.set_visible(True)
    s_mask = prop_lit_region_stratum_38 == stratum
    label_txt = region_label_map_38.get(int(stratum), 'Mixed/Unassigned' if stratum == -1 else f'Region {stratum}')
    box_d38, labels_38, gm_present_38 = [], [], []
    for gm in MODEL_NAMES_38:
        gm_mask = proposal_meta['group_model'].eq(gm).values
        vals = prop_median_year_38[gm_mask & s_mask]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0:
            box_d38.append(vals)
            labels_38.append(f'{gm}\n(n={len(vals)})')
            gm_present_38.append(gm)
    if box_d38:
        panel_df38 = prop_38_df.loc[s_mask & prop_38_df['group_model'].isin(gm_present_38)].copy()
        styled_boxplot_with_points(
            ax38,
            panel_df38,
            x='group_model',
            y='median_neighbor_year',
            order=gm_present_38,
            palette=colors,
            box_width=0.45,
            jitter=0.15,
            point_size=20,
            point_alpha=0.50,
            random_state=42,
            show_metadata_legend=False,
        )
        ax38.set_xticklabels(labels_38)
    ax38.set_title((f'Region {stratum}: ' if stratum >= 0 else '') + str(label_txt)[:60], fontsize=9, fontweight='bold')
    ax38.set_ylabel(f'Median pub year (k={K_YEAR_38})', fontsize=8)
    ax38.grid(True, alpha=0.3, axis='y', linestyle='--')

fig38.suptitle('Publication Year Recency of Nearest Literature - Within BERTopic Embedding Regions\nDiamond/error bar = mean bootstrap 95% CI', fontsize=12, fontweight='bold')
plt.tight_layout()
out_38 = FIGURES_DIR / 'lit_neighbor_year_by_group_within_bertopic_region.png'
fig38.savefig(out_38, dpi=220, bbox_inches='tight')
plt.show()
print(f'Saved: {out_38}')
if len(test_df_38) > 0:
    print('\nWithin-region MW tests:')
    print(test_df_38[['stratum', 'comparison', 'median_year_group', 'median_year_human', 'p_value', 'p_holm']].to_string(index=False))
else:
    print('No within-region tests (insufficient group sizes per stratum).')
print('Analysis 3.8 complete.')



## Unified Proposal-Level Metric Export

In [ ]:

proposal_metrics_master_df = proposal_meta[['proposal_uid', 'title', 'group_model', 'group_binary', 'is_ai']].copy()

def _dedup(df, cols):
    """Keep first row per proposal_uid to prevent many-to-many merge inflation."""
    return df[cols].drop_duplicates(subset=['proposal_uid'])

# Diversity proposal-level
proposal_metrics_master_df = proposal_metrics_master_df.merge(
    _dedup(pairwise_proposal_df, ['proposal_uid','mean_pairwise_dist']), on='proposal_uid', how='left'
)
proposal_metrics_master_df = proposal_metrics_master_df.merge(
    _dedup(centroid_proposal_df, ['proposal_uid','centroid_dist_raw','centroid_dist_loo']), on='proposal_uid', how='left'
)
proposal_metrics_master_df = proposal_metrics_master_df.merge(
    _dedup(between_group_global_centroid_df, ['proposal_uid','global_centroid_dist']), on='proposal_uid', how='left'
)
proposal_metrics_master_df = proposal_metrics_master_df.merge(
    _dedup(nn_proposal_df, ['proposal_uid','nn_dist_global','is_nn_outlier']), on='proposal_uid', how='left'
)
proposal_metrics_master_df = proposal_metrics_master_df.merge(
    _dedup(mean5_df, ['proposal_uid','mean_5nn_dist_global','is_mean5nn_outlier']), on='proposal_uid', how='left'
)
proposal_metrics_master_df = proposal_metrics_master_df.merge(
    _dedup(medoid_proposal_df, ['proposal_uid','medoid_dist']), on='proposal_uid', how='left'
)
print(f'master df rows after merges: {len(proposal_metrics_master_df)} (expected {len(proposal_meta)})')

# Group-level diversity mapped down
rc_map = pairwise_group_summary_df.set_index('group')['remote_clique'].to_dict()
ch_map = chamfer_group_summary_df.set_index('group')['chamfer'].to_dict()
mst_map = mst_group_summary_df.set_index('group')['mst_dispersion'].to_dict()
sp90_map = span90_group_summary_df.set_index('group')['span_90'].to_dict()
sp_map = sparseness_group_summary_df.set_index('group')['sparseness'].to_dict()
ent_map = entropy_group_summary_df.set_index('group')['grid_entropy'].to_dict()
entn_map = entropy_group_summary_df.set_index('group')['grid_entropy_normalized'].to_dict()

proposal_metrics_master_df['remote_clique_group'] = proposal_metrics_master_df['group_model'].map(rc_map)
proposal_metrics_master_df['chamfer_group'] = proposal_metrics_master_df['group_model'].map(ch_map)
proposal_metrics_master_df['mst_dispersion_group'] = proposal_metrics_master_df['group_model'].map(mst_map)
proposal_metrics_master_df['span90_group'] = proposal_metrics_master_df['group_model'].map(sp90_map)
proposal_metrics_master_df['sparseness_group'] = proposal_metrics_master_df['group_model'].map(sp_map)
proposal_metrics_master_df['grid_entropy_group'] = proposal_metrics_master_df['group_model'].map(ent_map)
proposal_metrics_master_df['grid_entropy_group_norm'] = proposal_metrics_master_df['group_model'].map(entn_map)

# Novelty continuous + flags
for c in ['element_novel_0','element_novel_1','element_novel_5','element_novel_10']:
    proposal_metrics_master_df[c] = element_novelty_df[c].to_numpy()
for c in ['mean_knn_5','mean_knn_10','mean_knn_20','mean_knn_50']:
    proposal_metrics_master_df[c] = mean_knn_novelty_df[c].to_numpy()
for c in ['novelty_ratio','novelty_z']:
    proposal_metrics_master_df[c] = novelty_local_density_df[c].to_numpy()
for c in ['is_lit_outlier_mean10','is_lit_outlier_element0','is_lit_outlier_z']:
    proposal_metrics_master_df[c] = lit_outlier_flags_df[c].to_numpy()


# Literature embedding-region coverage (BERTopic primary topic-region metrics)
_region_cov_cols = [
    'proposal_uid', 'dominant_region', 'dominant_region_label', 'max_region_weight',
    'region_entropy', 'effective_region_count', 'n_regions_gt5pct', 'unassigned_neighbor_frac'
]
if 'prop_region_cov_df' in globals() and prop_region_cov_df is not None:
    _region_cov_src = prop_region_cov_df.copy()
elif (TABLES_DIR / 'bertopic_region_coverage_per_proposal.csv').exists():
    _region_cov_src = pd.read_csv(TABLES_DIR / 'bertopic_region_coverage_per_proposal.csv')
else:
    _region_cov_src = None

if _region_cov_src is not None:
    _available_region_cols = [c for c in _region_cov_cols if c in _region_cov_src.columns]
    proposal_metrics_master_df = proposal_metrics_master_df.merge(
        _dedup(_region_cov_src, _available_region_cols), on='proposal_uid', how='left'
    )
    print('Merged BERTopic literature-region metrics into proposal_metrics_master_df')
else:
    print('BERTopic literature-region metrics not available for master export')

proposal_metrics_master_df.to_csv(TABLES_DIR / 'proposal_metrics_master.csv', index=False)
print('Saved proposal_metrics_master.csv')
print('Rows:', len(proposal_metrics_master_df))


----------------------------
# PART IV Style Baseline

Before interpreting embedding distances, clustering segregation (NMI/ARI), or topic separation as “conceptual” differences, quantify how much **purely stylistic** signals can separate Human vs. AI. If a style-only model predicts source well, then a substantial portion of downstream “segregation/diversity” effects may be stylistic rather than conceptual.

### Exract stylistic features

In [ ]:
# =============================================================================
# ANALYSIS 2.3.5–2.3.6: STYLE VS CONTENT (Baseline + Style-Controlled Sensitivity)
# Text scope: full proposals (full_text)
# =============================================================================

import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ----------------------------
# Style feature extraction
# ----------------------------

_HEDGE_WORDS = {
    'may','might','could','can','suggest','suggests','suggested','potential','potentially',
    'likely','unlikely','possibly','possible','approximately','estimate','estimated',
    'hypothesize','hypothesis','we propose','we aim','we will','we plan'
}

def _tokenize_words(text: str):
    return re.findall(r"[A-Za-z']+", text.lower())

def _split_sentences(text: str):
    # lightweight sentence split for readability/stylistic counts
    sents = re.split(r"[.!?]+\s+|\n+", text.strip())
    return [s for s in (sent.strip() for sent in sents) if s]

def _count_syllables(word: str) -> int:
    # heuristic syllable counter (good enough for comparative readability proxies)
    w = re.sub(r"[^a-z]", "", word.lower())
    if not w:
        return 0
    w = re.sub(r"e$", "", w)  # silent e
    groups = re.findall(r"[aeiouy]+", w)
    return max(1, len(groups))

def _flesch_kincaid(text: str):
    words = _tokenize_words(text)
    n_words = len(words)
    sents = _split_sentences(text)
    n_sents = len(sents)
    if n_words == 0 or n_sents == 0:
        return np.nan, np.nan
    syllables = sum(_count_syllables(w) for w in words)

    # Flesch Reading Ease & FK Grade Level (English)
    fre = 206.835 - 1.015 * (n_words / n_sents) - 84.6 * (syllables / n_words)
    fkgl = 0.39 * (n_words / n_sents) + 11.8 * (syllables / n_words) - 15.59
    return fre, fkgl

def extract_style_features(text: str) -> dict:
    text = text or ""
    words = _tokenize_words(text)
    n_words = len(words)
    n_chars = len(text)
    sents = _split_sentences(text)
    n_sents = len(sents)

    uniq = len(set(words)) if n_words else 0
    ttr = (uniq / n_words) if n_words else 0.0

    stop_ct = sum(1 for w in words if w in ENGLISH_STOP_WORDS)
    stop_rate = (stop_ct / n_words) if n_words else 0.0

    avg_word_len = (np.mean([len(w) for w in words]) if n_words else 0.0)
    avg_sent_len = (n_words / n_sents) if n_sents else 0.0

    # punctuation / formatting (normalized)
    punct = {
        'comma': text.count(','),
        'semicolon': text.count(';'),
        'colon': text.count(':'),
        'dash': text.count('-') + text.count('–') + text.count('—'),
        'paren': text.count('(') + text.count(')'),
        'quote': text.count('"') + text.count("'"),
        'newline': text.count('\n'),
        'bullet': len(re.findall(r"(^|\n)\s*[-*•]\s+", text))
    }
    punct_rate = {f"{k}_per_1k_chars": (v / max(1, n_chars)) * 1000.0 for k, v in punct.items()}

    # hedging / stance
    low = text.lower()
    hedge_hits = 0
    # count single-token hedges
    hedge_hits += sum(1 for w in words if w in _HEDGE_WORDS)
    # count a few multiword patterns
    hedge_hits += len(re.findall(r"\bwe\s+(propose|aim|plan|will)\b", low))
    hedge_rate = (hedge_hits / n_words) if n_words else 0.0

    # simple headers / sectioning markers (your full_text includes e.g., "Title:", "Abstract:")
    header_lines = 0
    for line in (ln.strip() for ln in text.splitlines()):
        if re.match(r"^[A-Z][A-Za-z0-9 /-]{1,40}:\s+", line):
            header_lines += 1
    header_rate = header_lines

    fre, fkgl = _flesch_kincaid(text)

    feats = {
        'n_words': n_words,
        'n_chars': n_chars,
        'n_sents': n_sents,
        'avg_word_len': avg_word_len,
        'avg_sent_len_words': avg_sent_len,
        'type_token_ratio': ttr,
        'stopword_rate': stop_rate,
        'hedge_rate': hedge_rate,
        'flesch_reading_ease': fre,
        'fk_grade_level': fkgl,
    }
    feats.update(punct_rate)
    return feats

# Build style feature table (full proposals)
human_texts_style = human_df['full_text'].fillna('').tolist()
ai_texts_style = ai_df['full_text'].fillna('').tolist()
all_texts_style = human_texts_style + ai_texts_style

style_rows = [extract_style_features(t) for t in all_texts_style]
style_df = pd.DataFrame(style_rows)
style_df['group'] = (['Human'] * len(human_texts_style)) + (['AI'] * len(ai_texts_style))
style_df['is_ai'] = (style_df['group'] == 'AI').astype(int)

print("="*85)
print("STYLE FEATURES: EXTRACTED")
print("="*85)
print(f"✓ Built style feature table: {style_df.shape[0]} docs × {style_df.shape[1]-2} features")
print(style_df.groupby('group')[['avg_sent_len_words','stopword_rate','hedge_rate','fk_grade_level']].mean().round(3))


In [ ]:
# Save: style features with titles
import pandas as pd; from pathlib import Path
out_dir = TABLES_DIR; out_dir.mkdir(parents=True, exist_ok=True)
titles = ([r.get('proposal_title', r.get('title','')) for r in human_metadata]
        + [r.get('title', r.get('proposal_title','')) for r in ai_metadata])
out = style_df.copy().reset_index(drop=True)
out.insert(0, 'title', titles)
out.to_csv(out_dir/'style_features.csv', index=False)
print(f"Saved style_features.csv  ({len(out)} rows x {len(out.columns)} cols)")


#### Visualization: Style feature distributions by group (Human vs each AI model)

**How to read:** Each subplot is one style feature.
- The **box** shows median (center line) and IQR (25th–75th percentile).
- The **black diamond** is the mean, and the **error bar** is ±1 standard deviation.

This makes it easy to compare **median vs mean** (skew/outliers) and overall spread across Human and each AI model.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Boxplots per feature across Human and each AI model
# Box = median/IQR, whiskers = spread, diamond/error bar = mean bootstrap 95% CI

if 'style_df' not in globals():
    raise RuntimeError("style_df not found. Run the style feature extraction cell first.")
if 'ai_df' not in globals():
    raise RuntimeError("ai_df not found. Run the AI proposal loading cell first.")

# Reconstruct per-model labels aligned to style_df construction order: Human rows first, then AI rows
n_h = int((style_df['group'] == 'Human').sum()) if 'group' in style_df.columns else 0
if n_h == 0:
    n_h = len(human_df)

ai_model_labels = ai_df['model'].tolist()
expected_ai = len(style_df) - n_h
if len(ai_model_labels) != expected_ai:
    raise ValueError(
        f"Alignment error: style_df has {len(style_df)} rows with n_h={n_h}, expected {expected_ai} AI rows, "
        f"but ai_df['model'] has {len(ai_model_labels)} labels."
    )

style_plot = style_df.copy()
style_plot['group_model'] = (['Human'] * n_h) + ai_model_labels
if 'proposal_meta' in globals() and len(proposal_meta) == len(style_plot):
    _style_meta_cols = [c for c in ['proposal_uid', 'title', 'funding', 'is_top5_ranked'] if c in proposal_meta.columns]
    for _c in _style_meta_cols:
        style_plot[_c] = proposal_meta[_c].to_numpy()

# Compact, interpretable features for quick comparison
features_to_plot = [
    'avg_sent_len_words',
    'type_token_ratio',
    'stopword_rate',
    'hedge_rate',
    'fk_grade_level',
    'header_lines', 'n_words', 'n_chars', 'n_sents', 'avg_word_len', 'avg_sent_len_words', 'flesch_reading_ease', 'comma_per_1k_chars', 'semicolon_per_1k_chars', 'colon_per_1k_chars', 'dash_per_1k_chars', 'paren_per_1k_chars', 'quote_per_1k_chars', 'newline_per_1k_chars', 'bullet_per_1k_chars'
]
# features_to_plot = [f for f in features_to_plot if f in style_plot.columns]
features_to_plot = [f for f in features_to_plot if f in style_plot.columns]
# Prepare long DF. Keep proposal metadata so jittered points can show funding/top-rank styling.
_style_id_cols = ['group_model'] + [
    c for c in ['proposal_uid', 'title', 'funding', 'is_top5_ranked']
    if c in style_plot.columns
]
long_df = style_plot[_style_id_cols + features_to_plot].replace([np.inf, -np.inf], np.nan)
for f in features_to_plot:
    long_df[f] = long_df[f].fillna(long_df[f].median())
long_df = long_df.melt(id_vars=_style_id_cols, var_name='feature', value_name='value')

# Group order (Human first, then models)
if 'ai_models' in globals():
    group_order = ['Human'] + [m for m in ai_models if m in set(style_plot['group_model'])]
else:
    group_order = ['Human'] + sorted([m for m in style_plot['group_model'].unique() if m != 'Human'])

# Plot subplots
n_feat = len(features_to_plot)
ncols = 3
nrows = int(np.ceil(n_feat / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4.8 * nrows))
axes = np.array(axes).reshape(-1)

# Use consistent group colors if defined earlier in the notebook
if 'colors' in globals() and isinstance(colors, dict):
    colors_local = dict(colors)
else:
    colors_local = {}

# Ensure every group has a color (fallback for any missing keys)
for idx, g in enumerate(group_order):
    colors_local.setdefault(g, f"C{idx % 10}")

palette_dict = {g: colors_local[g] for g in group_order}

for i, feat in enumerate(features_to_plot):
    ax = axes[i]
    df_f = long_df[long_df['feature'] == feat]

    # Standardized boxplot grammar with jittered proposal points and mean CI.
    styled_boxplot_with_points(
        ax,
        df_f,
        x='group_model',
        y='value',
        order=group_order,
        palette=palette_dict,
        box_width=0.45,
        jitter=0.15,
        point_size=20,
        point_alpha=0.50,
        random_state=42,
        show_metadata_legend=(i == 0),
        metadata_legend_loc='best',
    )

    ax.set_title(feat.replace('_', ' ').title(), fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Style feature distributions by group\n(Box = median/IQR; diamond/error bar = mean bootstrap 95% CI)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

out_path = FIGURES_DIR / 'style_features_by_model_boxplots.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Figure saved to: {out_path}")

# Print a compact table of mean/median/std
summary = style_plot.groupby('group_model')[features_to_plot].agg(['mean', 'median', 'std']).round(3)
print("\nSummary (mean / median / std) by group:")
print(summary.to_string())


### Analysis 2.3.5: Style-only baseline (can style predict source?)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, permutation_test_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

print("="*85)
print("STYLE-ONLY BASELINE: HUMAN VS AI")
print("="*85)

feature_cols = [c for c in style_df.columns if c not in {'group','is_ai'}]
print(f"features: {feature_cols}")
X = style_df[feature_cols].replace([np.inf, -np.inf], np.nan).values
y = style_df['is_ai'].values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=5000, class_weight='balanced', solver='liblinear', random_state=42)
)

auc_scores = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc')
bal_scores = cross_val_score(clf, X, y, cv=cv, scoring='balanced_accuracy')

print(f"CV AUROC: {auc_scores.mean():.3f} ± {auc_scores.std():.3f}")
print(f"CV balanced accuracy: {bal_scores.mean():.3f} ± {bal_scores.std():.3f}")

obs_score, perm_scores, p_value = permutation_test_score(
    clf, X, y, cv=cv, scoring='roc_auc', n_permutations=1000, n_jobs=-1, random_state=42
)

print()
print("Permutation test (AUROC, 1,000 label shuffles):")
print(f"  Observed AUROC: {obs_score:.3f}")
print(f"  Null mean AUROC: {perm_scores.mean():.3f} ± {perm_scores.std():.3f}")
print(f"  p-value: {p_value:.4f}")
print("  Note: Imputation now occurs INSIDE each CV fold (no leakage).")

if obs_score >= 0.80:
    print("  → Strong evidence that style alone separates Human vs AI.")
elif obs_score >= 0.60:
    print("  → Moderate evidence that style contributes to separation.")
else:
    print("  → Style-only separation is weak; downstream separation is less likely to be purely stylistic.")


In [ ]:
# ── Most predictive STYLE features (interpretable coefficients) ───────────────
import numpy as np
import pandas as pd

print("="*85)
print("TOP STYLE FEATURES PREDICTING AI VS HUMAN (LogReg coefficients)")
print("="*85)

# Fit the SAME pipeline on all data for interpretation
clf.fit(X, y)

# Pull fitted components
imputer = clf.named_steps['simpleimputer']
scaler  = clf.named_steps['standardscaler']
logreg  = clf.named_steps['logisticregression']

# Feature names from the current cell
feature_names = list(feature_cols)

# Coefs correspond to *scaled* features (comparable magnitudes)
coefs = logreg.coef_.ravel()  # shape: (n_features,)

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs,
    'abs_coef': np.abs(coefs)
}).sort_values('abs_coef', ascending=False)

k = min(15, len(coef_df))

print(f"\nTop {k} strongest predictors (by |coef|):")
print(coef_df.head(k)[['feature','coef']].to_string(index=False))

# Directional lists
ai_like = coef_df.sort_values('coef', ascending=False).head(k)[['feature','coef']]
human_like = coef_df.sort_values('coef', ascending=True).head(k)[['feature','coef']]

print(f"\nTop {k} features pushing prediction toward AI (positive coef):")
print(ai_like.to_string(index=False))

print(f"\nTop {k} features pushing prediction toward Human (negative coef):")
print(human_like.to_string(index=False))

print("\nNote: coefficients are on standardized features (after imputation + scaling).")
print("      Positive = more AI-like; Negative = more Human-like.")


#### Visualization: Style-only baseline results (CV + permutation test)

**How to read:** Left panel shows cross-validation fold performance (each dot = one CV fold; box = distribution). Right panel shows the permutation-test *null* distribution of AUROC when Human/AI labels are shuffled; the vertical red line is the observed AUROC—if it sits far in the right tail, style alone separates groups beyond chance (small p-value).

In [ ]:
# Visualization for style-only baseline results
import matplotlib.pyplot as plt

# 1) CV fold distributions (AUROC + balanced accuracy)
cv_plot_df = pd.DataFrame({
    'Fold': np.arange(1, len(auc_scores) + 1),
    'AUROC': auc_scores,
    'Balanced Accuracy': bal_scores,
})
cv_long = cv_plot_df.melt(id_vars='Fold', var_name='Metric', value_name='Score')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: CV score distributions
ax = axes[0]
styled_boxplot_with_points(
    ax,
    cv_long,
    x='Metric',
    y='Score',
    order=['AUROC', 'Balanced Accuracy'],
    palette={'AUROC': '#4A90E2', 'Balanced Accuracy': '#3CB371'},
    box_width=0.45,
    jitter=0.12,
    point_size=36,
    point_alpha=0.65,
    random_state=42,
)
ax.set_title('Cross-validation performance (5 folds)\nDiamond/error bar = mean bootstrap 95% CI', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Score')
ax.set_ylim(0.0, 1.05)
ax.grid(axis='y', alpha=0.3)

# Right: permutation null distribution
ax = axes[1]
sns.histplot(perm_scores, bins=30, kde=True, ax=ax, color='#8A8A8A', alpha=0.55)
ax.axvline(0.5, color='black', linestyle='--', linewidth=2, alpha=0.6, label='Chance AUROC = 0.5')
ax.axvline(obs_score, color='crimson', linestyle='-', linewidth=3,
           label=f'Observed AUROC = {obs_score:.3f} (p={p_value:.4f})')
ax.set_title('Permutation test null distribution (AUROC)', fontsize=13, fontweight='bold')
ax.set_xlabel('AUROC under label shuffling')
ax.set_ylabel('Count')
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=9)

plt.tight_layout()

out_path = FIGURES_DIR / 'style_only_baseline_viz.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Figure saved to: {out_path}")
print("\nInterpretation:")
print("- Left: tighter boxes/points = stable CV estimates; higher = better separation.")
print("- Right: if the red line is far to the right of the gray histogram, the result is unlikely under chance (small p).")


# Save All Proposals to a Single JSON

Combine original proposal content, rephrased text, and every per-proposal metric computed in this notebook into one JSON file (`all_proposals.json`). Each record contains the original and rephrased text, plus all diversity, novelty, style, and review-score metrics for cross-validation.

In [ ]:

import json as _json
import pandas as pd
import numpy as np
from pathlib import Path


def _norm(s):
    return str(s).strip().lower()

master_path = TABLES_DIR / 'proposal_metrics_master.csv'
style_path = TABLES_DIR / 'style_features.csv'
scores_path = PREPARED_DIR / 'review_scores_wide.csv'

master_df = pd.read_csv(master_path)
style_df = pd.read_csv(style_path) if style_path.exists() else pd.DataFrame()
scores_df = pd.read_csv(scores_path) if scores_path.exists() else pd.DataFrame()

master_df['_title_key'] = master_df['title'].map(_norm)
if 'title' in style_df.columns:
    style_df['_title_key'] = style_df['title'].map(_norm)
if 'title' in scores_df.columns:
    scores_df['_title_key'] = scores_df['title'].map(_norm)

# Read existing proposals JSON (prepared first, then canonical fallback)
in_json = PREPARED_ALL_PROPOSALS_PATH if PREPARED_ALL_PROPOSALS_PATH.exists() else (TABLES_DIR / 'all_proposals.json')
with open(in_json, 'r') as f:
    records = _json.load(f)

master_uid_lut = {str(r['proposal_uid']): r for _, r in master_df.iterrows() if pd.notna(r.get('proposal_uid', None))}
master_title_lut = {_norm(r['title']): r for _, r in master_df.iterrows()}

style_cols = [c for c in style_df.columns if c not in {'title','group','is_ai','_title_key'}]
score_cols = [c for c in scores_df.columns if c not in {'title','author','_title_key'}]
style_lut = {r['_title_key']: {c: r.get(c, None) for c in style_cols} for _, r in style_df.iterrows()} if len(style_df) else {}
scores_lut = {r['_title_key']: {c: r.get(c, None) for c in score_cols} for _, r in scores_df.iterrows()} if len(scores_df) else {}

missing_master = 0
for rec in records:
    title = rec.get('title', '')
    uid = rec.get('proposal_uid', None)
    key = _norm(title)

    mrow = None
    if uid is not None and str(uid) in master_uid_lut:
        mrow = master_uid_lut[str(uid)]
    elif key in master_title_lut:
        mrow = master_title_lut[key]

    if mrow is None:
        missing_master += 1
        continue

    rec['proposal_uid'] = mrow.get('proposal_uid', uid)
    rec['metrics'] = rec.get('metrics', {})
    for col, val in mrow.items():
        if col in {'proposal_uid', 'title', 'group_model', 'group_binary', 'is_ai', '_title_key'}:
            continue
        rec['metrics'][col] = None if pd.isna(val) else (bool(val) if isinstance(val, (np.bool_, bool)) else val)

    if key in style_lut:
        rec['style_features'] = {k: (None if pd.isna(v) else v) for k,v in style_lut[key].items()}
    if key in scores_lut:
        rec['review_scores'] = {k: (None if pd.isna(v) else v) for k,v in scores_lut[key].items()}

# write merged json
out_json = TABLES_DIR / 'all_proposals.json'
with open(out_json, 'w') as f:
    _json.dump(records, f, indent=2)

# sanity checks
print('='*85)
print('JSON MERGE SANITY CHECKS')
print('='*85)
print('Input JSON:', in_json)
print('Output JSON:', out_json)
print('Master rows:', len(master_df))
print('Records in output:', len(records))
print('Records missing a master row:', missing_master)

# family-wise missingness in master
families = {
    'diversity': [c for c in master_df.columns if c.startswith(('mean_pairwise','centroid_','global_centroid','nn_','mean_5nn','medoid_','remote_clique','chamfer','mst_','span90','sparseness','grid_entropy'))],
    'novelty': [c for c in master_df.columns if c.startswith(('element_novel_','mean_knn_','novelty_','is_lit_outlier_'))],
}
for fam, cols in families.items():
    if cols:
        miss = master_df[cols].isna().sum().sum()
        print(f'Missing values in {fam} family: {miss}')

# outlier alignment checks
chk_cols = ['mean_knn_10', 'is_lit_outlier_mean10', 'element_novel_0', 'is_lit_outlier_element0', 'novelty_z', 'is_lit_outlier_z']
if all(c in master_df.columns for c in chk_cols):
    print('Outlier flag alignment checks:')
    for value_col, flag_col in [('mean_knn_10','is_lit_outlier_mean10'), ('element_novel_0','is_lit_outlier_element0'), ('novelty_z','is_lit_outlier_z')]:
        valid = master_df[[value_col, flag_col]].dropna()
        if len(valid):
            thr = np.percentile(valid[value_col], 90)
            implied = valid[value_col] > thr
            agree = float((implied == valid[flag_col].astype(bool)).mean())
            print(f'  {flag_col}: agreement with top-10% rule = {agree:.3f}')
